# EquiNexus model sprint

This GPU notebook runs the repository's from-scratch, per-annotator filament U-Net against the frozen observation-grouped split and organizer scorer. The model uses only competition-provided training annotations; test inference reads only the attached H-alpha test images. No external labels or pretrained weights are loaded.

The source files are embedded from the local checkout by `scripts/build_kaggle_model_notebook.py`, so the run does not depend on a GitHub push. This version validates a 1024-pixel patch model; final training is run only after selecting the validation-supported model and postprocessing.


In [ ]:
import base64
from pathlib import Path
REPOSITORY_PATH = Path('/kaggle/working/EquiNexus')
SOURCE_COMMIT = '9668cdee92a5ef1f9150bb585b54dbe7881e7f7a'
SOURCE_BUNDLE_SHA256 = 'f7f17d514c01ca6eaa0960fde5e0039d8b46490c9d24622ad51d4a079ca63326'
SOURCE_FILES = {"src/filament_baseline/__init__.py":"IiIiU2NvcmVyLWFsaWduZWQgdmFsaWRhdGlvbiBiYXNlbGluZSBmb3IgRXF1aU5leHVzLiIiIgoKX192ZXJzaW9uX18gPSAiMC4xLjAiCg==","src/filament_baseline/classical.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmRhdGEgaW1wb3J0IElNQUdFX1NJWkUKZnJvbSAucHJlZGljdG9yIGltcG9ydCBQcmVkaWN0ZWRJbnN0YW5jZSwgZW5jb2RlX2Nyb3BwZWRfYmluYXJ5X21hc2sKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDbGFzc2ljYWxDb25maWc6CiAgICB6X3RocmVzaG9sZDogZmxvYXQgPSAyLjU1CiAgICBicmlkZ2Vfc2l6ZTogaW50ID0gOQogICAgbWluX2luc3RhbmNlX2FyZWE6IGludCA9IDE2NQogICAgbGltYl9mcmFjdGlvbjogZmxvYXQgPSAwLjk0NQoKCmRlZiBfZGlza19nZW9tZXRyeShncmF5OiBucC5uZGFycmF5KSAtPiB0dXBsZVtpbnQsIGludCwgaW50XToKICAgIGhlaWdodCwgd2lkdGggPSBncmF5LnNoYXBlCiAgICByZWR1Y2VkID0gY3YyLnJlc2l6ZShncmF5LCAobWF4KDEsIHdpZHRoIC8vIDQpLCBtYXgoMSwgaGVpZ2h0IC8vIDQpKSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfQVJFQSkKICAgIF8sIGJyaWdodCA9IGN2Mi50aHJlc2hvbGQocmVkdWNlZCwgMCwgMjU1LCBjdjIuVEhSRVNIX0JJTkFSWSB8IGN2Mi5USFJFU0hfT1RTVSkKICAgIGNvbnRvdXJzLCBfID0gY3YyLmZpbmRDb250b3VycyhicmlnaHQsIGN2Mi5SRVRSX0VYVEVSTkFMLCBjdjIuQ0hBSU5fQVBQUk9YX1NJTVBMRSkKICAgIGlmIGNvbnRvdXJzOgogICAgICAgIChjZW50ZXJfeCwgY2VudGVyX3kpLCByYWRpdXMgPSBjdjIubWluRW5jbG9zaW5nQ2lyY2xlKG1heChjb250b3Vycywga2V5PWN2Mi5jb250b3VyQXJlYSkpCiAgICAgICAgY2VudGVyX3gsIGNlbnRlcl95LCByYWRpdXMgPSBpbnQocm91bmQoY2VudGVyX3ggKiA0KSksIGludChyb3VuZChjZW50ZXJfeSAqIDQpKSwgaW50KHJvdW5kKHJhZGl1cyAqIDQpKQogICAgICAgIGlmIDY1MCA8PSByYWRpdXMgPD0gMTAyMDoKICAgICAgICAgICAgcmV0dXJuIGNlbnRlcl94LCBjZW50ZXJfeSwgcmFkaXVzCiAgICByZXR1cm4gd2lkdGggLy8gMiwgaGVpZ2h0IC8vIDIsIGludCgwLjQzICogbWluKGhlaWdodCwgd2lkdGgpKQoKCmRlZiBjb21wdXRlX3NhbGllbmNlX3ptYXAoZ3JheV9pbWFnZTogbnAubmRhcnJheSwgbGltYl9mcmFjdGlvbjogZmxvYXQgPSAwLjk0NSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICAiIiJFc3RpbWF0ZSBkaXNrLW5vcm1hbGl6ZWQgZGFyayBmaWxhbWVudCByZXNwb25zZSBhbmQgYSBzYWZlLWRpc2sgbWFzay4iIiIKICAgIGlmIGdyYXlfaW1hZ2UubmRpbSAhPSAyIG9yIGdyYXlfaW1hZ2UuZHR5cGUgIT0gbnAudWludDg6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgYW4gOC1iaXQgZ3JheXNjYWxlIGltYWdlIikKICAgIGlmIGdyYXlfaW1hZ2Uuc2hhcGUgIT0gKElNQUdFX1NJWkUsIElNQUdFX1NJWkUpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCB7SU1BR0VfU0laRX14e0lNQUdFX1NJWkV9LCBnb3Qge2dyYXlfaW1hZ2Uuc2hhcGV9IikKCiAgICByZWR1Y2VkID0gY3YyLnJlc2l6ZShncmF5X2ltYWdlLCAoMTAyNCwgMTAyNCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0FSRUEpLmFzdHlwZShucC5mbG9hdDMyKQogICAgY2VudGVyX3gsIGNlbnRlcl95LCByYWRpdXMgPSBfZGlza19nZW9tZXRyeShyZWR1Y2VkLmFzdHlwZShucC51aW50OCkpCiAgICB5eSwgeHggPSBucC5vZ3JpZFs6MTAyNCwgOjEwMjRdCiAgICBzYWZlX2Rpc2sgPSAoeHggLSBjZW50ZXJfeCkgKiogMiArICh5eSAtIGNlbnRlcl95KSAqKiAyIDw9IChsaW1iX2ZyYWN0aW9uICogcmFkaXVzKSAqKiAyCgogICAgYnJvYWQgPSBjdjIuR2F1c3NpYW5CbHVyKHJlZHVjZWQsICgwLCAwKSwgc2lnbWFYPTI4LjApCiAgICBuYXJyb3cgPSBjdjIuR2F1c3NpYW5CbHVyKHJlZHVjZWQsICgwLCAwKSwgc2lnbWFYPTE0LjApCiAgICBsb2NhbF9iYWNrZ3JvdW5kID0gMC42NSAqIGJyb2FkICsgMC4zNSAqIG5hcnJvdwogICAgYWJzb3JwdGlvbiA9IG5wLmNsaXAoCiAgICAgICAgKGxvY2FsX2JhY2tncm91bmQgLSByZWR1Y2VkKSAvIG5wLm1heGltdW0obG9jYWxfYmFja2dyb3VuZCwgMjUuMCksIDAuMCwgMS4wCiAgICApCgogICAgcmVkdWNlZF91OCA9IG5wLmNsaXAocmVkdWNlZCwgMCwgMjU1KS5hc3R5cGUobnAudWludDgpCiAgICByZXNwb25zZXMgPSBbXQogICAgZm9yIHNpemUgaW4gKDksIDE5LCAzMyk6CiAgICAgICAgZWxlbWVudCA9IGN2Mi5nZXRTdHJ1Y3R1cmluZ0VsZW1lbnQoY3YyLk1PUlBIX0VMTElQU0UsIChzaXplLCBzaXplKSkKICAgICAgICByZXNwb25zZSA9IGN2Mi5tb3JwaG9sb2d5RXgocmVkdWNlZF91OCwgY3YyLk1PUlBIX0JMQUNLSEFULCBlbGVtZW50KQogICAgICAgIHJlc3BvbnNlcy5hcHBlbmQocmVzcG9uc2UuYXN0eXBlKG5wLmZsb2F0MzIpIC8gMjU1LjApCiAgICBzYWxpZW5jZSA9ICgKICAgICAgICAwLjUwICogYWJzb3JwdGlvbgogICAgICAgICsgMC4xNSAqIHJlc3BvbnNlc1swXQogICAgICAgICsgMC4yMCAqIHJlc3BvbnNlc1sxXQogICAgICAgICsgMC4xNSAqIHJlc3BvbnNlc1syXQogICAgKQogICAgZGlza192YWx1ZXMgPSBzYWxpZW5jZVtzYWZlX2Rpc2tdCiAgICBtZWRpYW4gPSBmbG9hdChucC5tZWRpYW4oZGlza192YWx1ZXMpKQogICAgcm9idXN0X3NjYWxlID0gMS40ODI2ICogZmxvYXQobnAubWVkaWFuKG5wLmFicyhkaXNrX3ZhbHVlcyAtIG1lZGlhbikpKSArIDFlLTUKICAgIHpfbWFwID0gKHNhbGllbmNlIC0gbWVkaWFuKSAvIHJvYnVzdF9zY2FsZQogICAgel9tYXBbfnNhZmVfZGlza10gPSAtbnAuaW5mCiAgICByZXR1cm4gel9tYXAuYXN0eXBlKG5wLmZsb2F0MzIpLCBzYWZlX2Rpc2sKCgpkZWYgcHJlZGljdGlvbnNfZnJvbV96bWFwKAogICAgel9tYXA6IG5wLm5kYXJyYXksCiAgICBzYWZlX2Rpc2s6IG5wLm5kYXJyYXksCiAgICBmaWxlX25hbWU6IHN0ciwKICAgIGNvbmZpZzogQ2xhc3NpY2FsQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV06CiAgICBjb25maWcgPSBjb25maWcgb3IgQ2xhc3NpY2FsQ29uZmlnKCkKICAgIGlmIHpfbWFwLnNoYXBlICE9ICgxMDI0LCAxMDI0KSBvciBzYWZlX2Rpc2suc2hhcGUgIT0gel9tYXAuc2hhcGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgYSAxMDI0eDEwMjQgc2FsaWVuY2UgbWFwIGFuZCBkaXNrIG1hc2siKQogICAgaWYgY29uZmlnLmJyaWRnZV9zaXplIDwgMSBvciBjb25maWcuYnJpZGdlX3NpemUgJSAyID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYnJpZGdlX3NpemUgbXVzdCBiZSBhIHBvc2l0aXZlIG9kZCBudW1iZXIiKQoKICAgIHdlYWsgPSAoKHpfbWFwID49IGNvbmZpZy56X3RocmVzaG9sZCkgJiBzYWZlX2Rpc2spLmFzdHlwZShucC51aW50OCkKICAgIHN0cm9uZyA9ICh6X21hcCA+PSBjb25maWcuel90aHJlc2hvbGQgKyAwLjU1KSAmIHNhZmVfZGlzawogICAgaWYgY29uZmlnLmJyaWRnZV9zaXplID4gMToKICAgICAgICBicmlkZ2UgPSBjdjIuZ2V0U3RydWN0dXJpbmdFbGVtZW50KAogICAgICAgICAgICBjdjIuTU9SUEhfRUxMSVBTRSwgKGNvbmZpZy5icmlkZ2Vfc2l6ZSwgY29uZmlnLmJyaWRnZV9zaXplKQogICAgICAgICkKICAgICAgICB3ZWFrID0gY3YyLm1vcnBob2xvZ3lFeCh3ZWFrLCBjdjIuTU9SUEhfQ0xPU0UsIGJyaWRnZSkKICAgICAgICB3ZWFrW35zYWZlX2Rpc2tdID0gMAoKICAgIGNvbXBvbmVudF9jb3VudCwgbGFiZWxzLCBzdGF0cywgXyA9IGN2Mi5jb25uZWN0ZWRDb21wb25lbnRzV2l0aFN0YXRzKAogICAgICAgIHdlYWssIGNvbm5lY3Rpdml0eT04CiAgICApCiAgICBtaW5fcmVkdWNlZF9hcmVhID0gbWF4KDIwLCBjb25maWcubWluX2luc3RhbmNlX2FyZWEgLy8gNCkKICAgIGltYWdlX3N0ZW0gPSBQYXRoKGZpbGVfbmFtZSkuc3RlbQogICAgcHJlZGljdGlvbnMgPSBbXQogICAgZm9yIGNvbXBvbmVudF9pZCBpbiByYW5nZSgxLCBjb21wb25lbnRfY291bnQpOgogICAgICAgIHJlZHVjZWRfYXJlYSA9IGludChzdGF0c1tjb21wb25lbnRfaWQsIGN2Mi5DQ19TVEFUX0FSRUFdKQogICAgICAgIGlmIHJlZHVjZWRfYXJlYSA8IG1pbl9yZWR1Y2VkX2FyZWE6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY29tcG9uZW50ID0gbGFiZWxzID09IGNvbXBvbmVudF9pZAogICAgICAgIGlmIGludChucC5jb3VudF9ub256ZXJvKHN0cm9uZ1tjb21wb25lbnRdKSkgPCA0OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNvbnRvdXJzLCBfID0gY3YyLmZpbmRDb250b3VycygKICAgICAgICAgICAgY29tcG9uZW50LmFzdHlwZShucC51aW50OCksIGN2Mi5SRVRSX0VYVEVSTkFMLCBjdjIuQ0hBSU5fQVBQUk9YX05PTkUKICAgICAgICApCiAgICAgICAgaWYgY29udG91cnM6CiAgICAgICAgICAgIHBlcmltZXRlciA9IG1heChmbG9hdChjdjIuYXJjTGVuZ3RoKGNvbnRvdXJzWzBdLCBUcnVlKSksIDEuMCkKICAgICAgICAgICAgY29tcGFjdG5lc3MgPSA0LjAgKiBucC5waSAqIHJlZHVjZWRfYXJlYSAvIChwZXJpbWV0ZXIgKiBwZXJpbWV0ZXIpCiAgICAgICAgICAgIGlmIGNvbXBhY3RuZXNzID4gMC43OCBhbmQgcmVkdWNlZF9hcmVhIDwgMTgwIGFuZCBmbG9hdCh6X21hcFtjb21wb25lbnRdLm1heCgpKSA+IDYuNToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHgwID0gaW50KHN0YXRzW2NvbXBvbmVudF9pZCwgY3YyLkNDX1NUQVRfTEVGVF0pCiAgICAgICAgeTAgPSBpbnQoc3RhdHNbY29tcG9uZW50X2lkLCBjdjIuQ0NfU1RBVF9UT1BdKQogICAgICAgIGJveF93aWR0aCA9IGludChzdGF0c1tjb21wb25lbnRfaWQsIGN2Mi5DQ19TVEFUX1dJRFRIXSkKICAgICAgICBib3hfaGVpZ2h0ID0gaW50KHN0YXRzW2NvbXBvbmVudF9pZCwgY3YyLkNDX1NUQVRfSEVJR0hUXSkKICAgICAgICBzbWFsbF9jcm9wID0gY29tcG9uZW50W3kwIDogeTAgKyBib3hfaGVpZ2h0LCB4MCA6IHgwICsgYm94X3dpZHRoXS5hc3R5cGUobnAudWludDgpCiAgICAgICAgZnVsbF9jcm9wID0gY3YyLnJlc2l6ZSgKICAgICAgICAgICAgc21hbGxfY3JvcCwKICAgICAgICAgICAgKGJveF93aWR0aCAqIDIsIGJveF9oZWlnaHQgKiAyKSwKICAgICAgICAgICAgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfTElORUFSLAogICAgICAgICkgPiAwCiAgICAgICAgY3JvcF95LCBjcm9wX3ggPSBucC5ub256ZXJvKGZ1bGxfY3JvcCkKICAgICAgICBmdWxsX3ggPSB4MCAqIDIgKyBpbnQoY3JvcF94Lm1pbigpKQogICAgICAgIGZ1bGxfeSA9IHkwICogMiArIGludChjcm9wX3kubWluKCkpCiAgICAgICAgdGlnaHRfY3JvcCA9IGZ1bGxfY3JvcFsKICAgICAgICAgICAgY3JvcF95Lm1pbigpIDogY3JvcF95Lm1heCgpICsgMSwKICAgICAgICAgICAgY3JvcF94Lm1pbigpIDogY3JvcF94Lm1heCgpICsgMSwKICAgICAgICBdCiAgICAgICAgYXJlYSA9IGludChucC5jb3VudF9ub256ZXJvKHRpZ2h0X2Nyb3ApKQogICAgICAgIGlmIGFyZWEgPCBjb25maWcubWluX2luc3RhbmNlX2FyZWE6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcHJlZGljdGlvbnMuYXBwZW5kKAogICAgICAgICAgICBQcmVkaWN0ZWRJbnN0YW5jZSgKICAgICAgICAgICAgICAgIGZpbGFtZW50X2lkPWYie2ltYWdlX3N0ZW19X3tsZW4ocHJlZGljdGlvbnMpICsgMX0iLAogICAgICAgICAgICAgICAgaW1hZ2Vfc3RlbT1pbWFnZV9zdGVtLAogICAgICAgICAgICAgICAgYXJlYT1hcmVhLAogICAgICAgICAgICAgICAgcmxlPWVuY29kZV9jcm9wcGVkX2JpbmFyeV9tYXNrKAogICAgICAgICAgICAgICAgICAgIHRpZ2h0X2Nyb3AsCiAgICAgICAgICAgICAgICAgICAgZnVsbF94LAogICAgICAgICAgICAgICAgICAgIGZ1bGxfeSwKICAgICAgICAgICAgICAgICAgICAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSksCiAgICAgICAgICAgICAgICApLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIHByZWRpY3Rpb25zCgoKZGVmIHByZWRpY3RfY2xhc3NpY2FsKAogICAgZ3JheV9pbWFnZTogbnAubmRhcnJheSwKICAgIGZpbGVfbmFtZTogc3RyLAogICAgY29uZmlnOiBDbGFzc2ljYWxDb25maWcgfCBOb25lID0gTm9uZSwKKSAtPiBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXToKICAgIGNvbmZpZyA9IGNvbmZpZyBvciBDbGFzc2ljYWxDb25maWcoKQogICAgel9tYXAsIHNhZmVfZGlzayA9IGNvbXB1dGVfc2FsaWVuY2Vfem1hcChncmF5X2ltYWdlLCBjb25maWcubGltYl9mcmFjdGlvbikKICAgIHJldHVybiBwcmVkaWN0aW9uc19mcm9tX3ptYXAoel9tYXAsIHNhZmVfZGlzaywgZmlsZV9uYW1lLCBjb25maWcpCg==","src/filament_baseline/classical_runner.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmNsYXNzaWNhbCBpbXBvcnQgQ2xhc3NpY2FsQ29uZmlnLCBjb21wdXRlX3NhbGllbmNlX3ptYXAsIHByZWRpY3Rpb25zX2Zyb21fem1hcApmcm9tIC5kYXRhIGltcG9ydCBpbnZlbnRvcnksIGxvYWRfZGF0YXNldApmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IHJlYWRfZ3JheXNjYWxlCmZyb20gLnJ1bm5lciBpbXBvcnQgKAogICAgX2dpdF9zdGF0dXMsCiAgICBfcmVzb2x2ZV9kYXRhX3Jvb3QsCiAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzLAogICAgX3dyaXRlX292ZXJsYXlzLAogICAgX3dyaXRlX3ByZWRpY3Rpb25zLAopCmZyb20gLnJ1bnRpbWUgaW1wb3J0IGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAuc2NvcmVyIGltcG9ydCBzY29yZV92YWxpZGF0aW9uCmZyb20gLnNwbGl0IGltcG9ydCBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQKZnJvbSAudmlzdWFscyBpbXBvcnQgd3JpdGVfY29tcGFyaXNvbl9vdmVybGF5cwoKCmRlZiBydW5fY2xhc3NpY2FsX3ZhbGlkYXRpb24oCiAgICAqLAogICAgcmVwb19yb290OiBzdHIgfCBQYXRoLAogICAgZGF0YV9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSwKICAgIHJ1bl9uYW1lOiBzdHIsCiAgICBvYnNlcnZhdGlvbl9saW1pdDogaW50IHwgTm9uZSA9IE5vbmUsCiAgICBzaW5nbGVfY29uZmlnOiBDbGFzc2ljYWxDb25maWcgfCBOb25lID0gTm9uZSwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoX3Jlc29sdmVfZGF0YV9yb290KGRhdGFfcm9vdCwgcmVwb19yb290KSkKICAgIGNvbmZpZyA9IGpzb24ubG9hZHMoKHJlcG9fcm9vdCAvICJjb25maWdzIiAvICJ2YWxpZGF0aW9uLWJhc2VsaW5lLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGNvbmZpZ19zaGEyNTYgPSBoYXNobGliLnNoYTI1NigKICAgICAgICAocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfYnl0ZXMoKQogICAgKS5oZXhkaWdlc3QoKQogICAgc3BsaXQgPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBzcGxpdC5zaGEyNTYgIT0gY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBjbGFzc2ljYWwgZXhwZXJpbWVudCByZXF1aXJlcyB0aGUgZnJvemVuIHNwbGl0IikKICAgIGFsbF92YWxpZGF0aW9uX25hbWVzID0gbGlzdChzcGxpdC52YWxpZGF0aW9uX25hbWVzKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGFsbF92YWxpZGF0aW9uX25hbWVzCiAgICBpZiBvYnNlcnZhdGlvbl9saW1pdCBpcyBub3QgTm9uZToKICAgICAgICBpZiBvYnNlcnZhdGlvbl9saW1pdCA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm9ic2VydmF0aW9uX2xpbWl0IG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIGlmIG9ic2VydmF0aW9uX2xpbWl0IDwgbGVuKGFsbF92YWxpZGF0aW9uX25hbWVzKToKICAgICAgICAgICAgaWYgb2JzZXJ2YXRpb25fbGltaXQgPT0gMToKICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fbmFtZXMgPSBbYWxsX3ZhbGlkYXRpb25fbmFtZXNbbGVuKGFsbF92YWxpZGF0aW9uX25hbWVzKSAvLyAyXV0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGluZGljZXMgPSBzb3J0ZWQoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICByb3VuZCgKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluZGV4ICogKGxlbihhbGxfdmFsaWRhdGlvbl9uYW1lcykgLSAxKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgLyAob2JzZXJ2YXRpb25fbGltaXQgLSAxKQogICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpbmRleCBpbiByYW5nZShvYnNlcnZhdGlvbl9saW1pdCkKICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICB2YWxpZGF0aW9uX25hbWVzID0gW2FsbF92YWxpZGF0aW9uX25hbWVzW2luZGV4XSBmb3IgaW5kZXggaW4gaW5kaWNlc10KICAgIHJ1bl9yb290ID0gcmVwb19yb290IC8gImFydGlmYWN0cyIgLyAiY2xhc3NpY2FsLWV4cGVyaW1lbnRzIiAvIHJ1bl9uYW1lCiAgICBydW5fcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByZXBvc2l0b3J5X2luZm8gPSBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3QpCiAgICBjYWNoZV9kaXIgPSBydW5fcm9vdCAvICJzYWxpZW5jZS16bWFwcyIKICAgIGNhY2hlX2Rpci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGZvciBpbmRleCwgZmlsZV9uYW1lIGluIGVudW1lcmF0ZSh2YWxpZGF0aW9uX25hbWVzLCBzdGFydD0xKToKICAgICAgICBjYWNoZV9wYXRoID0gY2FjaGVfZGlyIC8gZiJ7UGF0aChmaWxlX25hbWUpLnN0ZW19Lm5weiIKICAgICAgICBpZiBjYWNoZV9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpbWFnZSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIHpfbWFwLCBkaXNrID0gY29tcHV0ZV9zYWxpZW5jZV96bWFwKGltYWdlKQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgICAgIGNhY2hlX3BhdGgsCiAgICAgICAgICAgIHpfbWFwPXpfbWFwLmFzdHlwZShucC5mbG9hdDE2KSwKICAgICAgICAgICAgZGlzaz1kaXNrLmFzdHlwZShucC51aW50OCksCiAgICAgICAgKQogICAgICAgIGlmIGluZGV4ICUgMjAgPT0gMCBvciBpbmRleCA9PSBsZW4odmFsaWRhdGlvbl9uYW1lcyk6CiAgICAgICAgICAgIHByaW50KGYiQ2FjaGVkIHtpbmRleH0ve2xlbih2YWxpZGF0aW9uX25hbWVzKX0gY2xhc3NpY2FsIHNhbGllbmNlIG1hcHMiLCBmbHVzaD1UcnVlKQogICAgcHJlcHJvY2Vzc2luZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKCiAgICBjYW5kaWRhdGVzID0gKAogICAgICAgIFtzaW5nbGVfY29uZmlnXQogICAgICAgIGlmIHNpbmdsZV9jb25maWcgaXMgbm90IE5vbmUKICAgICAgICBlbHNlIFsKICAgICAgICAgICAgQ2xhc3NpY2FsQ29uZmlnKAogICAgICAgICAgICAgICAgel90aHJlc2hvbGQ9el90aHJlc2hvbGQsCiAgICAgICAgICAgICAgICBicmlkZ2Vfc2l6ZT1icmlkZ2Vfc2l6ZSwKICAgICAgICAgICAgICAgIG1pbl9pbnN0YW5jZV9hcmVhPW1pbl9hcmVhLAogICAgICAgICAgICApCiAgICAgICAgICAgIGZvciB6X3RocmVzaG9sZCwgYnJpZGdlX3NpemUsIG1pbl9hcmVhIGluIGl0ZXJ0b29scy5wcm9kdWN0KAogICAgICAgICAgICAgICAgKDIuOSwgMy4yKSwgKDUsIDksIDEzKSwgKDI1MCwgNTAwKQogICAgICAgICAgICApCiAgICAgICAgXQogICAgKQogICAgc3dlZXBfcmVzdWx0cyA9IFtdCiAgICBiZXN0X3BxID0gLTEuMAogICAgc2NvcmluZ19zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIGluZGV4LCBjYW5kaWRhdGUgaW4gZW51bWVyYXRlKGNhbmRpZGF0ZXMsIHN0YXJ0PTEpOgogICAgICAgIHByZWRpY3Rpb25zID0ge30KICAgICAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgICAgIGNhY2hlX3BhdGggPSBjYWNoZV9kaXIgLyBmIntQYXRoKGZpbGVfbmFtZSkuc3RlbX0ubnB6IgogICAgICAgICAgICB3aXRoIG5wLmxvYWQoY2FjaGVfcGF0aCkgYXMgY2FjaGVkOgogICAgICAgICAgICAgICAgcHJlZGljdGlvbnNbZmlsZV9uYW1lXSA9IHByZWRpY3Rpb25zX2Zyb21fem1hcCgKICAgICAgICAgICAgICAgICAgICBjYWNoZWRbInpfbWFwIl0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgICAgIGNhY2hlZFsiZGlzayJdLmFzdHlwZShib29sKSwKICAgICAgICAgICAgICAgICAgICBmaWxlX25hbWUsCiAgICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlLAogICAgICAgICAgICAgICAgKQogICAgICAgIG1ldHJpY3MsIHJvd3MgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgICAgICBkYXRhc2V0LAogICAgICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdChjb25maWdbImlvdV90aHJlc2hvbGQiXSksCiAgICAgICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0pLAogICAgICAgICkKICAgICAgICBnbG9iYWxfbWV0cmljcyA9IG1ldHJpY3NbImdsb2JhbCJdCiAgICAgICAgcm93ID0gewogICAgICAgICAgICAiY29uZmlnIjogY2FuZGlkYXRlLl9fZGljdF9fLAogICAgICAgICAgICAicHEiOiBnbG9iYWxfbWV0cmljc1sicHEiXSwKICAgICAgICAgICAgInRwIjogZ2xvYmFsX21ldHJpY3NbInRwIl0sCiAgICAgICAgICAgICJmcCI6IGdsb2JhbF9tZXRyaWNzWyJmcCJdLAogICAgICAgICAgICAiZm4iOiBnbG9iYWxfbWV0cmljc1siZm4iXSwKICAgICAgICAgICAgInByZWRpY3Rpb25zX2Fjcm9zc192aWV3cyI6IGdsb2JhbF9tZXRyaWNzWwogICAgICAgICAgICAgICAgInByZWRpY3Rpb25fY291bnRfYWNyb3NzX2Fubm90YXRvcl92aWV3cyIKICAgICAgICAgICAgXSwKICAgICAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLAogICAgICAgIH0KICAgICAgICBzd2VlcF9yZXN1bHRzLmFwcGVuZChyb3cpCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiW3tpbmRleH0ve2xlbihjYW5kaWRhdGVzKX1dIHo9e2NhbmRpZGF0ZS56X3RocmVzaG9sZDouMmZ9ICIKICAgICAgICAgICAgZiJicmlkZ2U9e2NhbmRpZGF0ZS5icmlkZ2Vfc2l6ZX0gYXJlYT17Y2FuZGlkYXRlLm1pbl9pbnN0YW5jZV9hcmVhfSAiCiAgICAgICAgICAgIGYiUFE9e3Jvd1sncHEnXTouNmZ9IFRQPXtyb3dbJ3RwJ119IEZQPXtyb3dbJ2ZwJ119IEZOPXtyb3dbJ2ZuJ119IiwKICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICApCiAgICAgICAgaWYgcm93WyJwcSJdID4gYmVzdF9wcToKICAgICAgICAgICAgYmVzdF9wcSA9IHJvd1sicHEiXQogICAgICAgICAgICBiZXN0X3Jvb3QgPSBydW5fcm9vdCAvICgKICAgICAgICAgICAgICAgICJiZXN0LXNjcmVlbiIgaWYgb2JzZXJ2YXRpb25fbGltaXQgaXMgbm90IE5vbmUgZWxzZSAiYmVzdCIKICAgICAgICAgICAgKQogICAgICAgICAgICBiZXN0X3Jvb3QubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgKGJlc3Rfcm9vdCAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7InNwbGl0X3NoYTI1NiI6IHNwbGl0LnNoYTI1NiwgImNvbmZpZyI6IGNhbmRpZGF0ZS5fX2RpY3RfXywgIm1ldHJpY3MiOiBtZXRyaWNzfSwKICAgICAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICsgIlxuIiwKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKICAgICAgICAgICAgX3dyaXRlX3ByZWRpY3Rpb25zKGJlc3Rfcm9vdCAvICJwcmVkaWN0aW9ucy5jc3YiLCBwcmVkaWN0aW9ucykKICAgICAgICAgICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcyhiZXN0X3Jvb3QgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIHJvd3MsIHByZWRpY3Rpb25zKQogICAgICAgICAgICBfd3JpdGVfb3ZlcmxheXMoZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3Rfcm9vdCAvICJvdmVybGF5cyIsIDEyKQogICAgICAgICAgICB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzKAogICAgICAgICAgICAgICAgZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3Rfcm9vdCAvICJjb21wYXJpc29uLW92ZXJsYXlzIiwgMTIKICAgICAgICAgICAgKQogICAgc2NvcmluZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNjb3Jpbmdfc3RhcnRlZAoKICAgIHN3ZWVwX3Jlc3VsdHMuc29ydChrZXk9bGFtYmRhIGl0ZW06IGl0ZW1bInBxIl0sIHJldmVyc2U9VHJ1ZSkKICAgIHN3ZWVwX3N0ZW0gPSAic2NyZWVuLXN3ZWVwIiBpZiBvYnNlcnZhdGlvbl9saW1pdCBpcyBub3QgTm9uZSBlbHNlICJzd2VlcCIKICAgIChydW5fcm9vdCAvIGYie3N3ZWVwX3N0ZW19Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMoc3dlZXBfcmVzdWx0cywgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHdpdGggKHJ1bl9yb290IC8gZiJ7c3dlZXBfc3RlbX0uY3N2Iikub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICBmaWVsZHMgPSAoCiAgICAgICAgICAgICpDbGFzc2ljYWxDb25maWcoKS5fX2RpY3RfXy5rZXlzKCksCiAgICAgICAgICAgICJwcSIsCiAgICAgICAgICAgICJ0cCIsCiAgICAgICAgICAgICJmcCIsCiAgICAgICAgICAgICJmbiIsCiAgICAgICAgKQogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1maWVsZHMsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHN3ZWVwX3Jlc3VsdHM6CiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAqKnJvd1siY29uZmlnIl0sCiAgICAgICAgICAgICAgICAgICAgKip7CiAgICAgICAgICAgICAgICAgICAgICAgIGZpZWxkOiByb3dbZmllbGRdCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBmaWVsZCBpbiBmaWVsZHNbbGVuKHJvd1siY29uZmlnIl0pIDpdCiAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQoKICAgIHJlc3VsdDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIgaWYgb2JzZXJ2YXRpb25fbGltaXQgaXMgTm9uZSBlbHNlICJ2YWxpZGF0aW9uX3N1YnNldCIsCiAgICAgICAgInJ1bl9uYW1lIjogcnVuX25hbWUsCiAgICAgICAgInJlcG9zaXRvcnkiOiByZXBvc2l0b3J5X2luZm8sCiAgICAgICAgImNvbmZpZ3VyYXRpb25fc2hhMjU2IjogY29uZmlnX3NoYTI1NiwKICAgICAgICAiZGF0YXNldF9pbnZlbnRvcnkiOiBpbnZlbnRvcnkoZGF0YXNldCksCiAgICAgICAgInNwbGl0X3NoYTI1NiI6IHNwbGl0LnNoYTI1NiwKICAgICAgICAidmFsaWRhdGlvbl9vYnNlcnZhdGlvbnMiOiBsZW4odmFsaWRhdGlvbl9uYW1lcyksCiAgICAgICAgImZ1bGxfaG9sZG91dF9vYnNlcnZhdGlvbnMiOiBsZW4oYWxsX3ZhbGlkYXRpb25fbmFtZXMpLAogICAgICAgICJtZXRob2QiOiAiaW5kZXBlbmRlbnQgaW1wbGVtZW50YXRpb24gb2YgcHVibGljIHJhZGlhbC1ub3JtYWxpemF0aW9uLCBtdWx0aXNjYWxlIGJsYWNrLWhhdCwgcm9idXN0IHotc2NvcmUsIGh5c3RlcmVzaXMsIGFuZCBjb21wb25lbnQgZmlsdGVyaW5nIGFwcHJvYWNoIiwKICAgICAgICAicmVmZXJlbmNlIjogImh0dHBzOi8vd3d3LmthZ2dsZS5jb20vY29kZS9yb2JpbmFtaXJiYWhhci9zb2xhci1maWxhbWVudC1wYW5vcHRpYy1tb3JwaG9sb2d5LXNlZ21lbnRhdGlvbiIsCiAgICAgICAgImJlc3QiOiBzd2VlcF9yZXN1bHRzWzBdLAogICAgICAgICJwcmVwcm9jZXNzaW5nX3NlY29uZHMiOiBwcmVwcm9jZXNzaW5nX3NlY29uZHMsCiAgICAgICAgInN3ZWVwX2FuZF9zY29yaW5nX3NlY29uZHMiOiBzY29yaW5nX3NlY29uZHMsCiAgICAgICAgInJ1bnRpbWUiOiB7KipoYXJkd2FyZV9pbmZvKCksICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpfSwKICAgICAgICAiYXJ0aWZhY3RzIjogewogICAgICAgICAgICAic3dlZXBfY3N2IjogZiJ7c3dlZXBfc3RlbX0uY3N2IiwKICAgICAgICAgICAgInN3ZWVwX2pzb24iOiBmIntzd2VlcF9zdGVtfS5qc29uIiwKICAgICAgICAgICAgImJlc3RfZGlyIjogImJlc3Qtc2NyZWVuIiBpZiBvYnNlcnZhdGlvbl9saW1pdCBpcyBub3QgTm9uZSBlbHNlICJiZXN0IiwKICAgICAgICB9LAogICAgfQogICAgKHJ1bl9yb290IC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludCgKICAgICAgICBmIkJlc3QgY2xhc3NpY2FsIFBRIG9uIHtsZW4odmFsaWRhdGlvbl9uYW1lcyl9IG9ic2VydmF0aW9uczoge3N3ZWVwX3Jlc3VsdHNbMF1bJ3BxJ106LjZmfSAiCiAgICAgICAgZiJ3aXRoIHtzd2VlcF9yZXN1bHRzWzBdWydjb25maWcnXX07IG91dHB1dHMge3J1bl9yb290fSIsCiAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICkKICAgIHJldHVybiByZXN1bHQK","src/filament_baseline/data.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKQU5OT1RBVElPTl9GSUxFTkFNRSA9ICJNQUdGaUxPXzEuMF9Bbm5vdGF0aW9uc19rYWdnbGUyMDI2X3RyYWluLmpzb24iCklNQUdFX1NJWkUgPSAyMDQ4CgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSW1hZ2VWaWV3OgogICAgaW1hZ2VfaWQ6IHN0cgogICAgZmlsZV9uYW1lOiBzdHIKICAgIHdpZHRoOiBpbnQKICAgIGhlaWdodDogaW50CiAgICB5ZWFyOiBpbnQKICAgIHNpdGU6IHN0cgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIENvbXBldGl0aW9uRGF0YXNldDoKICAgIHJvb3Q6IFBhdGgKICAgIHRyYWluX2ltYWdlX2RpcjogUGF0aAogICAgdGVzdF9pbWFnZV9kaXI6IFBhdGgKICAgIGFubm90YXRpb25fcGF0aDogUGF0aAogICAgaW1hZ2Vfdmlld3M6IHR1cGxlW0ltYWdlVmlldywgLi4uXQogICAgYW5ub3RhdGlvbnNfYnlfaW1hZ2VfaWQ6IGRpY3Rbc3RyLCB0dXBsZVtkaWN0W3N0ciwgQW55XSwgLi4uXV0KICAgIHZpZXdzX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgdHVwbGVbSW1hZ2VWaWV3LCAuLi5dXQogICAgY2F0ZWdvcmllczogdHVwbGVbZGljdFtzdHIsIEFueV0sIC4uLl0KCiAgICBAcHJvcGVydHkKICAgIGRlZiBvYnNlcnZhdGlvbl9uYW1lcyhzZWxmKSAtPiB0dXBsZVtzdHIsIC4uLl06CiAgICAgICAgcmV0dXJuIHR1cGxlKHNvcnRlZChzZWxmLnZpZXdzX2J5X29ic2VydmF0aW9uKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0ZXN0X2ltYWdlX2NvdW50KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc3VtKHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIHNlbGYudGVzdF9pbWFnZV9kaXIuaXRlcmRpcigpKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGFubm90YXRpb25fY291bnQoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzdW0obGVuKGl0ZW1zKSBmb3IgaXRlbXMgaW4gc2VsZi5hbm5vdGF0aW9uc19ieV9pbWFnZV9pZC52YWx1ZXMoKSkKCgpkZWYgX2xvb2tzX2xpa2VfZGF0YXNldChyb290OiBQYXRoKSAtPiBib29sOgogICAgcmV0dXJuICgKICAgICAgICAocm9vdCAvICJ0cmFpbiIgLyBBTk5PVEFUSU9OX0ZJTEVOQU1FKS5pc19maWxlKCkKICAgICAgICBhbmQgKHJvb3QgLyAidHJhaW4iIC8gInRyYWluX2ltYWdlcyIpLmlzX2RpcigpCiAgICAgICAgYW5kIChyb290IC8gInRlc3QiIC8gInRlc3RfaW1hZ2VzIikuaXNfZGlyKCkKICAgICkKCgpkZWYgZmluZF9kYXRhc2V0X3Jvb3Qoc2VhcmNoX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJGaW5kIHRoZSBjb21wZXRpdGlvbiBpbnB1dCBmb2xkZXIgd2l0aG91dCByZWx5aW5nIG9uIGEgZml4ZWQgS2FnZ2xlIG1vdW50IG5hbWUuIiIiCiAgICBzZWFyY2hfcm9vdCA9IFBhdGgoc2VhcmNoX3Jvb3QpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBzZWFyY2hfcm9vdC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkRhdGEgc2VhcmNoIHBhdGggZG9lcyBub3QgZXhpc3Q6IHtzZWFyY2hfcm9vdH0iKQogICAgaWYgX2xvb2tzX2xpa2VfZGF0YXNldChzZWFyY2hfcm9vdCk6CiAgICAgICAgcmV0dXJuIHNlYXJjaF9yb290CgogICAgbWF0Y2hlcyA9IHNvcnRlZCgKICAgICAgICB7cGF0aC5wYXJlbnQucGFyZW50LnJlc29sdmUoKSBmb3IgcGF0aCBpbiBzZWFyY2hfcm9vdC5yZ2xvYihBTk5PVEFUSU9OX0ZJTEVOQU1FKX0KICAgICAgICBpZiBzZWFyY2hfcm9vdC5pc19kaXIoKQogICAgICAgIGVsc2Ugc2V0KCkKICAgICkKICAgIG1hdGNoZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiBtYXRjaGVzIGlmIF9sb29rc19saWtlX2RhdGFzZXQocGF0aCldCiAgICBpZiBsZW4obWF0Y2hlcykgPT0gMToKICAgICAgICByZXR1cm4gbWF0Y2hlc1swXQogICAgaWYgbm90IG1hdGNoZXM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiQ291bGQgbm90IGZpbmQge0FOTk9UQVRJT05fRklMRU5BTUV9IHdpdGggdHJhaW4vdHJhaW5faW1hZ2VzIGFuZCAiCiAgICAgICAgICAgIGYidGVzdC90ZXN0X2ltYWdlcyBiZWxvdyB7c2VhcmNoX3Jvb3R9IgogICAgICAgICkKICAgIHJlbmRlcmVkID0gIlxuIi5qb2luKHN0cihwYXRoKSBmb3IgcGF0aCBpbiBtYXRjaGVzKQogICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICJNb3JlIHRoYW4gb25lIGNvbXBldGl0aW9uIGRhdGFzZXQgd2FzIGZvdW5kOyBwYXNzIGFuIGV4cGxpY2l0IGRhdGFzZXQgcm9vdDpcbiIKICAgICAgICBmIntyZW5kZXJlZH0iCiAgICApCgoKZGVmIF95ZWFyX2FuZF9zaXRlKGZpbGVfbmFtZTogc3RyKSAtPiB0dXBsZVtpbnQsIHN0cl06CiAgICBzdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIGlmIGxlbihzdGVtKSA8IDQgb3Igbm90IHN0ZW1bOjRdLmlzZGlnaXQoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2Fubm90IGRlcml2ZSBjYXB0dXJlIHllYXIgZnJvbSBpbWFnZSBmaWxlbmFtZToge2ZpbGVfbmFtZX0iKQogICAgeWVhciA9IGludChzdGVtWzo0XSkKICAgIHNpdGVfbWF0Y2ggPSByZS5mdWxsbWF0Y2gociJcZHsxNH0oW0EtWmEtel17Mn0pIiwgc3RlbSkKICAgIHNpdGUgPSBzaXRlX21hdGNoLmdyb3VwKDEpIGlmIHNpdGVfbWF0Y2ggZWxzZSAidW5rbm93biIKICAgIHJldHVybiB5ZWFyLCBzaXRlCgoKZGVmIGxvYWRfZGF0YXNldChkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IENvbXBldGl0aW9uRGF0YXNldDoKICAgIHJvb3QgPSBmaW5kX2RhdGFzZXRfcm9vdChkYXRhX3Jvb3QpCiAgICBhbm5vdGF0aW9uX3BhdGggPSByb290IC8gInRyYWluIiAvIEFOTk9UQVRJT05fRklMRU5BTUUKICAgIHRyYWluX2ltYWdlX2RpciA9IHJvb3QgLyAidHJhaW4iIC8gInRyYWluX2ltYWdlcyIKICAgIHRlc3RfaW1hZ2VfZGlyID0gcm9vdCAvICJ0ZXN0IiAvICJ0ZXN0X2ltYWdlcyIKICAgIHBheWxvYWQgPSBqc29uLmxvYWRzKGFubm90YXRpb25fcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCgogICAgaW1hZ2Vfdmlld3M6IGxpc3RbSW1hZ2VWaWV3XSA9IFtdCiAgICBpbWFnZV9pZHM6IHNldFtzdHJdID0gc2V0KCkKICAgIHZpZXdzX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgbGlzdFtJbWFnZVZpZXddXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3IgcmVjb3JkIGluIHBheWxvYWQuZ2V0KCJpbWFnZXMiLCBbXSk6CiAgICAgICAgaW1hZ2VfaWQgPSBzdHIocmVjb3JkWyJpZCJdKQogICAgICAgIGZpbGVfbmFtZSA9IHN0cihyZWNvcmRbImZpbGVfbmFtZSJdKQogICAgICAgIGlmIFBhdGgoZmlsZV9uYW1lKS5uYW1lICE9IGZpbGVfbmFtZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGEgYmFzZW5hbWUgaW4gQ09DTyBmaWxlX25hbWUsIGdvdDoge2ZpbGVfbmFtZX0iKQogICAgICAgIGlmIGltYWdlX2lkIGluIGltYWdlX2lkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkR1cGxpY2F0ZSBDT0NPIGltYWdlIGlkOiB7aW1hZ2VfaWR9IikKICAgICAgICBpbWFnZV9pZHMuYWRkKGltYWdlX2lkKQogICAgICAgIHllYXIsIHNpdGUgPSBfeWVhcl9hbmRfc2l0ZShmaWxlX25hbWUpCiAgICAgICAgdmlldyA9IEltYWdlVmlldygKICAgICAgICAgICAgaW1hZ2VfaWQ9aW1hZ2VfaWQsCiAgICAgICAgICAgIGZpbGVfbmFtZT1maWxlX25hbWUsCiAgICAgICAgICAgIHdpZHRoPWludChyZWNvcmRbIndpZHRoIl0pLAogICAgICAgICAgICBoZWlnaHQ9aW50KHJlY29yZFsiaGVpZ2h0Il0pLAogICAgICAgICAgICB5ZWFyPXllYXIsCiAgICAgICAgICAgIHNpdGU9c2l0ZSwKICAgICAgICApCiAgICAgICAgaWYgdmlldy53aWR0aCAhPSBJTUFHRV9TSVpFIG9yIHZpZXcuaGVpZ2h0ICE9IElNQUdFX1NJWkU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0sIGdvdCB7dmlldy53aWR0aH14e3ZpZXcuaGVpZ2h0fSAiCiAgICAgICAgICAgICAgICBmImZvciB7ZmlsZV9uYW1lfSIKICAgICAgICAgICAgKQogICAgICAgIGltYWdlX3ZpZXdzLmFwcGVuZCh2aWV3KQogICAgICAgIHZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV0uYXBwZW5kKHZpZXcpCgogICAgYW5ub3RhdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W2RpY3Rbc3RyLCBBbnldXV0gPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIGFubm90YXRpb24gaW4gcGF5bG9hZC5nZXQoImFubm90YXRpb25zIiwgW10pOgogICAgICAgIGltYWdlX2lkID0gc3RyKGFubm90YXRpb25bImltYWdlX2lkIl0pCiAgICAgICAgaWYgaW1hZ2VfaWQgbm90IGluIGltYWdlX2lkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkFubm90YXRpb24gcmVmZXJlbmNlcyB1bmtub3duIGltYWdlIGlkOiB7aW1hZ2VfaWR9IikKICAgICAgICBpZiBpbnQoYW5ub3RhdGlvbi5nZXQoImlzY3Jvd2QiLCAwKSkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhlIGNvbXBldGl0aW9uIHRyYWluaW5nIGFubm90YXRpb25zIHVuZXhwZWN0ZWRseSBjb250YWluIGNyb3dkIG1hc2tzIikKICAgICAgICBhbm5vdGF0aW9uc1tpbWFnZV9pZF0uYXBwZW5kKGFubm90YXRpb24pCgogICAgZm9yIGltYWdlX2lkIGluIGltYWdlX2lkczoKICAgICAgICBhbm5vdGF0aW9uc1tpbWFnZV9pZF0uc29ydChrZXk9bGFtYmRhIGl0ZW06IHN0cihpdGVtWyJpZCJdKSkKICAgICAgICBpZiBub3QgYW5ub3RhdGlvbnNbaW1hZ2VfaWRdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQW5ub3RhdG9yIHZpZXcge2ltYWdlX2lkfSBoYXMgbm8gZmlsYW1lbnQgYW5ub3RhdGlvbnMiKQoKICAgIGZvciBmaWxlX25hbWUsIHZpZXdzIGluIHZpZXdzX2J5X29ic2VydmF0aW9uLml0ZW1zKCk6CiAgICAgICAgZXhwZWN0ZWRfcGF0aCA9IHRyYWluX2ltYWdlX2RpciAvIGZpbGVfbmFtZQogICAgICAgIGlmIG5vdCBleHBlY3RlZF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDT0NPIGltYWdlIGZpbGUgaXMgbWlzc2luZzoge2V4cGVjdGVkX3BhdGh9IikKICAgICAgICB2aWV3cy5zb3J0KGtleT1sYW1iZGEgdmlldzogdmlldy5pbWFnZV9pZCkKCiAgICB0ZXN0X2ZpbGVzID0gW3BhdGggZm9yIHBhdGggaW4gdGVzdF9pbWFnZV9kaXIuaXRlcmRpcigpIGlmIHBhdGguaXNfZmlsZSgpXQogICAgaWYgbm90IHRlc3RfZmlsZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk5vIHRlc3QgaW1hZ2VzIGZvdW5kIGluIHt0ZXN0X2ltYWdlX2Rpcn0iKQoKICAgIHJldHVybiBDb21wZXRpdGlvbkRhdGFzZXQoCiAgICAgICAgcm9vdD1yb290LAogICAgICAgIHRyYWluX2ltYWdlX2Rpcj10cmFpbl9pbWFnZV9kaXIsCiAgICAgICAgdGVzdF9pbWFnZV9kaXI9dGVzdF9pbWFnZV9kaXIsCiAgICAgICAgYW5ub3RhdGlvbl9wYXRoPWFubm90YXRpb25fcGF0aCwKICAgICAgICBpbWFnZV92aWV3cz10dXBsZShzb3J0ZWQoaW1hZ2Vfdmlld3MsIGtleT1sYW1iZGEgdmlldzogKHZpZXcuZmlsZV9uYW1lLCB2aWV3LmltYWdlX2lkKSkpLAogICAgICAgIGFubm90YXRpb25zX2J5X2ltYWdlX2lkPXsKICAgICAgICAgICAgaW1hZ2VfaWQ6IHR1cGxlKGl0ZW1zKSBmb3IgaW1hZ2VfaWQsIGl0ZW1zIGluIGFubm90YXRpb25zLml0ZW1zKCkKICAgICAgICB9LAogICAgICAgIHZpZXdzX2J5X29ic2VydmF0aW9uPXsKICAgICAgICAgICAgZmlsZV9uYW1lOiB0dXBsZSh2aWV3cykKICAgICAgICAgICAgZm9yIGZpbGVfbmFtZSwgdmlld3MgaW4gc29ydGVkKHZpZXdzX2J5X29ic2VydmF0aW9uLml0ZW1zKCkpCiAgICAgICAgfSwKICAgICAgICBjYXRlZ29yaWVzPXR1cGxlKHBheWxvYWQuZ2V0KCJjYXRlZ29yaWVzIiwgW10pKSwKICAgICkKCgpkZWYgaW52ZW50b3J5KGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCkgLT4gZGljdFtzdHIsIGludF06CiAgICByZXR1cm4gewogICAgICAgICJvYnNlcnZhdGlvbl9jb3VudCI6IGxlbihkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uKSwKICAgICAgICAiYW5ub3RhdG9yX3ZpZXdfY291bnQiOiBsZW4oZGF0YXNldC5pbWFnZV92aWV3cyksCiAgICAgICAgImFubm90YXRpb25fY291bnQiOiBkYXRhc2V0LmFubm90YXRpb25fY291bnQsCiAgICAgICAgInRyYWluX2ltYWdlX2ZpbGVfY291bnQiOiBzdW0oCiAgICAgICAgICAgIHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyLml0ZXJkaXIoKQogICAgICAgICksCiAgICAgICAgInRlc3RfaW1hZ2VfZmlsZV9jb3VudCI6IGRhdGFzZXQudGVzdF9pbWFnZV9jb3VudCwKICAgIH0K","src/filament_baseline/gpu_smoke.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCgpkZWYgcnVuX2dwdV9zbW9rZShvdXRwdXRfcGF0aDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGltcG9ydCB0b3JjaAoKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgIlB5VG9yY2ggY2Fubm90IHNlZSBhIENVREEgR1BVLiBDb25maXJtIHRoZSBDVURBLWVuYWJsZWQgd2hlZWwgYW5kIE5WSURJQSBkcml2ZXIuIgogICAgICAgICkKICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiKQogICAgbW9kZWwgPSB0b3JjaC5ubi5Db252MmQoMSwgOCwga2VybmVsX3NpemU9MywgcGFkZGluZz0xKS50byhkZXZpY2UpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTFlLTMpCiAgICBpbWFnZSA9IHRvcmNoLnJhbmRuKDEsIDEsIDI1NiwgMjU2LCBkZXZpY2U9ZGV2aWNlKQogICAgdGFyZ2V0ID0gdG9yY2gucmFuZG4oMSwgOCwgMjU2LCAyNTYsIGRldmljZT1kZXZpY2UpCgogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9ImN1ZGEiLCBkdHlwZT10b3JjaC5mbG9hdDE2KToKICAgICAgICBwcmVkaWN0aW9uID0gbW9kZWwoaW1hZ2UpCiAgICAgICAgbG9zcyA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwubXNlX2xvc3MocHJlZGljdGlvbiwgdGFyZ2V0KQogICAgbG9zcy5iYWNrd2FyZCgpCiAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAoKICAgIHByb3BzID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic3RhdHVzIjogInBhc3NlZCIsCiAgICAgICAgInRvcmNoX3ZlcnNpb24iOiB0b3JjaC5fX3ZlcnNpb25fXywKICAgICAgICAidG9yY2hfY3VkYV92ZXJzaW9uIjogdG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgICJkZXZpY2VfbmFtZSI6IHByb3BzLm5hbWUsCiAgICAgICAgImRldmljZV9tZW1vcnlfYnl0ZXMiOiBpbnQocHJvcHMudG90YWxfbWVtb3J5KSwKICAgICAgICAiY29tcHV0ZV9jYXBhYmlsaXR5IjogZiJ7cHJvcHMubWFqb3J9Lntwcm9wcy5taW5vcn0iLAogICAgICAgICJsb3NzIjogZmxvYXQobG9zcy5kZXRhY2goKS5jcHUoKSksCiAgICAgICAgImVsYXBzZWRfc2Vjb25kcyI6IGVsYXBzZWQsCiAgICAgICAgInBlYWtfbWVtb3J5X2J5dGVzIjogaW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoMCkpLAogICAgICAgICJzaGFwZSI6IFsxLCAxLCAyNTYsIDI1Nl0sCiAgICAgICAgImFtcCI6ICJmbG9hdDE2IiwKICAgIH0KICAgIGlmIG91dHB1dF9wYXRoIGlzIG5vdCBOb25lOgogICAgICAgIG91dHB1dF9wYXRoID0gUGF0aChvdXRwdXRfcGF0aCkKICAgICAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIG91dHB1dF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgaW1wb3J0IGFyZ3BhcnNlCgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlZlcmlmeSBsb2NhbCBDVURBIGZvcndhcmQvYmFja3dhcmQgc3VwcG9ydC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1vdXRwdXQiLAogICAgICAgIGRlZmF1bHQ9ImFydGlmYWN0cy9sb2NhbC1ncHUtc21va2UvZ3B1X3Ntb2tlLmpzb24iLAogICAgICAgIGhlbHA9Ik91dHB1dCBKU09OIHBhdGg7IGtlZXAgaXQgaW5zaWRlIHRoZSByZXBvc2l0b3J5LiIsCiAgICApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmV0dXJuIDAgaWYgcnVuX2dwdV9zbW9rZShhcmdzLm91dHB1dClbInN0YXR1cyJdID09ICJwYXNzZWQiIGVsc2UgMQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK","src/filament_baseline/inference.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gc2NpcHkgaW1wb3J0IG5kaW1hZ2UgYXMgbmRpCmZyb20gc2tpbWFnZS5zZWdtZW50YXRpb24gaW1wb3J0IHdhdGVyc2hlZAoKZnJvbSAuZGF0YSBpbXBvcnQgSU1BR0VfU0laRQpmcm9tIC5tb2RlbCBpbXBvcnQgRmlsYW1lbnRVTmV0CmZyb20gLnByZWRpY3RvciBpbXBvcnQgUHJlZGljdGVkSW5zdGFuY2UsIGVuY29kZV9sYWJlbF9jb21wb25lbnRzCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSW5zdGFuY2VDb25maWc6CiAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZDogZmxvYXQgPSAwLjM1CiAgICBib3VuZGFyeV90aHJlc2hvbGQ6IGZsb2F0ID0gMC41CiAgICBib3VuZGFyeV93ZWlnaHQ6IGZsb2F0ID0gMS41CiAgICBzZWVkX3RocmVzaG9sZDogZmxvYXQgPSAwLjQ1CiAgICBzZWVkX21pbl9hcmVhOiBpbnQgPSA4CiAgICBtaW5faW5zdGFuY2VfYXJlYTogaW50ID0gNTAKICAgIHNvbGFyX3JhZGl1czogaW50ID0gOTUwCgoKZGVmIF90aWxlX3N0YXJ0cyhsZW5ndGg6IGludCwgdGlsZV9zaXplOiBpbnQsIHN0cmlkZTogaW50KSAtPiBsaXN0W2ludF06CiAgICBpZiB0aWxlX3NpemUgPiBsZW5ndGggb3Igc3RyaWRlIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0aWxlIHNpemUgbXVzdCBmaXQgdGhlIGltYWdlIGFuZCBzdHJpZGUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBzdGFydHMgPSBsaXN0KHJhbmdlKDAsIGxlbmd0aCAtIHRpbGVfc2l6ZSArIDEsIHN0cmlkZSkpCiAgICBmaW5hbCA9IGxlbmd0aCAtIHRpbGVfc2l6ZQogICAgaWYgbm90IHN0YXJ0cyBvciBzdGFydHNbLTFdICE9IGZpbmFsOgogICAgICAgIHN0YXJ0cy5hcHBlbmQoZmluYWwpCiAgICByZXR1cm4gc3RhcnRzCgoKZGVmIF90cmFuc2Zvcm0odGVuc29yOiB0b3JjaC5UZW5zb3IsIG9wZXJhdGlvbjogaW50KSAtPiB0b3JjaC5UZW5zb3I6CiAgICBpZiBvcGVyYXRpb24gPT0gMToKICAgICAgICByZXR1cm4gdG9yY2guZmxpcCh0ZW5zb3IsIGRpbXM9KC0xLCkpCiAgICBpZiBvcGVyYXRpb24gPT0gMjoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAodGVuc29yLCAyLCBkaW1zPSgtMiwgLTEpKQogICAgaWYgb3BlcmF0aW9uID09IDM6CiAgICAgICAgcmV0dXJuIHRvcmNoLmZsaXAodGVuc29yLCBkaW1zPSgtMiwpKQogICAgaWYgb3BlcmF0aW9uID09IDQ6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJvdDkwKHRlbnNvciwgMSwgZGltcz0oLTIsIC0xKSkKICAgIGlmIG9wZXJhdGlvbiA9PSA1OgogICAgICAgIHJldHVybiB0b3JjaC5yb3Q5MCh0ZW5zb3IsIDMsIGRpbXM9KC0yLCAtMSkpCiAgICBpZiBvcGVyYXRpb24gPT0gNjoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAodG9yY2guZmxpcCh0ZW5zb3IsIGRpbXM9KC0xLCkpLCAxLCBkaW1zPSgtMiwgLTEpKQogICAgaWYgb3BlcmF0aW9uID09IDc6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJvdDkwKHRvcmNoLmZsaXAodGVuc29yLCBkaW1zPSgtMSwpKSwgMywgZGltcz0oLTIsIC0xKSkKICAgIHJldHVybiB0ZW5zb3IKCgpkZWYgX2ludmVyc2VfdHJhbnNmb3JtKHRlbnNvcjogdG9yY2guVGVuc29yLCBvcGVyYXRpb246IGludCkgLT4gdG9yY2guVGVuc29yOgogICAgaWYgb3BlcmF0aW9uID09IDQ6CiAgICAgICAgcmV0dXJuIHRvcmNoLnJvdDkwKHRlbnNvciwgMywgZGltcz0oLTIsIC0xKSkKICAgIGlmIG9wZXJhdGlvbiA9PSA1OgogICAgICAgIHJldHVybiB0b3JjaC5yb3Q5MCh0ZW5zb3IsIDEsIGRpbXM9KC0yLCAtMSkpCiAgICByZXR1cm4gX3RyYW5zZm9ybSh0ZW5zb3IsIG9wZXJhdGlvbikKCgpAdG9yY2guaW5mZXJlbmNlX21vZGUoKQpkZWYgcHJlZGljdF9wcm9iYWJpbGl0aWVzKAogICAgbW9kZWw6IEZpbGFtZW50VU5ldCwKICAgIGdyYXlfaW1hZ2U6IG5wLm5kYXJyYXksCiAgICAqLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UgfCBOb25lID0gTm9uZSwKICAgIHBhdGNoX3NpemU6IGludCA9IDUxMiwKICAgIHN0cmlkZTogaW50ID0gMzg0LAogICAgdHRhOiBpbnQgPSAxLAopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgaWYgZ3JheV9pbWFnZS5uZGltICE9IDIgb3IgZ3JheV9pbWFnZS5kdHlwZSAhPSBucC51aW50ODoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCBhbiA4LWJpdCBncmF5c2NhbGUgaW1hZ2UiKQogICAgaWYgZ3JheV9pbWFnZS5zaGFwZSAhPSAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0sIGdvdCB7Z3JheV9pbWFnZS5zaGFwZX0iKQogICAgaWYgcGF0Y2hfc2l6ZSAlIDMyIG9yIHBhdGNoX3NpemUgPiBJTUFHRV9TSVpFOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBhdGNoX3NpemUgbXVzdCBiZSBkaXZpc2libGUgYnkgMzIgYW5kIG5vIGxhcmdlciB0aGFuIHRoZSBpbWFnZSIpCiAgICBpZiB0dGEgbm90IGluICgxLCAyLCA0LCA4KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0dGEgbXVzdCBiZSAxLCAyLCA0LCBvciA4IikKICAgIGRldmljZSA9IGRldmljZSBvciBuZXh0KG1vZGVsLnBhcmFtZXRlcnMoKSkuZGV2aWNlCiAgICBtb2RlbC5ldmFsKCkKCiAgICBzY2FsZWQgPSBncmF5X2ltYWdlLmFzdHlwZShucC5mbG9hdDMyKSAvIDI1NS4wCiAgICB5eSwgeHggPSBucC5vZ3JpZFs6SU1BR0VfU0laRSwgOklNQUdFX1NJWkVdCiAgICBkaXNrID0gKHl5IC0gMTAyMy4wKSAqKiAyICsgKHh4IC0gMTAyMy4wKSAqKiAyIDw9IDk1MCoqMgogICAgdmFsdWVzID0gc2NhbGVkW2Rpc2tdCiAgICBub3JtYWxpemVkID0gbnAuY2xpcCgoc2NhbGVkIC0gZmxvYXQodmFsdWVzLm1lYW4oKSkpIC8gbWF4KGZsb2F0KHZhbHVlcy5zdGQoKSksIDFlLTMpLCAtNSwgNSkKCiAgICBvbmVfZGltZW5zaW9uYWwgPSBucC5oYW5uaW5nKHBhdGNoX3NpemUpLmFzdHlwZShucC5mbG9hdDMyKQogICAgb25lX2RpbWVuc2lvbmFsID0gbnAubWF4aW11bShvbmVfZGltZW5zaW9uYWwsIDAuMDUpCiAgICB3ZWlnaHQgPSBucC5vdXRlcihvbmVfZGltZW5zaW9uYWwsIG9uZV9kaW1lbnNpb25hbCkKICAgIGFjY3VtdWxhdGlvbiA9IG5wLnplcm9zKCgyLCBJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgIHdlaWdodF9zdW0gPSBucC56ZXJvcygoSU1BR0VfU0laRSwgSU1BR0VfU0laRSksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBvcGVyYXRpb25zID0gKDAsIDEsIDIsIDMsIDQsIDUsIDYsIDcpWzp0dGFdCiAgICBncHUgPSBkZXZpY2UudHlwZSA9PSAiY3VkYSIKCiAgICBmb3IgeTAgaW4gX3RpbGVfc3RhcnRzKElNQUdFX1NJWkUsIHBhdGNoX3NpemUsIHN0cmlkZSk6CiAgICAgICAgZm9yIHgwIGluIF90aWxlX3N0YXJ0cyhJTUFHRV9TSVpFLCBwYXRjaF9zaXplLCBzdHJpZGUpOgogICAgICAgICAgICBjcm9wID0gbm9ybWFsaXplZFt5MCA6IHkwICsgcGF0Y2hfc2l6ZSwgeDAgOiB4MCArIHBhdGNoX3NpemVdLmNvcHkoKQogICAgICAgICAgICB0ZW5zb3IgPSB0b3JjaC5mcm9tX251bXB5KGNyb3BbTm9uZSwgTm9uZV0pLnRvKGRldmljZT1kZXZpY2UpCiAgICAgICAgICAgIGlmIGdwdToKICAgICAgICAgICAgICAgIHRlbnNvciA9IHRlbnNvci5jb250aWd1b3VzKG1lbW9yeV9mb3JtYXQ9dG9yY2guY2hhbm5lbHNfbGFzdCkKICAgICAgICAgICAgYmF0Y2ggPSB0b3JjaC5jYXQoW190cmFuc2Zvcm0odGVuc29yLCBvcCkgZm9yIG9wIGluIG9wZXJhdGlvbnNdLCBkaW09MCkKICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgKICAgICAgICAgICAgICAgIGRldmljZV90eXBlPWRldmljZS50eXBlLAogICAgICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgICAgIGVuYWJsZWQ9Z3B1LAogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoYmF0Y2gpCiAgICAgICAgICAgIHByb2JhYmlsaXRpZXMgPSBsb2dpdHMuZmxvYXQoKS5zaWdtb2lkKCkuY3B1KCkKICAgICAgICAgICAgYXZlcmFnZWQgPSB0b3JjaC56ZXJvcygoMiwgcGF0Y2hfc2l6ZSwgcGF0Y2hfc2l6ZSksIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgICAgIGZvciBpbmRleCwgb3BlcmF0aW9uIGluIGVudW1lcmF0ZShvcGVyYXRpb25zKToKICAgICAgICAgICAgICAgIGF2ZXJhZ2VkICs9IF9pbnZlcnNlX3RyYW5zZm9ybShwcm9iYWJpbGl0aWVzW2luZGV4XSwgb3BlcmF0aW9uKQogICAgICAgICAgICBhdmVyYWdlZCAvPSBsZW4ob3BlcmF0aW9ucykKICAgICAgICAgICAgcmVnaW9uID0gbnAuc19beTAgOiB5MCArIHBhdGNoX3NpemUsIHgwIDogeDAgKyBwYXRjaF9zaXplXQogICAgICAgICAgICBhY2N1bXVsYXRpb25bOiwgcmVnaW9uWzBdLCByZWdpb25bMV1dICs9IGF2ZXJhZ2VkLm51bXB5KCkgKiB3ZWlnaHRbTm9uZV0KICAgICAgICAgICAgd2VpZ2h0X3N1bVtyZWdpb25dICs9IHdlaWdodAoKICAgIG91dHB1dCA9IGFjY3VtdWxhdGlvbiAvIG5wLm1heGltdW0od2VpZ2h0X3N1bVtOb25lXSwgMWUtNikKICAgIHJldHVybiBvdXRwdXRbMF0sIG91dHB1dFsxXQoKCmRlZiBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgZm9yZWdyb3VuZF9wcm9iYWJpbGl0eTogbnAubmRhcnJheSwKICAgIGJvdW5kYXJ5X3Byb2JhYmlsaXR5OiBucC5uZGFycmF5LAogICAgZmlsZV9uYW1lOiBzdHIsCiAgICBjb25maWc6IEluc3RhbmNlQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV06CiAgICBjb25maWcgPSBjb25maWcgb3IgSW5zdGFuY2VDb25maWcoKQogICAgaWYgZm9yZWdyb3VuZF9wcm9iYWJpbGl0eS5zaGFwZSAhPSBib3VuZGFyeV9wcm9iYWJpbGl0eS5zaGFwZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGb3JlZ3JvdW5kIGFuZCBib3VuZGFyeSBwcm9iYWJpbGl0eSBtYXBzIG11c3QgaGF2ZSBlcXVhbCBzaGFwZXMiKQogICAgaGVpZ2h0LCB3aWR0aCA9IGZvcmVncm91bmRfcHJvYmFiaWxpdHkuc2hhcGUKICAgIGlmIChoZWlnaHQsIHdpZHRoKSAhPSAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0gcHJvYmFiaWxpdHkgbWFwcyIpCiAgICB5eSwgeHggPSBucC5vZ3JpZFs6aGVpZ2h0LCA6d2lkdGhdCiAgICBkaXNrID0gKHl5IC0gMTAyMy4wKSAqKiAyICsgKHh4IC0gMTAyMy4wKSAqKiAyIDw9IGNvbmZpZy5zb2xhcl9yYWRpdXMqKjIKICAgIGZvcmVncm91bmQgPSAoZm9yZWdyb3VuZF9wcm9iYWJpbGl0eSA+PSBjb25maWcuZm9yZWdyb3VuZF90aHJlc2hvbGQpICYgZGlzawogICAgaWYgbm90IG5wLmFueShmb3JlZ3JvdW5kKToKICAgICAgICByZXR1cm4gW10KCiAgICBzZWVkX21hc2sgPSAoCiAgICAgICAgZm9yZWdyb3VuZAogICAgICAgICYgKGZvcmVncm91bmRfcHJvYmFiaWxpdHkgPj0gY29uZmlnLnNlZWRfdGhyZXNob2xkKQogICAgICAgICYgKGJvdW5kYXJ5X3Byb2JhYmlsaXR5IDwgY29uZmlnLmJvdW5kYXJ5X3RocmVzaG9sZCkKICAgICkKICAgIG1hcmtlcnMsIG1hcmtlcl9jb3VudCA9IG5kaS5sYWJlbChzZWVkX21hc2ssIHN0cnVjdHVyZT1ucC5vbmVzKCgzLCAzKSwgZHR5cGU9bnAudWludDgpKQogICAgaWYgbWFya2VyX2NvdW50OgogICAgICAgIHNpemVzID0gbnAuYmluY291bnQobWFya2Vycy5yYXZlbCgpLCBtaW5sZW5ndGg9bWFya2VyX2NvdW50ICsgMSkKICAgICAgICBrZWVwID0gc2l6ZXMgPj0gY29uZmlnLnNlZWRfbWluX2FyZWEKICAgICAgICBrZWVwWzBdID0gRmFsc2UKICAgICAgICBzZWVkX21hc2sgJj0ga2VlcFttYXJrZXJzXQogICAgICAgIG1hcmtlcnMsIG1hcmtlcl9jb3VudCA9IG5kaS5sYWJlbCgKICAgICAgICAgICAgc2VlZF9tYXNrLCBzdHJ1Y3R1cmU9bnAub25lcygoMywgMyksIGR0eXBlPW5wLnVpbnQ4KQogICAgICAgICkKICAgIGlmIG1hcmtlcl9jb3VudCA9PSAwOgogICAgICAgIG1hcmtlcnMsIG1hcmtlcl9jb3VudCA9IG5kaS5sYWJlbCgKICAgICAgICAgICAgZm9yZWdyb3VuZCwgc3RydWN0dXJlPW5wLm9uZXMoKDMsIDMpLCBkdHlwZT1ucC51aW50OCkKICAgICAgICApCiAgICBlbGV2YXRpb24gPSAtZm9yZWdyb3VuZF9wcm9iYWJpbGl0eS5hc3R5cGUobnAuZmxvYXQzMikgKyAoCiAgICAgICAgY29uZmlnLmJvdW5kYXJ5X3dlaWdodCAqIGJvdW5kYXJ5X3Byb2JhYmlsaXR5LmFzdHlwZShucC5mbG9hdDMyKQogICAgKQogICAgbGFiZWxzID0gd2F0ZXJzaGVkKAogICAgICAgIGVsZXZhdGlvbiwKICAgICAgICBtYXJrZXJzLmFzdHlwZShucC5pbnQzMiksCiAgICAgICAgbWFzaz1mb3JlZ3JvdW5kLAogICAgICAgIGNvbm5lY3Rpdml0eT1ucC5vbmVzKCgzLCAzKSwgZHR5cGU9bnAudWludDgpLAogICAgICAgIHdhdGVyc2hlZF9saW5lPVRydWUsCiAgICApLmFzdHlwZShucC5pbnQzMiwgY29weT1GYWxzZSkKICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KGxhYmVscy5yYXZlbCgpKQogICAgYWNjZXB0ZWQgPSBbbGFiZWwgZm9yIGxhYmVsIGluIHJhbmdlKDEsIGxlbihjb3VudHMpKSBpZiBjb3VudHNbbGFiZWxdID49IGNvbmZpZy5taW5faW5zdGFuY2VfYXJlYV0KICAgIGlmIG5vdCBhY2NlcHRlZDoKICAgICAgICByZXR1cm4gW10KCiAgICBmaXJzdF9ieV9sYWJlbCA9IG5wLmZ1bGwobGVuKGNvdW50cyksIGxhYmVscy5zaXplLCBkdHlwZT1ucC5pbnQ2NCkKICAgIG5vbnplcm9fcG9zaXRpb25zID0gbnAuZmxhdG5vbnplcm8obGFiZWxzLnJhdmVsKCkpCiAgICBucC5taW5pbXVtLmF0KAogICAgICAgIGZpcnN0X2J5X2xhYmVsLAogICAgICAgIGxhYmVscy5yYXZlbCgpW25vbnplcm9fcG9zaXRpb25zXSwKICAgICAgICBub256ZXJvX3Bvc2l0aW9ucywKICAgICkKICAgIGZpcnN0X3Bvc2l0aW9ucyA9IHtsYWJlbDogaW50KGZpcnN0X2J5X2xhYmVsW2xhYmVsXSkgZm9yIGxhYmVsIGluIGFjY2VwdGVkfQogICAgYWNjZXB0ZWQuc29ydChrZXk9Zmlyc3RfcG9zaXRpb25zLl9fZ2V0aXRlbV9fKQogICAgZW5jb2RlZCA9IGVuY29kZV9sYWJlbF9jb21wb25lbnRzKGxhYmVscywgYWNjZXB0ZWQpCiAgICBpbWFnZV9zdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIHJldHVybiBbCiAgICAgICAgUHJlZGljdGVkSW5zdGFuY2UoCiAgICAgICAgICAgIGZpbGFtZW50X2lkPWYie2ltYWdlX3N0ZW19X3tzZXF1ZW5jZX0iLAogICAgICAgICAgICBpbWFnZV9zdGVtPWltYWdlX3N0ZW0sCiAgICAgICAgICAgIGFyZWE9aW50KGNvdW50c1tsYWJlbF0pLAogICAgICAgICAgICBybGU9ZW5jb2RlZFtsYWJlbF0sCiAgICAgICAgKQogICAgICAgIGZvciBzZXF1ZW5jZSwgbGFiZWwgaW4gZW51bWVyYXRlKGFjY2VwdGVkLCBzdGFydD0xKQogICAgXQo=","src/filament_baseline/learning.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBPcmRlcmVkRGljdApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCmZyb20gLmRhdGEgaW1wb3J0IENvbXBldGl0aW9uRGF0YXNldCwgSW1hZ2VWaWV3CgpUYXJnZXRNb2RlID0gTGl0ZXJhbFsiYW5ub3RhdG9yIiwgImNvbnNlbnN1cyJdCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSW5zdGFuY2VQb2x5Z29uczoKICAgIHBvbHlnb25zOiB0dXBsZVtucC5uZGFycmF5LCAuLi5dCiAgICBib3VuZHM6IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXQsIGZsb2F0XQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFZpZXdUYXJnZXRzOgogICAgdmlldzogSW1hZ2VWaWV3CiAgICBpbnN0YW5jZXM6IHR1cGxlW0luc3RhbmNlUG9seWdvbnMsIC4uLl0KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBQYXRjaFJlY29yZDoKICAgIGZpbGVfbmFtZTogc3RyCiAgICB2aWV3czogdHVwbGVbVmlld1RhcmdldHMsIC4uLl0KCgpkZWYgX3ZpZXdfdGFyZ2V0cyhkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsIHZpZXc6IEltYWdlVmlldykgLT4gVmlld1RhcmdldHM6CiAgICBpbnN0YW5jZXMgPSBbXQogICAgZm9yIGFubm90YXRpb24gaW4gZGF0YXNldC5hbm5vdGF0aW9uc19ieV9pbWFnZV9pZFt2aWV3LmltYWdlX2lkXToKICAgICAgICBzZWdtZW50YXRpb24gPSBhbm5vdGF0aW9uWyJzZWdtZW50YXRpb24iXQogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHNlZ21lbnRhdGlvbiwgbGlzdCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRyYWluaW5nIHBvbHlnb25zIG11c3QgZm9sbG93IHRoZSBjb21wZXRpdGlvbiBDT0NPIHBvbHlnb24gZm9ybWF0IikKICAgICAgICBwb2x5Z29ucyA9IHR1cGxlKAogICAgICAgICAgICBucC5hc2FycmF5KHBvbHlnb24sIGR0eXBlPW5wLmZsb2F0MzIpLnJlc2hhcGUoLTEsIDIpCiAgICAgICAgICAgIGZvciBwb2x5Z29uIGluIHNlZ21lbnRhdGlvbgogICAgICAgICAgICBpZiBsZW4ocG9seWdvbikgPj0gNgogICAgICAgICkKICAgICAgICBpZiBub3QgcG9seWdvbnM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcG9pbnRzID0gbnAuY29uY2F0ZW5hdGUocG9seWdvbnMsIGF4aXM9MCkKICAgICAgICB4X21pbiwgeV9taW4gPSBucC5taW4ocG9pbnRzLCBheGlzPTApCiAgICAgICAgeF9tYXgsIHlfbWF4ID0gbnAubWF4KHBvaW50cywgYXhpcz0wKQogICAgICAgIGluc3RhbmNlcy5hcHBlbmQoCiAgICAgICAgICAgIEluc3RhbmNlUG9seWdvbnMoCiAgICAgICAgICAgICAgICBwb2x5Z29ucywKICAgICAgICAgICAgICAgIChmbG9hdCh4X21pbiksIGZsb2F0KHlfbWluKSwgZmxvYXQoeF9tYXgpLCBmbG9hdCh5X21heCkpLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIFZpZXdUYXJnZXRzKHZpZXcsIHR1cGxlKGluc3RhbmNlcykpCgoKZGVmIHJlY29yZHNfZm9yX29ic2VydmF0aW9ucygKICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgIG9ic2VydmF0aW9uczogdHVwbGVbc3RyLCAuLi5dIHwgbGlzdFtzdHJdLAogICAgdGFyZ2V0X21vZGU6IFRhcmdldE1vZGUsCikgLT4gdHVwbGVbUGF0Y2hSZWNvcmQsIC4uLl06CiAgICByZWNvcmRzID0gW10KICAgIGZvciBmaWxlX25hbWUgaW4gc29ydGVkKG9ic2VydmF0aW9ucyk6CiAgICAgICAgdmlld3MgPSB0dXBsZSgKICAgICAgICAgICAgX3ZpZXdfdGFyZ2V0cyhkYXRhc2V0LCB2aWV3KQogICAgICAgICAgICBmb3IgdmlldyBpbiBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV0KICAgICAgICApCiAgICAgICAgaWYgdGFyZ2V0X21vZGUgPT0gImFubm90YXRvciI6CiAgICAgICAgICAgIHJlY29yZHMuZXh0ZW5kKFBhdGNoUmVjb3JkKGZpbGVfbmFtZSwgKHZpZXcsKSkgZm9yIHZpZXcgaW4gdmlld3MpCiAgICAgICAgZWxpZiB0YXJnZXRfbW9kZSA9PSAiY29uc2Vuc3VzIjoKICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQoUGF0Y2hSZWNvcmQoZmlsZV9uYW1lLCB2aWV3cykpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuc3VwcG9ydGVkIHRhcmdldCBtb2RlOiB7dGFyZ2V0X21vZGV9IikKICAgIHJldHVybiB0dXBsZShyZWNvcmRzKQoKCmRlZiBfaW5zdGFuY2VfbWFwKHZpZXc6IFZpZXdUYXJnZXRzLCB4MDogaW50LCB5MDogaW50LCBzaXplOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICBsYWJlbHMgPSBucC56ZXJvcygoc2l6ZSwgc2l6ZSksIGR0eXBlPW5wLnVpbnQxNikKICAgIHgxLCB5MSA9IHgwICsgc2l6ZSwgeTAgKyBzaXplCiAgICBmb3IgbGFiZWwsIGluc3RhbmNlIGluIGVudW1lcmF0ZSh2aWV3Lmluc3RhbmNlcywgc3RhcnQ9MSk6CiAgICAgICAgbGVmdCwgdG9wLCByaWdodCwgYm90dG9tID0gaW5zdGFuY2UuYm91bmRzCiAgICAgICAgaWYgcmlnaHQgPCB4MCBvciBib3R0b20gPCB5MCBvciBsZWZ0ID49IHgxIG9yIHRvcCA+PSB5MToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBsb2NhbF9wb2x5Z29ucyA9IFtdCiAgICAgICAgZm9yIHBvbHlnb24gaW4gaW5zdGFuY2UucG9seWdvbnM6CiAgICAgICAgICAgIHBvaW50cyA9IG5wLnJpbnQocG9seWdvbiAtIG5wLmFycmF5KFt4MCwgeTBdLCBkdHlwZT1ucC5mbG9hdDMyKSkKICAgICAgICAgICAgcG9pbnRzID0gbnAuY2xpcChwb2ludHMsIC1zaXplLCAyICogc2l6ZSkuYXN0eXBlKG5wLmludDMyKQogICAgICAgICAgICBpZiBsZW4ocG9pbnRzKSA+PSAzOgogICAgICAgICAgICAgICAgbG9jYWxfcG9seWdvbnMuYXBwZW5kKHBvaW50cykKICAgICAgICBpZiBsb2NhbF9wb2x5Z29uczoKICAgICAgICAgICAgY3YyLmZpbGxQb2x5KGxhYmVscywgbG9jYWxfcG9seWdvbnMsIGNvbG9yPWxhYmVsLCBsaW5lVHlwZT1jdjIuTElORV84KQogICAgcmV0dXJuIGxhYmVscwoKCmRlZiBfZm9yZWdyb3VuZF9hbmRfYm91bmRhcnkobGFiZWxzOiBucC5uZGFycmF5KSAtPiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5XToKICAgIGZvcmVncm91bmQgPSAobGFiZWxzID4gMCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBib3VuZGFyeSA9IG5wLnplcm9zKGxhYmVscy5zaGFwZSwgZHR5cGU9Ym9vbCkKICAgIGhvcml6b250YWwgPSBsYWJlbHNbOiwgMTpdICE9IGxhYmVsc1s6LCA6LTFdCiAgICB2ZXJ0aWNhbCA9IGxhYmVsc1sxOiwgOl0gIT0gbGFiZWxzWzotMSwgOl0KICAgIGJvdW5kYXJ5WzosIDE6XSB8PSBob3Jpem9udGFsICYgKChsYWJlbHNbOiwgMTpdID4gMCkgfCAobGFiZWxzWzosIDotMV0gPiAwKSkKICAgIGJvdW5kYXJ5WzosIDotMV0gfD0gaG9yaXpvbnRhbCAmICgobGFiZWxzWzosIDE6XSA+IDApIHwgKGxhYmVsc1s6LCA6LTFdID4gMCkpCiAgICBib3VuZGFyeVsxOiwgOl0gfD0gdmVydGljYWwgJiAoKGxhYmVsc1sxOiwgOl0gPiAwKSB8IChsYWJlbHNbOi0xLCA6XSA+IDApKQogICAgYm91bmRhcnlbOi0xLCA6XSB8PSB2ZXJ0aWNhbCAmICgobGFiZWxzWzE6LCA6XSA+IDApIHwgKGxhYmVsc1s6LTEsIDpdID4gMCkpCiAgICByZXR1cm4gZm9yZWdyb3VuZCwgYm91bmRhcnkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKY2xhc3MgRmlsYW1lbnRQYXRjaERhdGFzZXQoRGF0YXNldFt0dXBsZVt0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl1dKToKICAgICIiIk9uLXRoZS1mbHkgZnVsbC1yZXNvbHV0aW9uIGNyb3BzIGZyb20gb2ZmaWNpYWwgYW5ub3RhdGlvbiB2aWV3cy4iIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsCiAgICAgICAgb2JzZXJ2YXRpb25zOiB0dXBsZVtzdHIsIC4uLl0gfCBsaXN0W3N0cl0sCiAgICAgICAgKiwKICAgICAgICB0YXJnZXRfbW9kZTogVGFyZ2V0TW9kZSA9ICJjb25zZW5zdXMiLAogICAgICAgIHBhdGNoX3NpemU6IGludCA9IDUxMiwKICAgICAgICBwb3NpdGl2ZV9mcmFjdGlvbjogZmxvYXQgPSAwLjcsCiAgICAgICAgYXVnbWVudDogYm9vbCA9IEZhbHNlLAogICAgICAgIHNlZWQ6IGludCA9IDQyLAogICAgICAgIGltYWdlX2NhY2hlX3NpemU6IGludCA9IDgsCiAgICApIC0+IE5vbmU6CiAgICAgICAgaWYgcGF0Y2hfc2l6ZSA8IDEyOCBvciBwYXRjaF9zaXplICUgMzI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBhdGNoX3NpemUgbXVzdCBiZSBhdCBsZWFzdCAxMjggYW5kIGRpdmlzaWJsZSBieSAzMiIpCiAgICAgICAgaWYgbm90IDAuMCA8PSBwb3NpdGl2ZV9mcmFjdGlvbiA8PSAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBvc2l0aXZlX2ZyYWN0aW9uIG11c3QgYmUgYmV0d2VlbiAwIGFuZCAxIikKICAgICAgICBzZWxmLnJlY29yZHMgPSByZWNvcmRzX2Zvcl9vYnNlcnZhdGlvbnMoZGF0YXNldCwgb2JzZXJ2YXRpb25zLCB0YXJnZXRfbW9kZSkKICAgICAgICBzZWxmLnRhcmdldF9tb2RlID0gdGFyZ2V0X21vZGUKICAgICAgICBzZWxmLmltYWdlX2RpciA9IGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyCiAgICAgICAgc2VsZi5wYXRjaF9zaXplID0gcGF0Y2hfc2l6ZQogICAgICAgIHNlbGYucG9zaXRpdmVfZnJhY3Rpb24gPSBwb3NpdGl2ZV9mcmFjdGlvbgogICAgICAgIHNlbGYuYXVnbWVudCA9IGF1Z21lbnQKICAgICAgICBzZWxmLnNlZWQgPSBzZWVkCiAgICAgICAgc2VsZi5lcG9jaCA9IDAKICAgICAgICBzZWxmLmltYWdlX2NhY2hlX3NpemUgPSBpbWFnZV9jYWNoZV9zaXplCiAgICAgICAgc2VsZi5faW1hZ2VfY2FjaGU6IE9yZGVyZWREaWN0W3N0ciwgdHVwbGVbbnAubmRhcnJheSwgZmxvYXQsIGZsb2F0XV0gPSBPcmRlcmVkRGljdCgpCiAgICAgICAgc2VsZi5fZGlzayA9IG5wLnplcm9zKCgyMDQ4LCAyMDQ4KSwgZHR5cGU9bnAudWludDgpCiAgICAgICAgY3YyLmNpcmNsZShzZWxmLl9kaXNrLCAoMTAyMywgMTAyMyksIDk1MCwgY29sb3I9MSwgdGhpY2tuZXNzPS0xKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYucmVjb3JkcykKCiAgICBkZWYgc2V0X2Vwb2NoKHNlbGYsIGVwb2NoOiBpbnQpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5lcG9jaCA9IGVwb2NoCgogICAgZGVmIF9pbWFnZV9hbmRfc3RhdHMoc2VsZiwgZmlsZV9uYW1lOiBzdHIpIC0+IHR1cGxlW25wLm5kYXJyYXksIGZsb2F0LCBmbG9hdF06CiAgICAgICAgY2FjaGVkID0gc2VsZi5faW1hZ2VfY2FjaGUuZ2V0KGZpbGVfbmFtZSkKICAgICAgICBpZiBjYWNoZWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYuX2ltYWdlX2NhY2hlLm1vdmVfdG9fZW5kKGZpbGVfbmFtZSkKICAgICAgICAgICAgcmV0dXJuIGNhY2hlZAogICAgICAgIGltYWdlID0gY3YyLmltcmVhZChzdHIoc2VsZi5pbWFnZV9kaXIgLyBmaWxlX25hbWUpLCBjdjIuSU1SRUFEX0dSQVlTQ0FMRSkKICAgICAgICBpZiBpbWFnZSBpcyBOb25lIG9yIGltYWdlLnNoYXBlICE9ICgyMDQ4LCAyMDQ4KToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDb3VsZCBub3QgcmVhZCBhIDIwNDh4MjA0OCB0cmFpbmluZyBpbWFnZToge2ZpbGVfbmFtZX0iKQogICAgICAgIHNjYWxlZCA9IGltYWdlLmFzdHlwZShucC5mbG9hdDMyKSAvIDI1NS4wCiAgICAgICAgZGlza192YWx1ZXMgPSBzY2FsZWRbc2VsZi5fZGlzayA+IDBdCiAgICAgICAgbWVhbiA9IGZsb2F0KG5wLm1lYW4oZGlza192YWx1ZXMpKQogICAgICAgIHN0ZCA9IGZsb2F0KG5wLnN0ZChkaXNrX3ZhbHVlcykpCiAgICAgICAgY2FjaGVkID0gKGltYWdlLCBtZWFuLCBtYXgoc3RkLCAxZS0zKSkKICAgICAgICBzZWxmLl9pbWFnZV9jYWNoZVtmaWxlX25hbWVdID0gY2FjaGVkCiAgICAgICAgd2hpbGUgbGVuKHNlbGYuX2ltYWdlX2NhY2hlKSA+IHNlbGYuaW1hZ2VfY2FjaGVfc2l6ZToKICAgICAgICAgICAgc2VsZi5faW1hZ2VfY2FjaGUucG9waXRlbShsYXN0PUZhbHNlKQogICAgICAgIHJldHVybiBjYWNoZWQKCiAgICBkZWYgX2Nyb3Bfb3JpZ2luKHNlbGYsIHJlY29yZDogUGF0Y2hSZWNvcmQsIHJuZzogcmFuZG9tLlJhbmRvbSkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIGFubm90YXRpb25zID0gWwogICAgICAgICAgICBpbnN0YW5jZQogICAgICAgICAgICBmb3IgdmlldyBpbiByZWNvcmQudmlld3MKICAgICAgICAgICAgZm9yIGluc3RhbmNlIGluIHZpZXcuaW5zdGFuY2VzCiAgICAgICAgXQogICAgICAgIGlmIGFubm90YXRpb25zIGFuZCBybmcucmFuZG9tKCkgPCBzZWxmLnBvc2l0aXZlX2ZyYWN0aW9uOgogICAgICAgICAgICBpbnN0YW5jZSA9IHJuZy5jaG9pY2UoYW5ub3RhdGlvbnMpCiAgICAgICAgICAgIHBvbHlnb24gPSBybmcuY2hvaWNlKGluc3RhbmNlLnBvbHlnb25zKQogICAgICAgICAgICBjZW50ZXJfeCwgY2VudGVyX3kgPSBwb2x5Z29uW3JuZy5yYW5kcmFuZ2UobGVuKHBvbHlnb24pKV0KICAgICAgICBlbHNlOgogICAgICAgICAgICBjZW50ZXJfeCA9IHJuZy51bmlmb3JtKDAsIDIwNDcpCiAgICAgICAgICAgIGNlbnRlcl95ID0gcm5nLnVuaWZvcm0oMCwgMjA0NykKICAgICAgICBoYWxmID0gc2VsZi5wYXRjaF9zaXplIC8vIDIKICAgICAgICB4MCA9IG1pbihtYXgoaW50KGNlbnRlcl94KSAtIGhhbGYsIDApLCAyMDQ4IC0gc2VsZi5wYXRjaF9zaXplKQogICAgICAgIHkwID0gbWluKG1heChpbnQoY2VudGVyX3kpIC0gaGFsZiwgMCksIDIwNDggLSBzZWxmLnBhdGNoX3NpemUpCiAgICAgICAgcmV0dXJuIHgwLCB5MAoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleDogaW50KSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgcmVjb3JkID0gc2VsZi5yZWNvcmRzW2luZGV4XQogICAgICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VsZi5zZWVkICsgc2VsZi5lcG9jaCAqIDFfMDAwXzAwMyArIGluZGV4ICogOTcpCiAgICAgICAgaW1hZ2UsIG1lYW4sIHN0ZCA9IHNlbGYuX2ltYWdlX2FuZF9zdGF0cyhyZWNvcmQuZmlsZV9uYW1lKQogICAgICAgIHgwLCB5MCA9IHNlbGYuX2Nyb3Bfb3JpZ2luKHJlY29yZCwgcm5nKQogICAgICAgIHgxLCB5MSA9IHgwICsgc2VsZi5wYXRjaF9zaXplLCB5MCArIHNlbGYucGF0Y2hfc2l6ZQogICAgICAgIGltYWdlX2Nyb3AgPSBpbWFnZVt5MDp5MSwgeDA6eDFdLmFzdHlwZShucC5mbG9hdDMyKSAvIDI1NS4wCgogICAgICAgIHBlcl92aWV3X2ZvcmVncm91bmQgPSBbXQogICAgICAgIHBlcl92aWV3X2JvdW5kYXJ5ID0gW10KICAgICAgICBmb3IgdmlldyBpbiByZWNvcmQudmlld3M6CiAgICAgICAgICAgIGxhYmVscyA9IF9pbnN0YW5jZV9tYXAodmlldywgeDAsIHkwLCBzZWxmLnBhdGNoX3NpemUpCiAgICAgICAgICAgIGZvcmVncm91bmQsIGJvdW5kYXJ5ID0gX2ZvcmVncm91bmRfYW5kX2JvdW5kYXJ5KGxhYmVscykKICAgICAgICAgICAgcGVyX3ZpZXdfZm9yZWdyb3VuZC5hcHBlbmQoZm9yZWdyb3VuZCkKICAgICAgICAgICAgcGVyX3ZpZXdfYm91bmRhcnkuYXBwZW5kKGJvdW5kYXJ5KQogICAgICAgIHRhcmdldF9mb3JlZ3JvdW5kID0gbnAubWVhbihwZXJfdmlld19mb3JlZ3JvdW5kLCBheGlzPTAsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgdGFyZ2V0X2JvdW5kYXJ5ID0gbnAubWVhbihwZXJfdmlld19ib3VuZGFyeSwgYXhpcz0wLCBkdHlwZT1ucC5mbG9hdDMyKQoKICAgICAgICBpZiBzZWxmLmF1Z21lbnQ6CiAgICAgICAgICAgIHR1cm5zID0gcm5nLnJhbmRyYW5nZSg0KQogICAgICAgICAgICBpbWFnZV9jcm9wID0gbnAucm90OTAoaW1hZ2VfY3JvcCwgdHVybnMpLmNvcHkoKQogICAgICAgICAgICB0YXJnZXRfZm9yZWdyb3VuZCA9IG5wLnJvdDkwKHRhcmdldF9mb3JlZ3JvdW5kLCB0dXJucykuY29weSgpCiAgICAgICAgICAgIHRhcmdldF9ib3VuZGFyeSA9IG5wLnJvdDkwKHRhcmdldF9ib3VuZGFyeSwgdHVybnMpLmNvcHkoKQogICAgICAgICAgICBpZiBybmcucmFuZG9tKCkgPCAwLjU6CiAgICAgICAgICAgICAgICBpbWFnZV9jcm9wID0gbnAuZmxpcGxyKGltYWdlX2Nyb3ApLmNvcHkoKQogICAgICAgICAgICAgICAgdGFyZ2V0X2ZvcmVncm91bmQgPSBucC5mbGlwbHIodGFyZ2V0X2ZvcmVncm91bmQpLmNvcHkoKQogICAgICAgICAgICAgICAgdGFyZ2V0X2JvdW5kYXJ5ID0gbnAuZmxpcGxyKHRhcmdldF9ib3VuZGFyeSkuY29weSgpCiAgICAgICAgICAgIGlmIHJuZy5yYW5kb20oKSA8IDAuNToKICAgICAgICAgICAgICAgIGltYWdlX2Nyb3AgPSBucC5mbGlwdWQoaW1hZ2VfY3JvcCkuY29weSgpCiAgICAgICAgICAgICAgICB0YXJnZXRfZm9yZWdyb3VuZCA9IG5wLmZsaXB1ZCh0YXJnZXRfZm9yZWdyb3VuZCkuY29weSgpCiAgICAgICAgICAgICAgICB0YXJnZXRfYm91bmRhcnkgPSBucC5mbGlwdWQodGFyZ2V0X2JvdW5kYXJ5KS5jb3B5KCkKICAgICAgICAgICAgaW1hZ2VfY3JvcCA9IG5wLnBvd2VyKAogICAgICAgICAgICAgICAgbnAuY2xpcChpbWFnZV9jcm9wLCAwLjAsIDEuMCksIHJuZy51bmlmb3JtKDAuOCwgMS4yKQogICAgICAgICAgICApCiAgICAgICAgICAgIGltYWdlX2Nyb3AgPSBucC5jbGlwKAogICAgICAgICAgICAgICAgaW1hZ2VfY3JvcCAqIHJuZy51bmlmb3JtKDAuOSwgMS4xKSArIHJuZy51bmlmb3JtKC0wLjA0LCAwLjA0KSwKICAgICAgICAgICAgICAgIDAuMCwKICAgICAgICAgICAgICAgIDEuMCwKICAgICAgICAgICAgKQoKICAgICAgICBub3JtYWxpemVkID0gbnAuY2xpcCgoaW1hZ2VfY3JvcCAtIG1lYW4pIC8gc3RkLCAtNS4wLCA1LjApCiAgICAgICAgaW1hZ2VfdGVuc29yID0gdG9yY2guZnJvbV9udW1weShub3JtYWxpemVkW05vbmVdLmFzdHlwZShucC5mbG9hdDMyKSkKICAgICAgICB0YXJnZXRfdGVuc29yID0gdG9yY2guZnJvbV9udW1weSgKICAgICAgICAgICAgbnAuc3RhY2soKHRhcmdldF9mb3JlZ3JvdW5kLCB0YXJnZXRfYm91bmRhcnkpKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICApCiAgICAgICAgcmV0dXJuIGltYWdlX3RlbnNvciwgdGFyZ2V0X3RlbnNvcgoKCmRlZiB0cmFpbmluZ19sb3NzKGxvZ2l0czogdG9yY2guVGVuc29yLCB0YXJnZXQ6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgZm9yZWdyb3VuZF90YXJnZXQgPSB0YXJnZXRbOiwgMDoxXQogICAgYm91bmRhcnlfdGFyZ2V0ID0gdGFyZ2V0WzosIDE6Ml0KICAgIGZvcmVncm91bmRfYmNlID0gdG9yY2gubm4uZnVuY3Rpb25hbC5iaW5hcnlfY3Jvc3NfZW50cm9weV93aXRoX2xvZ2l0cygKICAgICAgICBsb2dpdHNbOiwgMDoxXSwgZm9yZWdyb3VuZF90YXJnZXQKICAgICkKICAgIGZvcmVncm91bmRfcHJvYmFiaWxpdHkgPSBsb2dpdHNbOiwgMDoxXS5zaWdtb2lkKCkKICAgIGludGVyc2VjdGlvbiA9IChmb3JlZ3JvdW5kX3Byb2JhYmlsaXR5ICogZm9yZWdyb3VuZF90YXJnZXQpLnN1bShkaW09KDEsIDIsIDMpKQogICAgZGVub21pbmF0b3IgPSBmb3JlZ3JvdW5kX3Byb2JhYmlsaXR5LnN1bShkaW09KDEsIDIsIDMpKSArIGZvcmVncm91bmRfdGFyZ2V0LnN1bSgKICAgICAgICBkaW09KDEsIDIsIDMpCiAgICApCiAgICBkaWNlX2xvc3MgPSAxLjAgLSAoKDIuMCAqIGludGVyc2VjdGlvbiArIDEuMCkgLyAoZGVub21pbmF0b3IgKyAxLjApKS5tZWFuKCkKICAgIGJvdW5kYXJ5X2JjZSA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMoCiAgICAgICAgbG9naXRzWzosIDE6Ml0sIGJvdW5kYXJ5X3RhcmdldCwgcG9zX3dlaWdodD10b3JjaC50ZW5zb3IoCiAgICAgICAgICAgIDUuMCwgZHR5cGU9bG9naXRzLmR0eXBlLCBkZXZpY2U9bG9naXRzLmRldmljZQogICAgICAgICkKICAgICkKICAgIHJldHVybiAwLjQ1ICogZm9yZWdyb3VuZF9iY2UgKyAwLjQ1ICogZGljZV9sb3NzICsgMC4xMCAqIGJvdW5kYXJ5X2JjZQo=","src/filament_baseline/model.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCgoKZGVmIF9ncm91cF9jb3VudChjaGFubmVsczogaW50KSAtPiBpbnQ6CiAgICBmb3IgZ3JvdXBzIGluICg4LCA0LCAyKToKICAgICAgICBpZiBjaGFubmVscyAlIGdyb3VwcyA9PSAwOgogICAgICAgICAgICByZXR1cm4gZ3JvdXBzCiAgICByZXR1cm4gMQoKCmNsYXNzIENvbnZCbG9jayhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzOiBpbnQsIG91dF9jaGFubmVsczogaW50KSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjJkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDMsIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkdyb3VwTm9ybShfZ3JvdXBfY291bnQob3V0X2NoYW5uZWxzKSwgb3V0X2NoYW5uZWxzKSwKICAgICAgICAgICAgbm4uU2lMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQob3V0X2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDMsIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkdyb3VwTm9ybShfZ3JvdXBfY291bnQob3V0X2NoYW5uZWxzKSwgb3V0X2NoYW5uZWxzKSwKICAgICAgICAgICAgbm4uU2lMVShpbnBsYWNlPVRydWUpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB2YWx1ZTogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHNlbGYubGF5ZXJzKHZhbHVlKQoKCmNsYXNzIEZpbGFtZW50VU5ldChubi5Nb2R1bGUpOgogICAgIiIiU21hbGwgZnJvbS1zY3JhdGNoIFUtTmV0IHdpdGggZm9yZWdyb3VuZCBhbmQgaW5zdGFuY2UtYm91bmRhcnkgaGVhZHMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzOiBpbnQgPSAxLCBiYXNlX2NoYW5uZWxzOiBpbnQgPSAxNikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICB3aWR0aHMgPSBbYmFzZV9jaGFubmVscyAqICgyKippbmRleCkgZm9yIGluZGV4IGluIHJhbmdlKDUpXQogICAgICAgIHNlbGYuZW5jb2RlcnMgPSBubi5Nb2R1bGVMaXN0KCkKICAgICAgICBwcmV2aW91cyA9IGluX2NoYW5uZWxzCiAgICAgICAgZm9yIHdpZHRoIGluIHdpZHRoc1s6LTFdOgogICAgICAgICAgICBzZWxmLmVuY29kZXJzLmFwcGVuZChDb252QmxvY2socHJldmlvdXMsIHdpZHRoKSkKICAgICAgICAgICAgcHJldmlvdXMgPSB3aWR0aAogICAgICAgIHNlbGYucG9vbCA9IG5uLk1heFBvb2wyZCgyKQogICAgICAgIHNlbGYuYm90dGxlbmVjayA9IENvbnZCbG9jayh3aWR0aHNbLTJdLCB3aWR0aHNbLTFdKQogICAgICAgIHNlbGYudXBjb252cyA9IG5uLk1vZHVsZUxpc3QoKQogICAgICAgIHNlbGYuZGVjb2RlcnMgPSBubi5Nb2R1bGVMaXN0KCkKICAgICAgICBjdXJyZW50ID0gd2lkdGhzWy0xXQogICAgICAgIGZvciB3aWR0aCBpbiByZXZlcnNlZCh3aWR0aHNbOi0xXSk6CiAgICAgICAgICAgIHNlbGYudXBjb252cy5hcHBlbmQobm4uQ29udlRyYW5zcG9zZTJkKGN1cnJlbnQsIHdpZHRoLCBrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9MikpCiAgICAgICAgICAgIHNlbGYuZGVjb2RlcnMuYXBwZW5kKENvbnZCbG9jayh3aWR0aCAqIDIsIHdpZHRoKSkKICAgICAgICAgICAgY3VycmVudCA9IHdpZHRoCiAgICAgICAgc2VsZi5oZWFkID0gbm4uQ29udjJkKHdpZHRoc1swXSwgMiwga2VybmVsX3NpemU9MSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZTogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgc2tpcHMgPSBbXQogICAgICAgIHZhbHVlID0gaW1hZ2UKICAgICAgICBmb3IgZW5jb2RlciBpbiBzZWxmLmVuY29kZXJzOgogICAgICAgICAgICB2YWx1ZSA9IGVuY29kZXIodmFsdWUpCiAgICAgICAgICAgIHNraXBzLmFwcGVuZCh2YWx1ZSkKICAgICAgICAgICAgdmFsdWUgPSBzZWxmLnBvb2wodmFsdWUpCiAgICAgICAgdmFsdWUgPSBzZWxmLmJvdHRsZW5lY2sodmFsdWUpCiAgICAgICAgZm9yIHVwY29udiwgZGVjb2Rlciwgc2tpcCBpbiB6aXAoCiAgICAgICAgICAgIHNlbGYudXBjb252cywgc2VsZi5kZWNvZGVycywgcmV2ZXJzZWQoc2tpcHMpLCBzdHJpY3Q9VHJ1ZQogICAgICAgICk6CiAgICAgICAgICAgIHZhbHVlID0gdXBjb252KHZhbHVlKQogICAgICAgICAgICBpZiB2YWx1ZS5zaGFwZVstMjpdICE9IHNraXAuc2hhcGVbLTI6XToKICAgICAgICAgICAgICAgIHZhbHVlID0gbm4uZnVuY3Rpb25hbC5pbnRlcnBvbGF0ZSgKICAgICAgICAgICAgICAgICAgICB2YWx1ZSwgc2l6ZT1za2lwLnNoYXBlWy0yOl0sIG1vZGU9ImJpbGluZWFyIiwgYWxpZ25fY29ybmVycz1GYWxzZQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICB2YWx1ZSA9IGRlY29kZXIodG9yY2guY2F0KChza2lwLCB2YWx1ZSksIGRpbT0xKSkKICAgICAgICByZXR1cm4gc2VsZi5oZWFkKHZhbHVlKQo=","src/filament_baseline/model_runner.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gLmRhdGEgaW1wb3J0IGludmVudG9yeSwgbG9hZF9kYXRhc2V0CmZyb20gLmluZmVyZW5jZSBpbXBvcnQgKAogICAgSW5zdGFuY2VDb25maWcsCiAgICBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzLAogICAgcHJlZGljdF9wcm9iYWJpbGl0aWVzLAopCmZyb20gLmxlYXJuaW5nIGltcG9ydCBGaWxhbWVudFBhdGNoRGF0YXNldCwgVGFyZ2V0TW9kZQpmcm9tIC5tb2RlbCBpbXBvcnQgRmlsYW1lbnRVTmV0CmZyb20gLnByZWRpY3RvciBpbXBvcnQgcmVhZF9ncmF5c2NhbGUKZnJvbSAucnVubmVyIGltcG9ydCAoCiAgICBfZ2l0X3N0YXR1cywKICAgIF9yZXNvbHZlX2RhdGFfcm9vdCwKICAgIF93cml0ZV9vYnNlcnZhdGlvbl9zY29yZXMsCiAgICBfd3JpdGVfb3ZlcmxheXMsCiAgICBfd3JpdGVfcHJlZGljdGlvbnMsCikKZnJvbSAucnVudGltZSBpbXBvcnQgaGFyZHdhcmVfaW5mbywgcGFja2FnZV92ZXJzaW9ucwpmcm9tIC5zY29yZXIgaW1wb3J0IHNjb3JlX3ZhbGlkYXRpb24KZnJvbSAuc3BsaXQgaW1wb3J0IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdApmcm9tIC50cmFpbmluZyBpbXBvcnQgdHJhaW5fbW9kZWwKZnJvbSAudmlzdWFscyBpbXBvcnQgd3JpdGVfY29tcGFyaXNvbl9vdmVybGF5cwoKCmRlZiBfd2l0aGluKHBhdGg6IFBhdGgsIHBhcmVudDogUGF0aCkgLT4gYm9vbDoKICAgIHRyeToKICAgICAgICBwYXRoLnJlc29sdmUoKS5yZWxhdGl2ZV90byhwYXJlbnQucmVzb2x2ZSgpKQogICAgICAgIHJldHVybiBUcnVlCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgX2lzX2thZ2dsZV9ydW50aW1lKCkgLT4gYm9vbDoKICAgIHJldHVybiBib29sKG9zLmVudmlyb24uZ2V0KCJLQUdHTEVfS0VSTkVMX1JVTl9UWVBFIikpIG9yICgKICAgICAgICBQYXRoKCIva2FnZ2xlL2lucHV0IikuaXNfZGlyKCkgYW5kIFBhdGgoIi9rYWdnbGUvd29ya2luZyIpLmlzX2RpcigpCiAgICApCgoKZGVmIF9sb2FkX21vZGVsKGNoZWNrcG9pbnRfcGF0aDogUGF0aCwgZGV2aWNlOiB0b3JjaC5kZXZpY2UpIC0+IEZpbGFtZW50VU5ldDoKICAgIGNoZWNrcG9pbnQgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnRfcGF0aCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgIG1vZGVsID0gRmlsYW1lbnRVTmV0KCoqY2hlY2twb2ludC5nZXQoIm1vZGVsX2NvbmZpZyIsIHt9KSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChjaGVja3BvaW50WyJtb2RlbCJdKQogICAgbW9kZWwudG8oZGV2aWNlKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIG1vZGVsLnRvKG1lbW9yeV9mb3JtYXQ9dG9yY2guY2hhbm5lbHNfbGFzdCkKICAgIHJldHVybiBtb2RlbC5ldmFsKCkKCgpkZWYgX3NhdmVfcHJvYmFiaWxpdHlfY2FjaGUoCiAgICBvdXRwdXRfZGlyOiBQYXRoLAogICAgb2JzZXJ2YXRpb246IHN0ciwKICAgIGZvcmVncm91bmQ6IG5wLm5kYXJyYXksCiAgICBib3VuZGFyeTogbnAubmRhcnJheSwKKSAtPiBQYXRoOgogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvdXRwdXRfcGF0aCA9IG91dHB1dF9kaXIgLyBmIntQYXRoKG9ic2VydmF0aW9uKS5zdGVtfS5ucHoiCiAgICBucC5zYXZlel9jb21wcmVzc2VkKAogICAgICAgIG91dHB1dF9wYXRoLAogICAgICAgIGZvcmVncm91bmQ9Zm9yZWdyb3VuZC5hc3R5cGUobnAuZmxvYXQxNiksCiAgICAgICAgYm91bmRhcnk9Ym91bmRhcnkuYXN0eXBlKG5wLmZsb2F0MTYpLAogICAgKQogICAgcmV0dXJuIG91dHB1dF9wYXRoCgoKZGVmIHJ1bl9tb2RlbF9leHBlcmltZW50KAogICAgKiwKICAgIHJlcG9fcm9vdDogc3RyIHwgUGF0aCwKICAgIGRhdGFfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUsCiAgICBydW5fbmFtZTogc3RyLAogICAgdGFyZ2V0X21vZGU6IFRhcmdldE1vZGUgPSAiY29uc2Vuc3VzIiwKICAgIHN0ZXBzOiBpbnQgPSA1XzAwMCwKICAgIGJhdGNoX3NpemU6IGludCA9IDIsCiAgICBwYXRjaF9zaXplOiBpbnQgPSA1MTIsCiAgICB0dGE6IGludCA9IDEsCiAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZDogZmxvYXQgPSAwLjM1LAogICAgYm91bmRhcnlfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwKICAgIGJvdW5kYXJ5X3dlaWdodDogZmxvYXQgPSAxLjUsCiAgICBtaW5faW5zdGFuY2VfYXJlYTogaW50ID0gNTAsCiAgICB0cmFpbl9vYnNlcnZhdGlvbl9saW1pdDogaW50IHwgTm9uZSA9IE5vbmUsCiAgICB2YWxpZGF0aW9uX2xpbWl0OiBpbnQgfCBOb25lID0gTm9uZSwKICAgIG92ZXJsYXlfY291bnQ6IGludCA9IDgsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBpZiBub3QgcmUuZnVsbG1hdGNoKHIiW0EtWmEtejAtOV1bQS1aYS16MC05Ll8tXXswLDYzfSIsIHJ1bl9uYW1lKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJydW5fbmFtZSBtdXN0IHVzZSBvbmx5IGxldHRlcnMsIGRpZ2l0cywgZG90LCB1bmRlcnNjb3JlLCBvciBkYXNoIikKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGthZ2dsZV9ydW50aW1lID0gX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgIGRhdGFzZXRfcm9vdCA9IF9yZXNvbHZlX2RhdGFfcm9vdChkYXRhX3Jvb3QsIHJlcG9fcm9vdCkKICAgIGlmIG5vdCBrYWdnbGVfcnVudGltZSBhbmQgbm90IF93aXRoaW4oZGF0YXNldF9yb290LCByZXBvX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJMb2NhbCBjb21wZXRpdGlvbiBkYXRhIG11c3Qgc3RheSBpbnNpZGUgdGhlIHJlcG9zaXRvcnk6IHtkYXRhc2V0X3Jvb3R9IikKICAgIHJ1bl9yb290ID0gKAogICAgICAgIFBhdGgoIi9rYWdnbGUvd29ya2luZyIpIC8gImVxdWluZXh1cyIgLyBydW5fbmFtZQogICAgICAgIGlmIGthZ2dsZV9ydW50aW1lCiAgICAgICAgZWxzZSByZXBvX3Jvb3QgLyAiYXJ0aWZhY3RzIiAvICJtb2RlbC1leHBlcmltZW50cyIgLyBydW5fbmFtZQogICAgKQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihydW5fcm9vdCwgcmVwb19yb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgZXhwZXJpbWVudCBvdXRwdXRzIG11c3Qgc3RheSBpbiB0aGUgcmVwb3NpdG9yeToge3J1bl9yb290fSIpCiAgICBydW5fcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByZXBvc2l0b3J5X2luZm8gPSBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3QpCgogICAgY29uZmlnID0ganNvbi5sb2FkcygocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgY29uZmlnX3NoYTI1NiA9IGhhc2hsaWIuc2hhMjU2KAogICAgICAgIChyZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIikucmVhZF9ieXRlcygpCiAgICApLmhleGRpZ2VzdCgpCiAgICBkYXRhc2V0ID0gbG9hZF9kYXRhc2V0KGRhdGFzZXRfcm9vdCkKICAgIGRhdGFzZXRfaW52ZW50b3J5ID0gaW52ZW50b3J5KGRhdGFzZXQpCiAgICBleHBlY3RlZF9pbnZlbnRvcnkgPSB7CiAgICAgICAgIm9ic2VydmF0aW9uX2NvdW50IjogNzA3LAogICAgICAgICJhbm5vdGF0b3Jfdmlld19jb3VudCI6IDExNTQsCiAgICAgICAgImFubm90YXRpb25fY291bnQiOiA4MTk5LAogICAgfQogICAgZm9yIGtleSwgZXhwZWN0ZWQgaW4gZXhwZWN0ZWRfaW52ZW50b3J5Lml0ZW1zKCk6CiAgICAgICAgaWYgZGF0YXNldF9pbnZlbnRvcnlba2V5XSAhPSBleHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIHtleHBlY3RlZH0ge2tleX0sIGZvdW5kIHtkYXRhc2V0X2ludmVudG9yeVtrZXldfSIpCgogICAgc3BsaXQgPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBzcGxpdC5zaGEyNTYgIT0gY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiRnJvemVuIHNwbGl0IGRpZ2VzdCBjaGFuZ2VkOiB7c3BsaXQuc2hhMjU2fSAhPSB7Y29uZmlnWydleHBlY3RlZF9zcGxpdF9zaGEyNTYnXX0iCiAgICAgICAgKQogICAgdHJhaW5fbmFtZXMgPSBsaXN0KHNwbGl0LnRyYWluaW5nX25hbWVzKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIGlmIHRyYWluX29ic2VydmF0aW9uX2xpbWl0IGlzIG5vdCBOb25lOgogICAgICAgIHRyYWluX25hbWVzID0gdHJhaW5fbmFtZXNbOnRyYWluX29ic2VydmF0aW9uX2xpbWl0XQogICAgaWYgdmFsaWRhdGlvbl9saW1pdCBpcyBub3QgTm9uZToKICAgICAgICB2YWxpZGF0aW9uX25hbWVzID0gdmFsaWRhdGlvbl9uYW1lc1s6dmFsaWRhdGlvbl9saW1pdF0KICAgIGlmIHNldCh0cmFpbl9uYW1lcykgJiBzZXQodmFsaWRhdGlvbl9uYW1lcyk6CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIlRyYWluaW5nIGFuZCB2YWxpZGF0aW9uIG9ic2VydmF0aW9ucyBvdmVybGFwIikKCiAgICB0cmFpbl9kYXRhc2V0ID0gRmlsYW1lbnRQYXRjaERhdGFzZXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICB0cmFpbl9uYW1lcywKICAgICAgICB0YXJnZXRfbW9kZT10YXJnZXRfbW9kZSwKICAgICAgICBwYXRjaF9zaXplPXBhdGNoX3NpemUsCiAgICAgICAgYXVnbWVudD1UcnVlLAogICAgICAgIHNlZWQ9aW50KGNvbmZpZ1sic2VlZCJdKSwKICAgICkKICAgIHZhbGlkYXRpb25fZGF0YXNldCA9IEZpbGFtZW50UGF0Y2hEYXRhc2V0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICB0YXJnZXRfbW9kZT10YXJnZXRfbW9kZSwKICAgICAgICBwYXRjaF9zaXplPXBhdGNoX3NpemUsCiAgICAgICAgYXVnbWVudD1GYWxzZSwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICApCiAgICBjaGVja3BvaW50X3BhdGggPSBydW5fcm9vdCAvICJiZXN0X21vZGVsLnB0IgogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHByaW50KAogICAgICAgIGYiVHJhaW5pbmcge3RhcmdldF9tb2RlfSBVLU5ldDoge2xlbih0cmFpbl9uYW1lcyl9IHRyYWluIGdyb3VwcywgIgogICAgICAgIGYie2xlbih2YWxpZGF0aW9uX25hbWVzKX0gdmFsaWRhdGlvbiBncm91cHMsIHtzdGVwc30gc3RlcHMsICIKICAgICAgICBmInBhdGNoIHtwYXRjaF9zaXplfSwgYmF0Y2gge2JhdGNoX3NpemV9IiwKICAgICAgICBmbHVzaD1UcnVlLAogICAgKQogICAgdHJhaW5pbmdfcmVzdWx0ID0gdHJhaW5fbW9kZWwoCiAgICAgICAgdHJhaW5fZGF0YXNldD10cmFpbl9kYXRhc2V0LAogICAgICAgIHZhbGlkYXRpb25fZGF0YXNldD12YWxpZGF0aW9uX2RhdGFzZXQsCiAgICAgICAgb3V0cHV0X3BhdGg9Y2hlY2twb2ludF9wYXRoLAogICAgICAgIHN0ZXBzPXN0ZXBzLAogICAgICAgIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9pbnRlcnZhbD1tYXgoMTAwLCBtaW4oNTAwLCBzdGVwcyAvLyA1KSksCiAgICApCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsID0gX2xvYWRfbW9kZWwoY2hlY2twb2ludF9wYXRoLCBkZXZpY2UpCiAgICBjYWNoZV9kaXIgPSBydW5fcm9vdCAvICJwcm9iYWJpbGl0aWVzIgogICAgcHJvYmFiaWxpdHlfY2FjaGVfcGF0aHMgPSB7fQogICAgaW5mZXJlbmNlX3N0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBmb3IgaW5kZXgsIGZpbGVfbmFtZSBpbiBlbnVtZXJhdGUodmFsaWRhdGlvbl9uYW1lcywgc3RhcnQ9MSk6CiAgICAgICAgaW1hZ2UgPSByZWFkX2dyYXlzY2FsZShkYXRhc2V0LnRyYWluX2ltYWdlX2RpciAvIGZpbGVfbmFtZSkKICAgICAgICBmb3JlZ3JvdW5kLCBib3VuZGFyeSA9IHByZWRpY3RfcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgbW9kZWwsCiAgICAgICAgICAgIGltYWdlLAogICAgICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgICAgICBwYXRjaF9zaXplPXBhdGNoX3NpemUsCiAgICAgICAgICAgIHN0cmlkZT1tYXgocGF0Y2hfc2l6ZSAvLyAyLCBpbnQocGF0Y2hfc2l6ZSAqIDAuNzUpKSwKICAgICAgICAgICAgdHRhPXR0YSwKICAgICAgICApCiAgICAgICAgcHJvYmFiaWxpdHlfY2FjaGVfcGF0aHNbZmlsZV9uYW1lXSA9IHN0cigKICAgICAgICAgICAgX3NhdmVfcHJvYmFiaWxpdHlfY2FjaGUoY2FjaGVfZGlyLCBmaWxlX25hbWUsIGZvcmVncm91bmQsIGJvdW5kYXJ5KQogICAgICAgICkKICAgICAgICBpZiBpbmRleCAlIDEwID09IDAgb3IgaW5kZXggPT0gbGVuKHZhbGlkYXRpb25fbmFtZXMpOgogICAgICAgICAgICBwcmludChmIlByZWRpY3RlZCB7aW5kZXh9L3tsZW4odmFsaWRhdGlvbl9uYW1lcyl9IHZhbGlkYXRpb24gb2JzZXJ2YXRpb25zIiwgZmx1c2g9VHJ1ZSkKICAgIGluZmVyZW5jZV9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyZW5jZV9zdGFydGVkCgogICAgaW5zdGFuY2VfY29uZmlnID0gSW5zdGFuY2VDb25maWcoCiAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9Zm9yZWdyb3VuZF90aHJlc2hvbGQsCiAgICAgICAgYm91bmRhcnlfdGhyZXNob2xkPWJvdW5kYXJ5X3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV93ZWlnaHQ9Ym91bmRhcnlfd2VpZ2h0LAogICAgICAgIG1pbl9pbnN0YW5jZV9hcmVhPW1pbl9pbnN0YW5jZV9hcmVhLAogICAgKQogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0XSA9IHt9CiAgICBmb3IgZmlsZV9uYW1lLCBjYWNoZV9wYXRoIGluIHByb2JhYmlsaXR5X2NhY2hlX3BhdGhzLml0ZW1zKCk6CiAgICAgICAgd2l0aCBucC5sb2FkKGNhY2hlX3BhdGgpIGFzIGNhY2hlOgogICAgICAgICAgICBwcmVkaWN0aW9uc1tmaWxlX25hbWVdID0gaW5zdGFuY2VzX2Zyb21fcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgICAgIGNhY2hlWyJmb3JlZ3JvdW5kIl0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgY2FjaGVbImJvdW5kYXJ5Il0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgZmlsZV9uYW1lLAogICAgICAgICAgICAgICAgaW5zdGFuY2VfY29uZmlnLAogICAgICAgICAgICApCgogICAgc2NvcmluZ19zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgbWV0cmljcywgb2JzZXJ2YXRpb25fcm93cyA9IHNjb3JlX3ZhbGlkYXRpb24oCiAgICAgICAgZGF0YXNldCwKICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICB2YWxpZGF0aW9uX25hbWVzLAogICAgICAgIGlvdV90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnWyJpb3VfdGhyZXNob2xkIl0pLAogICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0pLAogICAgKQogICAgc2NvcmluZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNjb3Jpbmdfc3RhcnRlZAogICAgX3dyaXRlX3ByZWRpY3Rpb25zKHJ1bl9yb290IC8gInByZWRpY3Rpb25zLmNzdiIsIHByZWRpY3Rpb25zKQogICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcyhydW5fcm9vdCAvICJwZXJfb2JzZXJ2YXRpb24uY3N2Iiwgb2JzZXJ2YXRpb25fcm93cywgcHJlZGljdGlvbnMpCiAgICBvdmVybGF5cyA9IF93cml0ZV9vdmVybGF5cyhkYXRhc2V0LCBwcmVkaWN0aW9ucywgcnVuX3Jvb3QgLyAib3ZlcmxheXMiLCBvdmVybGF5X2NvdW50KQogICAgY29tcGFyaXNvbl9vdmVybGF5cyA9IHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMoCiAgICAgICAgZGF0YXNldCwgcHJlZGljdGlvbnMsIHJ1bl9yb290IC8gImNvbXBhcmlzb24tb3ZlcmxheXMiLCBvdmVybGF5X2NvdW50CiAgICApCgogICAgaGFyZHdhcmUgPSBoYXJkd2FyZV9pbmZvKCkKICAgIGlmIG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCIpOgogICAgICAgIHJlcG9zaXRvcnlfaW5mb1siY29tbWl0Il0gPSBvcy5lbnZpcm9uWyJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCJdCiAgICAgICAgcmVwb3NpdG9yeV9pbmZvWyJ3b3JrdHJlZV9kaXJ0eSJdID0gVHJ1ZQogICAgcmVwb3NpdG9yeV9pbmZvWyJzb3VyY2VfYnVuZGxlX3NoYTI1NiJdID0gb3MuZW52aXJvbi5nZXQoCiAgICAgICAgIkVRVUlORVhVU19TT1VSQ0VfQlVORExFX1NIQTI1NiIKICAgICkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic3RhdHVzIjogImNvbXBsZXRlIiBpZiB2YWxpZGF0aW9uX2xpbWl0IGlzIE5vbmUgZWxzZSAidmFsaWRhdGlvbl9zdWJzZXQiLAogICAgICAgICJjcmVhdGVkX3V0YyI6IHRpbWUuc3RyZnRpbWUoIiVZLSVtLSVkVCVIOiVNOiVTWiIsIHRpbWUuZ210aW1lKCkpLAogICAgICAgICJydW5fbmFtZSI6IHJ1bl9uYW1lLAogICAgICAgICJyZXBvc2l0b3J5IjogcmVwb3NpdG9yeV9pbmZvLAogICAgICAgICJjb25maWd1cmF0aW9uX3NoYTI1NiI6IGNvbmZpZ19zaGEyNTYsCiAgICAgICAgImthZ2dsZV9ub3RlYm9va192ZXJzaW9uIjogb3MuZW52aXJvbi5nZXQoIkVRVUlORVhVU19OT1RFQk9PS19WRVJTSU9OIiksCiAgICAgICAgImRhdGFzZXRfaW52ZW50b3J5IjogZGF0YXNldF9pbnZlbnRvcnksCiAgICAgICAgInNwbGl0IjogewogICAgICAgICAgICAic2hhMjU2Ijogc3BsaXQuc2hhMjU2LAogICAgICAgICAgICAiZXhwZWN0ZWRfc2hhMjU2IjogY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXSwKICAgICAgICAgICAgInRyYWluaW5nX29ic2VydmF0aW9ucyI6IGxlbihzcGxpdC50cmFpbmluZ19uYW1lcyksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGxlbihzcGxpdC52YWxpZGF0aW9uX25hbWVzKSwKICAgICAgICAgICAgImV2YWx1YXRlZF92YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGxlbih2YWxpZGF0aW9uX25hbWVzKSwKICAgICAgICB9LAogICAgICAgICJtZXRob2QiOiB7CiAgICAgICAgICAgICJhcmNoaXRlY3R1cmUiOiAiZnJvbV9zY3JhdGNoX2ZpdmVfbGV2ZWxfdW5ldF9mb3JlZ3JvdW5kX2JvdW5kYXJ5IiwKICAgICAgICAgICAgImJhc2VfY2hhbm5lbHMiOiAxNiwKICAgICAgICAgICAgInRhcmdldF9tb2RlIjogdGFyZ2V0X21vZGUsCiAgICAgICAgICAgICJwYXRjaF9zaXplIjogcGF0Y2hfc2l6ZSwKICAgICAgICAgICAgInN0ZXBzIjogc3RlcHMsCiAgICAgICAgICAgICJiYXRjaF9zaXplIjogYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgInNlZWQiOiBpbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgICAgICAidHRhIjogdHRhLAogICAgICAgICAgICAiaW5zdGFuY2VfY29uZmlnIjogaW5zdGFuY2VfY29uZmlnLl9fZGljdF9fLAogICAgICAgICAgICAiZXh0ZXJuYWxfc3VwZXJ2aXNlZF93ZWlnaHRzIjogRmFsc2UsCiAgICAgICAgICAgICJleHRlcm5hbF9ncm91bmRfdHJ1dGgiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgICJ0cmFpbmluZyI6IHRyYWluaW5nX3Jlc3VsdCwKICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgInJ1bnRpbWUiOiB7CiAgICAgICAgICAgICoqaGFyZHdhcmUsCiAgICAgICAgICAgICJ0cmFpbmluZ19zZWNvbmRzIjogdHJhaW5pbmdfcmVzdWx0WyJlbGFwc2VkX3NlY29uZHMiXSwKICAgICAgICAgICAgImluZmVyZW5jZV9zZWNvbmRzIjogaW5mZXJlbmNlX3NlY29uZHMsCiAgICAgICAgICAgICJzY29yaW5nX3NlY29uZHMiOiBzY29yaW5nX3NlY29uZHMsCiAgICAgICAgICAgICJlbmRfdG9fZW5kX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZCwKICAgICAgICAgICAgInBhY2thZ2VfdmVyc2lvbnMiOiBwYWNrYWdlX3ZlcnNpb25zKCksCiAgICAgICAgfSwKICAgICAgICAiYXJ0aWZhY3RzIjogewogICAgICAgICAgICAiY2hlY2twb2ludCI6IGNoZWNrcG9pbnRfcGF0aC5uYW1lLAogICAgICAgICAgICAidHJhaW5pbmdfaGlzdG9yeSI6IFBhdGgodHJhaW5pbmdfcmVzdWx0WyJoaXN0b3J5Il0pLm5hbWUsCiAgICAgICAgICAgICJwcm9iYWJpbGl0eV9jYWNoZV9kaXIiOiAicHJvYmFiaWxpdGllcyIsCiAgICAgICAgICAgICJwcmVkaWN0aW9ucyI6ICJwcmVkaWN0aW9ucy5jc3YiLAogICAgICAgICAgICAicGVyX29ic2VydmF0aW9uIjogInBlcl9vYnNlcnZhdGlvbi5jc3YiLAogICAgICAgICAgICAib3ZlcmxheXMiOiBvdmVybGF5cywKICAgICAgICAgICAgImNvbXBhcmlzb25fb3ZlcmxheXMiOiBjb21wYXJpc29uX292ZXJsYXlzLAogICAgICAgIH0sCiAgICB9CiAgICAocnVuX3Jvb3QgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHByaW50KAogICAgICAgIGYiVmFsaWRhdGlvbiBQUToge21ldHJpY3NbJ2dsb2JhbCddWydwcSddOi42Zn07ICIKICAgICAgICBmInNwbGl0IHtzcGxpdC5zaGEyNTZ9OyBvdXRwdXRzIHtydW5fcm9vdH0iLAogICAgICAgIGZsdXNoPVRydWUsCiAgICApCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIG1haW4oYXJndjogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUcmFpbiBhbmQgZXZhbHVhdGUgYSBncm91cGVkIGZpbGFtZW50IFUtTmV0LiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlcG8tcm9vdCIsIGRlZmF1bHQ9c3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ydW4tbmFtZSIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRhcmdldC1tb2RlIiwgY2hvaWNlcz0oImFubm90YXRvciIsICJjb25zZW5zdXMiKSwgZGVmYXVsdD0iY29uc2Vuc3VzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc3RlcHMiLCB0eXBlPWludCwgZGVmYXVsdD01XzAwMCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXBhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD01MTIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXR0YSIsIHR5cGU9aW50LCBjaG9pY2VzPSgxLCAyLCA0LCA4KSwgZGVmYXVsdD0xKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mb3JlZ3JvdW5kLXRocmVzaG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4zNSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYm91bmRhcnktdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXdlaWdodCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MS41KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1taW4taW5zdGFuY2UtYXJlYSIsIHR5cGU9aW50LCBkZWZhdWx0PTUwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10cmFpbi1vYnNlcnZhdGlvbi1saW1pdCIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXZhbGlkYXRpb24tbGltaXQiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdmVybGF5LWNvdW50IiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncyhhcmd2KQogICAgcmVzdWx0ID0gcnVuX21vZGVsX2V4cGVyaW1lbnQoCiAgICAgICAgcmVwb19yb290PWFyZ3MucmVwb19yb290LAogICAgICAgIGRhdGFfcm9vdD1hcmdzLmRhdGFfcm9vdCwKICAgICAgICBydW5fbmFtZT1hcmdzLnJ1bl9uYW1lLAogICAgICAgIHRhcmdldF9tb2RlPWFyZ3MudGFyZ2V0X21vZGUsCiAgICAgICAgc3RlcHM9YXJncy5zdGVwcywKICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICBwYXRjaF9zaXplPWFyZ3MucGF0Y2hfc2l6ZSwKICAgICAgICB0dGE9YXJncy50dGEsCiAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9YXJncy5mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV90aHJlc2hvbGQ9YXJncy5ib3VuZGFyeV90aHJlc2hvbGQsCiAgICAgICAgYm91bmRhcnlfd2VpZ2h0PWFyZ3MuYm91bmRhcnlfd2VpZ2h0LAogICAgICAgIG1pbl9pbnN0YW5jZV9hcmVhPWFyZ3MubWluX2luc3RhbmNlX2FyZWEsCiAgICAgICAgdHJhaW5fb2JzZXJ2YXRpb25fbGltaXQ9YXJncy50cmFpbl9vYnNlcnZhdGlvbl9saW1pdCwKICAgICAgICB2YWxpZGF0aW9uX2xpbWl0PWFyZ3MudmFsaWRhdGlvbl9saW1pdCwKICAgICAgICBvdmVybGF5X2NvdW50PWFyZ3Mub3ZlcmxheV9jb3VudCwKICAgICkKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0WyJtZXRyaWNzIl1bImdsb2JhbCJdLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpKQogICAgcmV0dXJuIDAK","src/filament_baseline/oof.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkS0ZvbGQKCmZyb20gLmRhdGEgaW1wb3J0IGludmVudG9yeSwgbG9hZF9kYXRhc2V0CmZyb20gLmluZmVyZW5jZSBpbXBvcnQgSW5zdGFuY2VDb25maWcsIGluc3RhbmNlc19mcm9tX3Byb2JhYmlsaXRpZXMsIHByZWRpY3RfcHJvYmFiaWxpdGllcwpmcm9tIC5sZWFybmluZyBpbXBvcnQgRmlsYW1lbnRQYXRjaERhdGFzZXQsIFRhcmdldE1vZGUKZnJvbSAubW9kZWxfcnVubmVyIGltcG9ydCBfbG9hZF9tb2RlbApmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IFByZWRpY3RlZEluc3RhbmNlLCByZWFkX2dyYXlzY2FsZQpmcm9tIC5ydW5uZXIgaW1wb3J0ICgKICAgIF9naXRfc3RhdHVzLAogICAgX3Jlc29sdmVfZGF0YV9yb290LAogICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcywKICAgIF93cml0ZV9vdmVybGF5cywKICAgIF93cml0ZV9wcmVkaWN0aW9ucywKKQpmcm9tIC5ydW50aW1lIGltcG9ydCBoYXJkd2FyZV9pbmZvLCBwYWNrYWdlX3ZlcnNpb25zCmZyb20gLnNjb3JlciBpbXBvcnQgc2NvcmVfdmFsaWRhdGlvbgpmcm9tIC5zcGxpdCBpbXBvcnQgY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0CmZyb20gLnRyYWluaW5nIGltcG9ydCB0cmFpbl9tb2RlbApmcm9tIC52aXN1YWxzIGltcG9ydCB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzCgoKZGVmIF9zYXZlX3Byb2JhYmlsaXR5X2NhY2hlKAogICAgb3V0cHV0X2RpcjogUGF0aCwKICAgIGZpbGVfbmFtZTogc3RyLAogICAgZm9yZWdyb3VuZDogbnAubmRhcnJheSwKICAgIGJvdW5kYXJ5OiBucC5uZGFycmF5LAopIC0+IE5vbmU6CiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgb3V0cHV0X2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiLAogICAgICAgIGZvcmVncm91bmQ9Zm9yZWdyb3VuZC5hc3R5cGUobnAuZmxvYXQxNiksCiAgICAgICAgYm91bmRhcnk9Ym91bmRhcnkuYXN0eXBlKG5wLmZsb2F0MTYpLAogICAgKQoKCmRlZiBydW5fdHJhaW5pbmdfcGFydGl0aW9uX29vZigKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lLAogICAgcnVuX25hbWU6IHN0ciwKICAgIHRhcmdldF9tb2RlOiBUYXJnZXRNb2RlID0gImNvbnNlbnN1cyIsCiAgICBmb2xkczogaW50ID0gNSwKICAgIHN0ZXBzOiBpbnQgPSA1XzAwMCwKICAgIGJhdGNoX3NpemU6IGludCA9IDIsCiAgICBwYXRjaF9zaXplOiBpbnQgPSA1MTIsCiAgICB0dGE6IGludCA9IDEsCiAgICBpbnN0YW5jZV9jb25maWc6IEluc3RhbmNlQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBpZiBmb2xkcyA8IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQXQgbGVhc3QgdHdvIE9PRiBmb2xkcyBhcmUgcmVxdWlyZWQiKQogICAgcmVwb19yb290ID0gUGF0aChyZXBvX3Jvb3QpLnJlc29sdmUoKQogICAgZGF0YXNldCA9IGxvYWRfZGF0YXNldChfcmVzb2x2ZV9kYXRhX3Jvb3QoZGF0YV9yb290LCByZXBvX3Jvb3QpKQogICAgaW52ZW50b3J5X2luZm8gPSBpbnZlbnRvcnkoZGF0YXNldCkKICAgIGlmIGludmVudG9yeV9pbmZvWyJvYnNlcnZhdGlvbl9jb3VudCJdICE9IDcwNzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCA3MDcgb2ZmaWNpYWwgdHJhaW5pbmcgb2JzZXJ2YXRpb25zIikKICAgIGlmIGludmVudG9yeV9pbmZvWyJhbm5vdGF0b3Jfdmlld19jb3VudCJdICE9IDFfMTU0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIDEsMTU0IG9mZmljaWFsIGFubm90YXRvciB2aWV3cyIpCiAgICBpZiBpbnZlbnRvcnlfaW5mb1siYW5ub3RhdGlvbl9jb3VudCJdICE9IDhfMTk5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIDgsMTk5IGNvbXBldGl0aW9uLXByb3ZpZGVkIGFubm90YXRpb25zIikKCiAgICB2YWxpZGF0aW9uX2NvbmZpZ19wYXRoID0gcmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIKICAgIHZhbGlkYXRpb25fY29uZmlnID0ganNvbi5sb2FkcygKICAgICAgICB2YWxpZGF0aW9uX2NvbmZpZ19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgKQogICAgdmFsaWRhdGlvbl9jb25maWdfc2hhMjU2ID0gaGFzaGxpYi5zaGEyNTYoCiAgICAgICAgdmFsaWRhdGlvbl9jb25maWdfcGF0aC5yZWFkX2J5dGVzKCkKICAgICkuaGV4ZGlnZXN0KCkKICAgIGZyb3plbl9zcGxpdCA9IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdCgKICAgICAgICBkYXRhc2V0LAogICAgICAgIHJlcG9fcm9vdCAvICJkYXRhIiAvICJzcGxpdHMiLAogICAgICAgIHNlZWQ9aW50KHZhbGlkYXRpb25fY29uZmlnWyJzZWVkIl0pLAogICAgICAgIHZhbGlkYXRpb25fZnJhY3Rpb249ZmxvYXQodmFsaWRhdGlvbl9jb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBmcm96ZW5fc3BsaXQuc2hhMjU2ICE9IHZhbGlkYXRpb25fY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBmcm96ZW4gdmFsaWRhdGlvbiBzcGxpdCBkaWdlc3QgY2hhbmdlZCIpCgogICAgb29mX25hbWVzID0gbGlzdChmcm96ZW5fc3BsaXQudHJhaW5pbmdfbmFtZXMpCiAgICBpZiBzZXQob29mX25hbWVzKSAmIHNldChmcm96ZW5fc3BsaXQudmFsaWRhdGlvbl9uYW1lcyk6CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIkZyb3plbiB2YWxpZGF0aW9uIG9ic2VydmF0aW9ucyBlbnRlcmVkIHRoZSBPT0YgdHJhaW5pbmcgcGFydGl0aW9uIikKICAgIHllYXJzID0gbnAuYXNhcnJheSgKICAgICAgICBbZGF0YXNldC52aWV3c19ieV9vYnNlcnZhdGlvbltuYW1lXVswXS55ZWFyIGZvciBuYW1lIGluIG9vZl9uYW1lc10sCiAgICAgICAgZHR5cGU9bnAuaW50MzIsCiAgICApCiAgICBzcGxpdHRlciA9IFN0cmF0aWZpZWRLRm9sZCgKICAgICAgICBuX3NwbGl0cz1mb2xkcywKICAgICAgICBzaHVmZmxlPVRydWUsCiAgICAgICAgcmFuZG9tX3N0YXRlPWludCh2YWxpZGF0aW9uX2NvbmZpZ1sic2VlZCJdKSwKICAgICkKICAgIGZvbGRfZm9yX25hbWU6IGRpY3Rbc3RyLCBpbnRdID0ge30KICAgIGZvbGRfc3BsaXRzOiBsaXN0W3R1cGxlW2xpc3Rbc3RyXSwgbGlzdFtzdHJdXV0gPSBbXQogICAgZm9yIGZvbGRfaW5kZXgsICh0cmFpbl9pbmRpY2VzLCB2YWxpZGF0aW9uX2luZGljZXMpIGluIGVudW1lcmF0ZSgKICAgICAgICBzcGxpdHRlci5zcGxpdChucC56ZXJvcyhsZW4ob29mX25hbWVzKSksIHllYXJzKSwgc3RhcnQ9MAogICAgKToKICAgICAgICBmb2xkX3RyYWluaW5nX25hbWVzID0gW29vZl9uYW1lc1tpbmRleF0gZm9yIGluZGV4IGluIHRyYWluX2luZGljZXNdCiAgICAgICAgZm9sZF92YWxpZGF0aW9uX25hbWVzID0gW29vZl9uYW1lc1tpbmRleF0gZm9yIGluZGV4IGluIHZhbGlkYXRpb25faW5kaWNlc10KICAgICAgICBpZiBzZXQoZm9sZF90cmFpbmluZ19uYW1lcykgJiBzZXQoZm9sZF92YWxpZGF0aW9uX25hbWVzKToKICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoZiJPYnNlcnZhdGlvbiBvdmVybGFwIGluIE9PRiBmb2xkIHtmb2xkX2luZGV4fSIpCiAgICAgICAgZm9sZF9zcGxpdHMuYXBwZW5kKChmb2xkX3RyYWluaW5nX25hbWVzLCBmb2xkX3ZhbGlkYXRpb25fbmFtZXMpKQogICAgICAgIGZvciBuYW1lIGluIGZvbGRfdmFsaWRhdGlvbl9uYW1lczoKICAgICAgICAgICAgaWYgbmFtZSBpbiBmb2xkX2Zvcl9uYW1lOgogICAgICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoZiJPT0Ygb2JzZXJ2YXRpb24gYXBwZWFycyBpbiBtdWx0aXBsZSBmb2xkczoge25hbWV9IikKICAgICAgICAgICAgZm9sZF9mb3JfbmFtZVtuYW1lXSA9IGZvbGRfaW5kZXgKICAgIGlmIHNldChmb2xkX2Zvcl9uYW1lKSAhPSBzZXQob29mX25hbWVzKToKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiT09GIGZvbGRzIGRvIG5vdCBjb3ZlciB0aGUgY29tcGxldGUgdHJhaW5pbmcgcGFydGl0aW9uIikKCiAgICBtYW5pZmVzdF9wYXRoID0gcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIgLyBmIm9ic2VydmF0aW9uX29vZl9zZWVke3ZhbGlkYXRpb25fY29uZmlnWydzZWVkJ119X3tmb2xkc31mb2xkLmNzdiIKICAgIHdpdGggbWFuaWZlc3RfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKAogICAgICAgICAgICBzdHJlYW0sCiAgICAgICAgICAgIGZpZWxkbmFtZXM9KCJmaWxlX25hbWUiLCAieWVhciIsICJzaXRlIiwgIm9vZl9mb2xkIiksCiAgICAgICAgICAgIGxpbmV0ZXJtaW5hdG9yPSJcbiIsCiAgICAgICAgKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIGZpbGVfbmFtZSBpbiBvb2ZfbmFtZXM6CiAgICAgICAgICAgIHZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV1bMF0KICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJmaWxlX25hbWUiOiBmaWxlX25hbWUsCiAgICAgICAgICAgICAgICAgICAgInllYXIiOiB2aWV3LnllYXIsCiAgICAgICAgICAgICAgICAgICAgInNpdGUiOiB2aWV3LnNpdGUsCiAgICAgICAgICAgICAgICAgICAgIm9vZl9mb2xkIjogZm9sZF9mb3JfbmFtZVtmaWxlX25hbWVdLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICBvb2ZfbWFuaWZlc3Rfc2hhMjU2ID0gaGFzaGxpYi5zaGEyNTYobWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpCgogICAgcnVuX3Jvb3QgPSByZXBvX3Jvb3QgLyAiYXJ0aWZhY3RzIiAvICJvb2YtZXhwZXJpbWVudHMiIC8gcnVuX25hbWUKICAgIHJ1bl9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJlcG9zaXRvcnlfaW5mbyA9IF9naXRfc3RhdHVzKHJlcG9fcm9vdCkKICAgIGNhbmRpZGF0ZV9jb25maWcgPSBpbnN0YW5jZV9jb25maWcgb3IgSW5zdGFuY2VDb25maWcoKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIGZvbGRfcmVzdWx0cyA9IFtdCiAgICBhbGxfb29mX3ByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV1dID0ge30KICAgIGFsbF9vYnNlcnZhdGlvbl9yb3dzID0gW10KICAgIHRvdGFsX3RyYWluaW5nX3NlY29uZHMgPSAwLjAKICAgIHRvdGFsX2luZmVyZW5jZV9zZWNvbmRzID0gMC4wCiAgICB0b3RhbF9zY29yaW5nX3NlY29uZHMgPSAwLjAKICAgIHBlYWtfZ3B1X21lbW9yeSA9IDAKICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCgogICAgZm9yIGZvbGRfaW5kZXgsIChmb2xkX3RyYWluaW5nX25hbWVzLCBmb2xkX3ZhbGlkYXRpb25fbmFtZXMpIGluIGVudW1lcmF0ZSgKICAgICAgICBmb2xkX3NwbGl0cywgc3RhcnQ9MAogICAgKToKICAgICAgICBmb2xkX3Jvb3QgPSBydW5fcm9vdCAvIGYiZm9sZC17Zm9sZF9pbmRleH0iCiAgICAgICAgZm9sZF9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICB0cmFpbmluZ19kYXRhc2V0ID0gRmlsYW1lbnRQYXRjaERhdGFzZXQoCiAgICAgICAgICAgIGRhdGFzZXQsCiAgICAgICAgICAgIGZvbGRfdHJhaW5pbmdfbmFtZXMsCiAgICAgICAgICAgIHRhcmdldF9tb2RlPXRhcmdldF9tb2RlLAogICAgICAgICAgICBwYXRjaF9zaXplPXBhdGNoX3NpemUsCiAgICAgICAgICAgIGF1Z21lbnQ9VHJ1ZSwKICAgICAgICAgICAgc2VlZD1pbnQodmFsaWRhdGlvbl9jb25maWdbInNlZWQiXSkgKyBmb2xkX2luZGV4LAogICAgICAgICkKICAgICAgICB2YWxpZGF0aW9uX2RhdGFzZXQgPSBGaWxhbWVudFBhdGNoRGF0YXNldCgKICAgICAgICAgICAgZGF0YXNldCwKICAgICAgICAgICAgZm9sZF92YWxpZGF0aW9uX25hbWVzLAogICAgICAgICAgICB0YXJnZXRfbW9kZT10YXJnZXRfbW9kZSwKICAgICAgICAgICAgcGF0Y2hfc2l6ZT1wYXRjaF9zaXplLAogICAgICAgICAgICBhdWdtZW50PUZhbHNlLAogICAgICAgICAgICBzZWVkPWludCh2YWxpZGF0aW9uX2NvbmZpZ1sic2VlZCJdKSArIGZvbGRfaW5kZXgsCiAgICAgICAgKQogICAgICAgIGNoZWNrcG9pbnRfcGF0aCA9IGZvbGRfcm9vdCAvICJiZXN0X21vZGVsLnB0IgogICAgICAgIHByaW50KAogICAgICAgICAgICBmIlN0YXJ0aW5nIE9PRiBmb2xkIHtmb2xkX2luZGV4ICsgMX0ve2ZvbGRzfTogIgogICAgICAgICAgICBmIntsZW4oZm9sZF90cmFpbmluZ19uYW1lcyl9IHRyYWluLCB7bGVuKGZvbGRfdmFsaWRhdGlvbl9uYW1lcyl9IHZhbGlkYXRpb24gZ3JvdXBzIiwKICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICApCiAgICAgICAgdHJhaW5pbmdfcmVzdWx0ID0gdHJhaW5fbW9kZWwoCiAgICAgICAgICAgIHRyYWluX2RhdGFzZXQ9dHJhaW5pbmdfZGF0YXNldCwKICAgICAgICAgICAgdmFsaWRhdGlvbl9kYXRhc2V0PXZhbGlkYXRpb25fZGF0YXNldCwKICAgICAgICAgICAgb3V0cHV0X3BhdGg9Y2hlY2twb2ludF9wYXRoLAogICAgICAgICAgICBzdGVwcz1zdGVwcywKICAgICAgICAgICAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAogICAgICAgICAgICBzZWVkPWludCh2YWxpZGF0aW9uX2NvbmZpZ1sic2VlZCJdKSArIGZvbGRfaW5kZXgsCiAgICAgICAgICAgIHZhbGlkYXRpb25faW50ZXJ2YWw9bWF4KDEwMCwgbWluKDUwMCwgc3RlcHMgLy8gNSkpLAogICAgICAgICkKICAgICAgICB0b3RhbF90cmFpbmluZ19zZWNvbmRzICs9IGZsb2F0KHRyYWluaW5nX3Jlc3VsdFsiZWxhcHNlZF9zZWNvbmRzIl0pCiAgICAgICAgcGVha19ncHVfbWVtb3J5ID0gbWF4KAogICAgICAgICAgICBwZWFrX2dwdV9tZW1vcnksIGludCh0cmFpbmluZ19yZXN1bHRbInBlYWtfZ3B1X21lbW9yeV9ieXRlcyJdKQogICAgICAgICkKCiAgICAgICAgbW9kZWwgPSBfbG9hZF9tb2RlbChjaGVja3BvaW50X3BhdGgsIGRldmljZSkKICAgICAgICBmb2xkX3ByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV1dID0ge30KICAgICAgICBpbmZlcmVuY2Vfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBmb3IgaW5kZXgsIGZpbGVfbmFtZSBpbiBlbnVtZXJhdGUoZm9sZF92YWxpZGF0aW9uX25hbWVzLCBzdGFydD0xKToKICAgICAgICAgICAgaW1hZ2UgPSByZWFkX2dyYXlzY2FsZShkYXRhc2V0LnRyYWluX2ltYWdlX2RpciAvIGZpbGVfbmFtZSkKICAgICAgICAgICAgZm9yZWdyb3VuZCwgYm91bmRhcnkgPSBwcmVkaWN0X3Byb2JhYmlsaXRpZXMoCiAgICAgICAgICAgICAgICBtb2RlbCwKICAgICAgICAgICAgICAgIGltYWdlLAogICAgICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgICAgIHBhdGNoX3NpemU9cGF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgIHN0cmlkZT1tYXgocGF0Y2hfc2l6ZSAvLyAyLCBpbnQocGF0Y2hfc2l6ZSAqIDAuNzUpKSwKICAgICAgICAgICAgICAgIHR0YT10dGEsCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9sZF9wcmVkaWN0aW9uc1tmaWxlX25hbWVdID0gaW5zdGFuY2VzX2Zyb21fcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgICAgIGZvcmVncm91bmQsIGJvdW5kYXJ5LCBmaWxlX25hbWUsIGNhbmRpZGF0ZV9jb25maWcKICAgICAgICAgICAgKQogICAgICAgICAgICBfc2F2ZV9wcm9iYWJpbGl0eV9jYWNoZSgKICAgICAgICAgICAgICAgIGZvbGRfcm9vdCAvICJwcm9iYWJpbGl0aWVzIiwgZmlsZV9uYW1lLCBmb3JlZ3JvdW5kLCBib3VuZGFyeQogICAgICAgICAgICApCiAgICAgICAgICAgIGlmIGluZGV4ICUgMjAgPT0gMCBvciBpbmRleCA9PSBsZW4oZm9sZF92YWxpZGF0aW9uX25hbWVzKToKICAgICAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgICAgIGYiRm9sZCB7Zm9sZF9pbmRleCArIDF9OiBwcmVkaWN0ZWQgIgogICAgICAgICAgICAgICAgICAgIGYie2luZGV4fS97bGVuKGZvbGRfdmFsaWRhdGlvbl9uYW1lcyl9IG9ic2VydmF0aW9ucyIsCiAgICAgICAgICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBpbmZlcmVuY2Vfc3RhcnRlZAogICAgICAgIHRvdGFsX2luZmVyZW5jZV9zZWNvbmRzICs9IGluZmVyZW5jZV9zZWNvbmRzCgogICAgICAgIHNjb3Jpbmdfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBmb2xkX21ldHJpY3MsIGZvbGRfb2JzZXJ2YXRpb25fcm93cyA9IHNjb3JlX3ZhbGlkYXRpb24oCiAgICAgICAgICAgIGRhdGFzZXQsCiAgICAgICAgICAgIGZvbGRfcHJlZGljdGlvbnMsCiAgICAgICAgICAgIGZvbGRfdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdCh2YWxpZGF0aW9uX2NvbmZpZ1siaW91X3RocmVzaG9sZCJdKSwKICAgICAgICAgICAgZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZD1mbG9hdCgKICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0KICAgICAgICAgICAgKSwKICAgICAgICApCiAgICAgICAgc2NvcmluZ19zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNjb3Jpbmdfc3RhcnRlZAogICAgICAgIHRvdGFsX3Njb3Jpbmdfc2Vjb25kcyArPSBzY29yaW5nX3NlY29uZHMKICAgICAgICBfd3JpdGVfcHJlZGljdGlvbnMoZm9sZF9yb290IC8gInByZWRpY3Rpb25zLmNzdiIsIGZvbGRfcHJlZGljdGlvbnMpCiAgICAgICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcygKICAgICAgICAgICAgZm9sZF9yb290IC8gInBlcl9vYnNlcnZhdGlvbi5jc3YiLAogICAgICAgICAgICBmb2xkX29ic2VydmF0aW9uX3Jvd3MsCiAgICAgICAgICAgIGZvbGRfcHJlZGljdGlvbnMsCiAgICAgICAgKQogICAgICAgIGZvbGRfcmVzdWx0ID0gewogICAgICAgICAgICAiZm9sZCI6IGZvbGRfaW5kZXgsCiAgICAgICAgICAgICJ0cmFpbmluZ19vYnNlcnZhdGlvbnMiOiBsZW4oZm9sZF90cmFpbmluZ19uYW1lcyksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGxlbihmb2xkX3ZhbGlkYXRpb25fbmFtZXMpLAogICAgICAgICAgICAidHJhaW5pbmciOiB0cmFpbmluZ19yZXN1bHQsCiAgICAgICAgICAgICJtZXRyaWNzIjogZm9sZF9tZXRyaWNzLAogICAgICAgICAgICAiaW5mZXJlbmNlX3NlY29uZHMiOiBpbmZlcmVuY2Vfc2Vjb25kcywKICAgICAgICAgICAgInNjb3Jpbmdfc2Vjb25kcyI6IHNjb3Jpbmdfc2Vjb25kcywKICAgICAgICAgICAgImNoZWNrcG9pbnQiOiBzdHIoY2hlY2twb2ludF9wYXRoLnJlbGF0aXZlX3RvKHJ1bl9yb290KSksCiAgICAgICAgfQogICAgICAgIChmb2xkX3Jvb3QgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAganNvbi5kdW1wcyhmb2xkX3Jlc3VsdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsCiAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgKQogICAgICAgIGZvbGRfcmVzdWx0cy5hcHBlbmQoZm9sZF9yZXN1bHQpCiAgICAgICAgYWxsX29vZl9wcmVkaWN0aW9ucy51cGRhdGUoZm9sZF9wcmVkaWN0aW9ucykKICAgICAgICBhbGxfb2JzZXJ2YXRpb25fcm93cy5leHRlbmQoZm9sZF9vYnNlcnZhdGlvbl9yb3dzKQogICAgICAgIHByaW50KAogICAgICAgICAgICBmIk9PRiBmb2xkIHtmb2xkX2luZGV4ICsgMX0ve2ZvbGRzfSBQUT0iCiAgICAgICAgICAgIGYie2ZvbGRfbWV0cmljc1snZ2xvYmFsJ11bJ3BxJ106LjZmfSIsCiAgICAgICAgICAgIGZsdXNoPVRydWUsCiAgICAgICAgKQoKICAgIGlmIHNldChhbGxfb29mX3ByZWRpY3Rpb25zKSAhPSBzZXQob29mX25hbWVzKToKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiT09GIHByZWRpY3Rpb24gY292ZXJhZ2UgZG9lcyBub3QgbWF0Y2ggdGhlIHRyYWluaW5nIHBhcnRpdGlvbiIpCiAgICBhZ2dyZWdhdGVfdHAgPSBzdW0oaXRlbVsibWV0cmljcyJdWyJnbG9iYWwiXVsidHAiXSBmb3IgaXRlbSBpbiBmb2xkX3Jlc3VsdHMpCiAgICBhZ2dyZWdhdGVfZnAgPSBzdW0oaXRlbVsibWV0cmljcyJdWyJnbG9iYWwiXVsiZnAiXSBmb3IgaXRlbSBpbiBmb2xkX3Jlc3VsdHMpCiAgICBhZ2dyZWdhdGVfZm4gPSBzdW0oaXRlbVsibWV0cmljcyJdWyJnbG9iYWwiXVsiZm4iXSBmb3IgaXRlbSBpbiBmb2xkX3Jlc3VsdHMpCiAgICBhZ2dyZWdhdGVfaW91ID0gc3VtKAogICAgICAgIGl0ZW1bIm1ldHJpY3MiXVsiZ2xvYmFsIl1bInRwX2lvdV9zdW0iXSBmb3IgaXRlbSBpbiBmb2xkX3Jlc3VsdHMKICAgICkKICAgIGRlbm9taW5hdG9yID0gYWdncmVnYXRlX3RwICsgMC41ICogYWdncmVnYXRlX2ZwICsgMC41ICogYWdncmVnYXRlX2ZuCiAgICBhZ2dyZWdhdGVfZ2xvYmFsID0gewogICAgICAgICJwcSI6IGFnZ3JlZ2F0ZV9pb3UgLyBkZW5vbWluYXRvciBpZiBkZW5vbWluYXRvciBlbHNlIDAuMCwKICAgICAgICAidHAiOiBhZ2dyZWdhdGVfdHAsCiAgICAgICAgImZwIjogYWdncmVnYXRlX2ZwLAogICAgICAgICJmbiI6IGFnZ3JlZ2F0ZV9mbiwKICAgICAgICAidHBfaW91X3N1bSI6IGFnZ3JlZ2F0ZV9pb3UsCiAgICAgICAgImd0X2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiOiBzdW0oCiAgICAgICAgICAgIGl0ZW1bIm1ldHJpY3MiXVsiZ2xvYmFsIl1bImd0X2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiXQogICAgICAgICAgICBmb3IgaXRlbSBpbiBmb2xkX3Jlc3VsdHMKICAgICAgICApLAogICAgICAgICJwcmVkaWN0aW9uX2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiOiBzdW0oCiAgICAgICAgICAgIGl0ZW1bIm1ldHJpY3MiXVsiZ2xvYmFsIl1bInByZWRpY3Rpb25fY291bnRfYWNyb3NzX2Fubm90YXRvcl92aWV3cyJdCiAgICAgICAgICAgIGZvciBpdGVtIGluIGZvbGRfcmVzdWx0cwogICAgICAgICksCiAgICAgICAgImFubm90YXRvcl92aWV3X2NvdW50Ijogc3VtKAogICAgICAgICAgICBpdGVtWyJtZXRyaWNzIl1bImdsb2JhbCJdWyJhbm5vdGF0b3Jfdmlld19jb3VudCJdCiAgICAgICAgICAgIGZvciBpdGVtIGluIGZvbGRfcmVzdWx0cwogICAgICAgICksCiAgICB9CiAgICBfd3JpdGVfcHJlZGljdGlvbnMocnVuX3Jvb3QgLyAib29mLXByZWRpY3Rpb25zLmNzdiIsIGFsbF9vb2ZfcHJlZGljdGlvbnMpCiAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKAogICAgICAgIHJ1bl9yb290IC8gIm9vZi1wZXItb2JzZXJ2YXRpb24uY3N2IiwKICAgICAgICBhbGxfb2JzZXJ2YXRpb25fcm93cywKICAgICAgICBhbGxfb29mX3ByZWRpY3Rpb25zLAogICAgKQogICAgX3dyaXRlX292ZXJsYXlzKGRhdGFzZXQsIGFsbF9vb2ZfcHJlZGljdGlvbnMsIHJ1bl9yb290IC8gIm92ZXJsYXlzIiwgMTIpCiAgICB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzKAogICAgICAgIGRhdGFzZXQsIGFsbF9vb2ZfcHJlZGljdGlvbnMsIHJ1bl9yb290IC8gImNvbXBhcmlzb24tb3ZlcmxheXMiLCAxMgogICAgKQoKICAgIHJlc3VsdDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIsCiAgICAgICAgImNyZWF0ZWRfdXRjIjogdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVNaIiwgdGltZS5nbXRpbWUoKSksCiAgICAgICAgInJ1bl9uYW1lIjogcnVuX25hbWUsCiAgICAgICAgInJlcG9zaXRvcnkiOiByZXBvc2l0b3J5X2luZm8sCiAgICAgICAgImNvbmZpZ3VyYXRpb25fc2hhMjU2IjogdmFsaWRhdGlvbl9jb25maWdfc2hhMjU2LAogICAgICAgICJkYXRhc2V0X2ludmVudG9yeSI6IGludmVudG9yeV9pbmZvLAogICAgICAgICJkYXRhX2JvdW5kYXJ5IjogewogICAgICAgICAgICAidHJhaW5pbmdfbGFiZWxzIjogImNvbXBldGl0aW9uLXByb3ZpZGVkIHRyYWluaW5nIGFubm90YXRpb25zIG9ubHkiLAogICAgICAgICAgICAiZXhjbHVkZWRfaG9sZG91dCI6ICJhbGwgMTQyIG9ic2VydmF0aW9ucyBpbiB0aGUgZnJvemVuIHZhbGlkYXRpb24gbWFuaWZlc3QiLAogICAgICAgICAgICAiZXh0ZXJuYWxfZ3JvdW5kX3RydXRoIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICAic3BsaXRzIjogewogICAgICAgICAgICAiZnJvemVuX2hvbGRvdXRfc2hhMjU2IjogZnJvemVuX3NwbGl0LnNoYTI1NiwKICAgICAgICAgICAgIm9vZl9tYW5pZmVzdCI6IHN0cihtYW5pZmVzdF9wYXRoKSwKICAgICAgICAgICAgIm9vZl9tYW5pZmVzdF9zaGEyNTYiOiBvb2ZfbWFuaWZlc3Rfc2hhMjU2LAogICAgICAgICAgICAib29mX29ic2VydmF0aW9ucyI6IGxlbihvb2ZfbmFtZXMpLAogICAgICAgICAgICAiZm9sZF9jb3VudCI6IGZvbGRzLAogICAgICAgIH0sCiAgICAgICAgIm1ldGhvZCI6IHsKICAgICAgICAgICAgImFyY2hpdGVjdHVyZSI6ICJmcm9tX3NjcmF0Y2hfZml2ZV9sZXZlbF91bmV0X2ZvcmVncm91bmRfYm91bmRhcnkiLAogICAgICAgICAgICAidGFyZ2V0X21vZGUiOiB0YXJnZXRfbW9kZSwKICAgICAgICAgICAgInBhdGNoX3NpemUiOiBwYXRjaF9zaXplLAogICAgICAgICAgICAic3RlcHNfcGVyX2ZvbGQiOiBzdGVwcywKICAgICAgICAgICAgImJhdGNoX3NpemUiOiBiYXRjaF9zaXplLAogICAgICAgICAgICAidHRhIjogdHRhLAogICAgICAgICAgICAiaW5zdGFuY2VfY29uZmlnIjogY2FuZGlkYXRlX2NvbmZpZy5fX2RpY3RfXywKICAgICAgICAgICAgImV4dGVybmFsX3N1cGVydmlzZWRfd2VpZ2h0cyI6IEZhbHNlLAogICAgICAgIH0sCiAgICAgICAgImZvbGRzIjogZm9sZF9yZXN1bHRzLAogICAgICAgICJhZ2dyZWdhdGVfb29mIjogYWdncmVnYXRlX2dsb2JhbCwKICAgICAgICAicnVudGltZSI6IHsKICAgICAgICAgICAgKipoYXJkd2FyZV9pbmZvKCksCiAgICAgICAgICAgICJ0cmFpbmluZ19zZWNvbmRzX3RvdGFsIjogdG90YWxfdHJhaW5pbmdfc2Vjb25kcywKICAgICAgICAgICAgImluZmVyZW5jZV9zZWNvbmRzX3RvdGFsIjogdG90YWxfaW5mZXJlbmNlX3NlY29uZHMsCiAgICAgICAgICAgICJzY29yaW5nX3NlY29uZHNfdG90YWwiOiB0b3RhbF9zY29yaW5nX3NlY29uZHMsCiAgICAgICAgICAgICJlbmRfdG9fZW5kX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZCwKICAgICAgICAgICAgInBlYWtfZ3B1X21lbW9yeV9ieXRlcyI6IHBlYWtfZ3B1X21lbW9yeSwKICAgICAgICAgICAgInBhY2thZ2VfdmVyc2lvbnMiOiBwYWNrYWdlX3ZlcnNpb25zKCksCiAgICAgICAgfSwKICAgICAgICAiYXJ0aWZhY3RzIjogewogICAgICAgICAgICAib29mX3ByZWRpY3Rpb25zIjogIm9vZi1wcmVkaWN0aW9ucy5jc3YiLAogICAgICAgICAgICAicGVyX29ic2VydmF0aW9uIjogIm9vZi1wZXItb2JzZXJ2YXRpb24uY3N2IiwKICAgICAgICAgICAgIm92ZXJsYXlzIjogIm92ZXJsYXlzIiwKICAgICAgICAgICAgImNvbXBhcmlzb25fb3ZlcmxheXMiOiAiY29tcGFyaXNvbi1vdmVybGF5cyIsCiAgICAgICAgfSwKICAgIH0KICAgIChydW5fcm9vdCAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IgogICAgKQogICAgcHJpbnQoCiAgICAgICAgZiJBZ2dyZWdhdGUgZ3JvdXBlZCB7Zm9sZHN9LWZvbGQgT09GIFBROiB7YWdncmVnYXRlX2dsb2JhbFsncHEnXTouNmZ9OyAiCiAgICAgICAgZiJzcGxpdCB7b29mX21hbmlmZXN0X3NoYTI1Nn07IG91dHB1dHMge3J1bl9yb290fSIsCiAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICkKICAgIHJldHVybiByZXN1bHQK","src/filament_baseline/predictor.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucApmcm9tIHB5Y29jb3Rvb2xzIGltcG9ydCBtYXNrIGFzIG1hc2tfdXRpbHMKCmZyb20gLmRhdGEgaW1wb3J0IElNQUdFX1NJWkUKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBCYXNlbGluZUNvbmZpZzoKICAgIGJsdXJfa2VybmVsOiBpbnQgPSAzMQogICAgc29sYXJfZGlza19yYWRpdXNfZnJhY3Rpb246IGZsb2F0ID0gMC40NAogICAgY2xvc2luZ19rZXJuZWw6IGludCA9IDMKICAgIGNvbXBvbmVudF9jb25uZWN0aXZpdHk6IGludCA9IDgKICAgIG1pbl9jb21wb25lbnRfYXJlYTogaW50ID0gNTAKICAgIG1heF9jb21wb25lbnRfYXJlYTogaW50ID0gNTBfMDAwCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUHJlZGljdGVkSW5zdGFuY2U6CiAgICBmaWxhbWVudF9pZDogc3RyCiAgICBpbWFnZV9zdGVtOiBzdHIKICAgIGFyZWE6IGludAogICAgcmxlOiBkaWN0W3N0ciwgQW55XQoKCmRlZiBlbmNvZGVfYmluYXJ5X21hc2soYmluYXJ5X21hc2s6IG5wLm5kYXJyYXkpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgaWYgYmluYXJ5X21hc2submRpbSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBhIDJEIG1hc2ssIGdvdCBzaGFwZSB7YmluYXJ5X21hc2suc2hhcGV9IikKICAgIGNvbnRpZ3VvdXMgPSBucC5hc2ZvcnRyYW5hcnJheSgoYmluYXJ5X21hc2sgPiAwKS5hc3R5cGUobnAudWludDgpKQogICAgZW5jb2RlZCA9IG1hc2tfdXRpbHMuZW5jb2RlKGNvbnRpZ3VvdXMpCiAgICBjb3VudHMgPSBlbmNvZGVkWyJjb3VudHMiXQogICAgaWYgaXNpbnN0YW5jZShjb3VudHMsIGJ5dGVzKToKICAgICAgICBjb3VudHMgPSBjb3VudHMuZGVjb2RlKCJhc2NpaSIpCiAgICByZXR1cm4geyJzaXplIjogW2ludChiaW5hcnlfbWFzay5zaGFwZVswXSksIGludChiaW5hcnlfbWFzay5zaGFwZVsxXSldLCAiY291bnRzIjogY291bnRzfQoKCmRlZiBlbmNvZGVfY3JvcHBlZF9iaW5hcnlfbWFzaygKICAgIGJpbmFyeV9jcm9wOiBucC5uZGFycmF5LAogICAgeF9vZmZzZXQ6IGludCwKICAgIHlfb2Zmc2V0OiBpbnQsCiAgICBmdWxsX3NoYXBlOiB0dXBsZVtpbnQsIGludF0sCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJFbmNvZGUgYSBzbWFsbCBtYXNrIGNyb3AgaW50byBhIGZ1bGwtaW1hZ2UgQ09DTyBSTEUgd2l0aG91dCBhIGZ1bGwgY2FudmFzLiIiIgogICAgaWYgYmluYXJ5X2Nyb3AubmRpbSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGEgMkQgYmluYXJ5IGNyb3AiKQogICAgaGVpZ2h0LCB3aWR0aCA9IGZ1bGxfc2hhcGUKICAgIGNyb3BfaGVpZ2h0LCBjcm9wX3dpZHRoID0gYmluYXJ5X2Nyb3Auc2hhcGUKICAgIGlmICgKICAgICAgICBoZWlnaHQgPCAxCiAgICAgICAgb3Igd2lkdGggPCAxCiAgICAgICAgb3IgeF9vZmZzZXQgPCAwCiAgICAgICAgb3IgeV9vZmZzZXQgPCAwCiAgICAgICAgb3IgeF9vZmZzZXQgKyBjcm9wX3dpZHRoID4gd2lkdGgKICAgICAgICBvciB5X29mZnNldCArIGNyb3BfaGVpZ2h0ID4gaGVpZ2h0CiAgICApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkJpbmFyeSBjcm9wIGZhbGxzIG91dHNpZGUgdGhlIGZ1bGwgaW1hZ2UgYm91bmRzIikKICAgIGNyb3AgPSBucC5hc2FycmF5KGJpbmFyeV9jcm9wLCBkdHlwZT1ib29sKQogICAgaWYgbm90IG5wLmFueShjcm9wKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3QgZW5jb2RlIGFuIGVtcHR5IGNvbXBvbmVudCBjcm9wIikKCiAgICBydW5fdmFsdWVzOiBsaXN0W2ludF0gPSBbXQogICAgY3VycmVudF92YWx1ZSA9IDAKICAgIGN1cnJlbnRfbGVuZ3RoID0gMAoKICAgIGRlZiBhcHBlbmRfcnVuKHZhbHVlOiBpbnQsIGxlbmd0aDogaW50KSAtPiBOb25lOgogICAgICAgIG5vbmxvY2FsIGN1cnJlbnRfdmFsdWUsIGN1cnJlbnRfbGVuZ3RoCiAgICAgICAgaWYgbGVuZ3RoIDw9IDA6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHZhbHVlID0gaW50KHZhbHVlKQogICAgICAgIGlmIHZhbHVlID09IGN1cnJlbnRfdmFsdWU6CiAgICAgICAgICAgIGN1cnJlbnRfbGVuZ3RoICs9IGxlbmd0aAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJ1bl92YWx1ZXMuYXBwZW5kKGN1cnJlbnRfbGVuZ3RoKQogICAgICAgICAgICBjdXJyZW50X3ZhbHVlID0gdmFsdWUKICAgICAgICAgICAgY3VycmVudF9sZW5ndGggPSBsZW5ndGgKCiAgICBhcHBlbmRfcnVuKDAsIHhfb2Zmc2V0ICogaGVpZ2h0KQogICAgZm9yIGNvbHVtbl9pbmRleCBpbiByYW5nZShjcm9wX3dpZHRoKToKICAgICAgICBjb2x1bW4gPSBjcm9wWzosIGNvbHVtbl9pbmRleF0KICAgICAgICBhcHBlbmRfcnVuKDAsIHlfb2Zmc2V0KQogICAgICAgIHN0YXJ0cyA9IG5wLnJfWzAsIG5wLmZsYXRub256ZXJvKGNvbHVtblsxOl0gIT0gY29sdW1uWzotMV0pICsgMV0KICAgICAgICBlbmRzID0gbnAucl9bc3RhcnRzWzE6XSwgY3JvcF9oZWlnaHRdCiAgICAgICAgZm9yIHN0YXJ0LCBlbmQgaW4gemlwKHN0YXJ0cywgZW5kcywgc3RyaWN0PVRydWUpOgogICAgICAgICAgICBhcHBlbmRfcnVuKGludChjb2x1bW5bc3RhcnRdKSwgaW50KGVuZCAtIHN0YXJ0KSkKICAgICAgICBhcHBlbmRfcnVuKDAsIGhlaWdodCAtIHlfb2Zmc2V0IC0gY3JvcF9oZWlnaHQpCiAgICBhcHBlbmRfcnVuKDAsICh3aWR0aCAtIHhfb2Zmc2V0IC0gY3JvcF93aWR0aCkgKiBoZWlnaHQpCiAgICBydW5fdmFsdWVzLmFwcGVuZChjdXJyZW50X2xlbmd0aCkKICAgIGVuY29kZWQgPSBtYXNrX3V0aWxzLmZyUHlPYmplY3RzKAogICAgICAgIHsic2l6ZSI6IFtoZWlnaHQsIHdpZHRoXSwgImNvdW50cyI6IHJ1bl92YWx1ZXN9LCBoZWlnaHQsIHdpZHRoCiAgICApCiAgICByZXR1cm4gewogICAgICAgICJzaXplIjogbGlzdChlbmNvZGVkWyJzaXplIl0pLAogICAgICAgICJjb3VudHMiOiBlbmNvZGVkWyJjb3VudHMiXS5kZWNvZGUoImFzY2lpIiksCiAgICB9CgoKZGVmIGRlY29kZV9ybGUocmxlOiBkaWN0W3N0ciwgQW55XSkgLT4gbnAubmRhcnJheToKICAgIGVuY29kZWQgPSBkaWN0KHJsZSkKICAgIGlmIGlzaW5zdGFuY2UoZW5jb2RlZFsiY291bnRzIl0sIHN0cik6CiAgICAgICAgZW5jb2RlZFsiY291bnRzIl0gPSBlbmNvZGVkWyJjb3VudHMiXS5lbmNvZGUoImFzY2lpIikKICAgIHJldHVybiBtYXNrX3V0aWxzLmRlY29kZShlbmNvZGVkKS5hc3R5cGUobnAudWludDgsIGNvcHk9RmFsc2UpCgoKZGVmIGVuY29kZV9sYWJlbF9jb21wb25lbnRzKAogICAgbGFiZWxzOiBucC5uZGFycmF5LAogICAgY29tcG9uZW50X2xhYmVsczogbGlzdFtpbnRdLAopIC0+IGRpY3RbaW50LCBkaWN0W3N0ciwgQW55XV06CiAgICAiIiJFbmNvZGUgc2VsZWN0ZWQgY29ubmVjdGVkIGNvbXBvbmVudHMgZnJvbSBvbmUgbGFiZWwgaW1hZ2UgaW4gYSBzaW5nbGUgc2Nhbi4iIiIKICAgIGhlaWdodCwgd2lkdGggPSBsYWJlbHMuc2hhcGUKICAgIGZsYXRfbGFiZWxzID0gbGFiZWxzLnJlc2hhcGUoLTEsIG9yZGVyPSJGIikKICAgIGZvcmVncm91bmRfcG9zaXRpb25zID0gbnAuZmxhdG5vbnplcm8oZmxhdF9sYWJlbHMpCiAgICBpZiBub3QgY29tcG9uZW50X2xhYmVscyBvciBmb3JlZ3JvdW5kX3Bvc2l0aW9ucy5zaXplID09IDA6CiAgICAgICAgcmV0dXJuIHt9CiAgICBmb3JlZ3JvdW5kX2xhYmVscyA9IGZsYXRfbGFiZWxzW2ZvcmVncm91bmRfcG9zaXRpb25zXQogICAgb3JkZXIgPSBucC5hcmdzb3J0KGZvcmVncm91bmRfbGFiZWxzLCBraW5kPSJzdGFibGUiKQogICAgc29ydGVkX3Bvc2l0aW9ucyA9IGZvcmVncm91bmRfcG9zaXRpb25zW29yZGVyXQogICAgc29ydGVkX2xhYmVscyA9IGZvcmVncm91bmRfbGFiZWxzW29yZGVyXQogICAgdW5pcXVlX2xhYmVscywgZ3JvdXBfc3RhcnRzID0gbnAudW5pcXVlKHNvcnRlZF9sYWJlbHMsIHJldHVybl9pbmRleD1UcnVlKQogICAgZ3JvdXBfZW5kcyA9IG5wLnJfW2dyb3VwX3N0YXJ0c1sxOl0sIGxlbihzb3J0ZWRfbGFiZWxzKV0KICAgIHJlc3VsdDogZGljdFtpbnQsIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICB0b3RhbF9waXhlbHMgPSBoZWlnaHQgKiB3aWR0aAoKICAgIGZvciBsYWJlbCBpbiBjb21wb25lbnRfbGFiZWxzOgogICAgICAgIGdyb3VwX2luZGV4ID0gaW50KG5wLnNlYXJjaHNvcnRlZCh1bmlxdWVfbGFiZWxzLCBsYWJlbCkpCiAgICAgICAgaWYgZ3JvdXBfaW5kZXggPj0gbGVuKHVuaXF1ZV9sYWJlbHMpIG9yIGludCh1bmlxdWVfbGFiZWxzW2dyb3VwX2luZGV4XSkgIT0gbGFiZWw6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcG9zaXRpb25zID0gc29ydGVkX3Bvc2l0aW9uc1tncm91cF9zdGFydHNbZ3JvdXBfaW5kZXhdIDogZ3JvdXBfZW5kc1tncm91cF9pbmRleF1dCiAgICAgICAgYm91bmRhcmllcyA9IG5wLmZsYXRub256ZXJvKG5wLmRpZmYocG9zaXRpb25zKSA+IDEpICsgMQogICAgICAgICMgVGhlIGV4Y2x1c2l2ZSBlbmRzIGFyZSBvbmUgcGFzdCB0aGUgbGFzdCBmb3JlZ3JvdW5kIHBpeGVsIGluIGVhY2ggcnVuLgogICAgICAgIHN0YXJ0c19hdCA9IG5wLnJfWzAsIGJvdW5kYXJpZXNdCiAgICAgICAgZW5kc19hdCA9IG5wLnJfW2JvdW5kYXJpZXMsIGxlbihwb3NpdGlvbnMpXQogICAgICAgIGNvdW50czogbGlzdFtpbnRdID0gW10KICAgICAgICBjdXJzb3IgPSAwCiAgICAgICAgZm9yIGJlZ2luLCBlbmQgaW4gemlwKHN0YXJ0c19hdCwgZW5kc19hdCk6CiAgICAgICAgICAgIGZvcmVncm91bmRfc3RhcnQgPSBpbnQocG9zaXRpb25zW2JlZ2luXSkKICAgICAgICAgICAgZm9yZWdyb3VuZF9lbmQgPSBpbnQocG9zaXRpb25zW2VuZCAtIDFdKSArIDEKICAgICAgICAgICAgY291bnRzLmFwcGVuZChmb3JlZ3JvdW5kX3N0YXJ0IC0gY3Vyc29yKQogICAgICAgICAgICBjb3VudHMuYXBwZW5kKGZvcmVncm91bmRfZW5kIC0gZm9yZWdyb3VuZF9zdGFydCkKICAgICAgICAgICAgY3Vyc29yID0gZm9yZWdyb3VuZF9lbmQKICAgICAgICB0YWlsID0gdG90YWxfcGl4ZWxzIC0gY3Vyc29yCiAgICAgICAgaWYgdGFpbDoKICAgICAgICAgICAgY291bnRzLmFwcGVuZCh0YWlsKQogICAgICAgIHVuY29tcHJlc3NlZCA9IHsic2l6ZSI6IFtoZWlnaHQsIHdpZHRoXSwgImNvdW50cyI6IGNvdW50c30KICAgICAgICBlbmNvZGVkID0gbWFza191dGlscy5mclB5T2JqZWN0cyh1bmNvbXByZXNzZWQsIGhlaWdodCwgd2lkdGgpCiAgICAgICAgaWYgaXNpbnN0YW5jZShlbmNvZGVkLCBsaXN0KToKICAgICAgICAgICAgZW5jb2RlZCA9IGVuY29kZWRbMF0KICAgICAgICByZXN1bHRbbGFiZWxdID0gX2NvY29fZW5jb2RlZF9ybGUoZW5jb2RlZCkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgX2NvY29fZW5jb2RlZF9ybGUoZW5jb2RlZDogZGljdFtzdHIsIEFueV0pIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgY291bnRzID0gZW5jb2RlZFsiY291bnRzIl0KICAgIGlmIGlzaW5zdGFuY2UoY291bnRzLCBieXRlcyk6CiAgICAgICAgY291bnRzID0gY291bnRzLmRlY29kZSgiYXNjaWkiKQogICAgcmV0dXJuIHsic2l6ZSI6IFtpbnQoZW5jb2RlZFsic2l6ZSJdWzBdKSwgaW50KGVuY29kZWRbInNpemUiXVsxXSldLCAiY291bnRzIjogY291bnRzfQoKCmRlZiBzb2xhcl9kaXNrX21hc2soaGVpZ2h0OiBpbnQsIHdpZHRoOiBpbnQsIHJhZGl1c19mcmFjdGlvbjogZmxvYXQpIC0+IG5wLm5kYXJyYXk6CiAgICByYWRpdXMgPSBpbnQocm91bmQocmFkaXVzX2ZyYWN0aW9uICogbWluKGhlaWdodCwgd2lkdGgpKSkKICAgIGNlbnRlciA9ICh3aWR0aCAvLyAyLCBoZWlnaHQgLy8gMikKICAgIGRpc2sgPSBucC56ZXJvcygoaGVpZ2h0LCB3aWR0aCksIGR0eXBlPW5wLnVpbnQ4KQogICAgY3YyLmNpcmNsZShkaXNrLCBjZW50ZXIsIHJhZGl1cywgY29sb3I9MSwgdGhpY2tuZXNzPS0xLCBsaW5lVHlwZT1jdjIuTElORV84KQogICAgcmV0dXJuIGRpc2suYXN0eXBlKGJvb2wpCgoKZGVmIHByZWRpY3RfaW1hZ2UoCiAgICBncmF5X2ltYWdlOiBucC5uZGFycmF5LAogICAgZmlsZV9uYW1lOiBzdHIsCiAgICBjb25maWc6IEJhc2VsaW5lQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV06CiAgICBjb25maWcgPSBjb25maWcgb3IgQmFzZWxpbmVDb25maWcoKQogICAgaWYgZ3JheV9pbWFnZS5uZGltICE9IDIgb3IgZ3JheV9pbWFnZS5kdHlwZSAhPSBucC51aW50ODoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCBhbiA4LWJpdCBncmF5c2NhbGUgaW1hZ2UiKQogICAgaGVpZ2h0LCB3aWR0aCA9IGdyYXlfaW1hZ2Uuc2hhcGUKICAgIGlmIGhlaWdodCAhPSBJTUFHRV9TSVpFIG9yIHdpZHRoICE9IElNQUdFX1NJWkU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGEge0lNQUdFX1NJWkV9eHtJTUFHRV9TSVpFfSBpbWFnZSwgZ290IHt3aWR0aH14e2hlaWdodH0iKQogICAgaWYgY29uZmlnLmJsdXJfa2VybmVsIDwgMSBvciBjb25maWcuYmx1cl9rZXJuZWwgJSAyID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYmx1cl9rZXJuZWwgbXVzdCBiZSBhIHBvc2l0aXZlIG9kZCBpbnRlZ2VyIikKICAgIGlmIGNvbmZpZy5jbG9zaW5nX2tlcm5lbCA8IDEgb3IgY29uZmlnLmNsb3Npbmdfa2VybmVsICUgMiA9PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNsb3Npbmdfa2VybmVsIG11c3QgYmUgYSBwb3NpdGl2ZSBvZGQgaW50ZWdlciIpCgogICAgYmx1cnJlZCA9IGN2Mi5HYXVzc2lhbkJsdXIoZ3JheV9pbWFnZSwgKGNvbmZpZy5ibHVyX2tlcm5lbCwgY29uZmlnLmJsdXJfa2VybmVsKSwgMCkKICAgIHJlc3BvbnNlID0gY3YyLnN1YnRyYWN0KGJsdXJyZWQsIGdyYXlfaW1hZ2UpCiAgICBkaXNrID0gc29sYXJfZGlza19tYXNrKGhlaWdodCwgd2lkdGgsIGNvbmZpZy5zb2xhcl9kaXNrX3JhZGl1c19mcmFjdGlvbikKICAgIGRpc2tfdmFsdWVzID0gcmVzcG9uc2VbZGlza10KICAgIGlmIGRpc2tfdmFsdWVzLnNpemUgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTb2xhciBkaXNrIHJlZ2lvbiBpcyBlbXB0eSIpCiAgICB0aHJlc2hvbGQsIF8gPSBjdjIudGhyZXNob2xkKAogICAgICAgIGRpc2tfdmFsdWVzLnJlc2hhcGUoLTEsIDEpLAogICAgICAgIDAsCiAgICAgICAgMjU1LAogICAgICAgIGN2Mi5USFJFU0hfQklOQVJZIHwgY3YyLlRIUkVTSF9PVFNVLAogICAgKQogICAgYmluYXJ5ID0gKChyZXNwb25zZSA+IHRocmVzaG9sZCkgJiBkaXNrKS5hc3R5cGUobnAudWludDgpCiAgICBrZXJuZWwgPSBucC5vbmVzKChjb25maWcuY2xvc2luZ19rZXJuZWwsIGNvbmZpZy5jbG9zaW5nX2tlcm5lbCksIGR0eXBlPW5wLnVpbnQ4KQogICAgY2xvc2VkID0gY3YyLm1vcnBob2xvZ3lFeChiaW5hcnksIGN2Mi5NT1JQSF9DTE9TRSwga2VybmVsKQogICAgY2xvc2VkW35kaXNrXSA9IDAKCiAgICBjb3VudCwgbGFiZWxzLCBzdGF0cywgXyA9IGN2Mi5jb25uZWN0ZWRDb21wb25lbnRzV2l0aFN0YXRzKAogICAgICAgIGNsb3NlZCwgY29ubmVjdGl2aXR5PWNvbmZpZy5jb21wb25lbnRfY29ubmVjdGl2aXR5CiAgICApCiAgICBhY2NlcHRlZF9sYWJlbHMgPSBbCiAgICAgICAgbGFiZWwKICAgICAgICBmb3IgbGFiZWwgaW4gcmFuZ2UoMSwgY291bnQpCiAgICAgICAgaWYgY29uZmlnLm1pbl9jb21wb25lbnRfYXJlYQogICAgICAgIDw9IGludChzdGF0c1tsYWJlbCwgY3YyLkNDX1NUQVRfQVJFQV0pCiAgICAgICAgPD0gY29uZmlnLm1heF9jb21wb25lbnRfYXJlYQogICAgXQogICAgZW5jb2RlZF9jb21wb25lbnRzID0gZW5jb2RlX2xhYmVsX2NvbXBvbmVudHMobGFiZWxzLCBhY2NlcHRlZF9sYWJlbHMpCiAgICBpbWFnZV9zdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIHByZWRpY3Rpb25zOiBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXSA9IFtdCiAgICBmb3Igc2VxdWVuY2UsIGxhYmVsIGluIGVudW1lcmF0ZShhY2NlcHRlZF9sYWJlbHMsIHN0YXJ0PTEpOgogICAgICAgIGFyZWEgPSBpbnQoc3RhdHNbbGFiZWwsIGN2Mi5DQ19TVEFUX0FSRUFdKQogICAgICAgIHByZWRpY3Rpb25zLmFwcGVuZCgKICAgICAgICAgICAgUHJlZGljdGVkSW5zdGFuY2UoCiAgICAgICAgICAgICAgICBmaWxhbWVudF9pZD1mIntpbWFnZV9zdGVtfV97c2VxdWVuY2V9IiwKICAgICAgICAgICAgICAgIGltYWdlX3N0ZW09aW1hZ2Vfc3RlbSwKICAgICAgICAgICAgICAgIGFyZWE9YXJlYSwKICAgICAgICAgICAgICAgIHJsZT1lbmNvZGVkX2NvbXBvbmVudHNbbGFiZWxdLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIHByZWRpY3Rpb25zCgoKZGVmIHJlYWRfZ3JheXNjYWxlKHBhdGg6IHN0ciB8IFBhdGgpIC0+IG5wLm5kYXJyYXk6CiAgICBpbWFnZSA9IGN2Mi5pbXJlYWQoc3RyKHBhdGgpLCBjdjIuSU1SRUFEX0dSQVlTQ0FMRSkKICAgIGlmIGltYWdlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDb3VsZCBub3QgcmVhZCBpbWFnZToge3BhdGh9IikKICAgIGlmIGltYWdlLnNoYXBlICE9IChJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgYSAyMDQ4eDIwNDggaW1hZ2UgYXQge3BhdGh9LCBnb3Qge2ltYWdlLnNoYXBlfSIpCiAgICByZXR1cm4gaW1hZ2UKCgpkZWYgd3JpdGVfcHJlZGljdGlvbl9vdmVybGF5KAogICAgZ3JheV9pbWFnZTogbnAubmRhcnJheSwKICAgIHByZWRpY3Rpb25zOiBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXSwKICAgIG91dHB1dF9wYXRoOiBzdHIgfCBQYXRoLAopIC0+IE5vbmU6CiAgICBjYW52YXMgPSBjdjIuY3Z0Q29sb3IoZ3JheV9pbWFnZSwgY3YyLkNPTE9SX0dSQVkyQkdSKQogICAgY29sb3JfbGF5ZXIgPSBjYW52YXMuY29weSgpCiAgICBmb3IgaW5kZXgsIHByZWRpY3Rpb24gaW4gZW51bWVyYXRlKHByZWRpY3Rpb25zKToKICAgICAgICBodWUgPSAoaW5kZXggKiAzNyArIDExKSAlIDE4MAogICAgICAgIGhzdiA9IG5wLnVpbnQ4KFtbW2h1ZSwgMjIwLCAyNTVdXV0pCiAgICAgICAgY29sb3IgPSB0dXBsZShpbnQodmFsdWUpIGZvciB2YWx1ZSBpbiBjdjIuY3Z0Q29sb3IoaHN2LCBjdjIuQ09MT1JfSFNWMkJHUilbMCwgMF0pCiAgICAgICAgbWFzayA9IGRlY29kZV9ybGUocHJlZGljdGlvbi5ybGUpCiAgICAgICAgY29sb3JfbGF5ZXJbbWFzayA+IDBdID0gY29sb3IKICAgIGNhbnZhcyA9IGN2Mi5hZGRXZWlnaHRlZChjYW52YXMsIDAuNjIsIGNvbG9yX2xheWVyLCAwLjM4LCAwKQogICAgY3YyLnB1dFRleHQoCiAgICAgICAgY2FudmFzLAogICAgICAgIGYiUHJlZGljdGVkIGluc3RhbmNlczoge2xlbihwcmVkaWN0aW9ucyl9IiwKICAgICAgICAoMzIsIDU2KSwKICAgICAgICBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsCiAgICAgICAgMS4wLAogICAgICAgICgyNTUsIDI1NSwgMjU1KSwKICAgICAgICAyLAogICAgICAgIGN2Mi5MSU5FX0FBLAogICAgKQogICAgb3V0cHV0X3BhdGggPSBQYXRoKG91dHB1dF9wYXRoKQogICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIG5vdCBjdjIuaW13cml0ZShzdHIob3V0cHV0X3BhdGgpLCBjYW52YXMpOgogICAgICAgIHJhaXNlIE9TRXJyb3IoZiJDb3VsZCBub3Qgd3JpdGUgcHJlZGljdGlvbiBvdmVybGF5OiB7b3V0cHV0X3BhdGh9IikK","src/filament_baseline/reinference.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gLmRhdGEgaW1wb3J0IGludmVudG9yeSwgbG9hZF9kYXRhc2V0CmZyb20gLmluZmVyZW5jZSBpbXBvcnQgSW5zdGFuY2VDb25maWcsIGluc3RhbmNlc19mcm9tX3Byb2JhYmlsaXRpZXMsIHByZWRpY3RfcHJvYmFiaWxpdGllcwpmcm9tIC5tb2RlbF9ydW5uZXIgaW1wb3J0IF9naXRfc3RhdHVzLCBfaXNfa2FnZ2xlX3J1bnRpbWUsIF9sb2FkX21vZGVsLCBfcmVzb2x2ZV9kYXRhX3Jvb3QsIF93aXRoaW4KZnJvbSAucHJlZGljdG9yIGltcG9ydCBQcmVkaWN0ZWRJbnN0YW5jZSwgcmVhZF9ncmF5c2NhbGUKZnJvbSAucnVubmVyIGltcG9ydCBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzLCBfd3JpdGVfb3ZlcmxheXMsIF93cml0ZV9wcmVkaWN0aW9ucwpmcm9tIC5ydW50aW1lIGltcG9ydCBoYXJkd2FyZV9pbmZvLCBwYWNrYWdlX3ZlcnNpb25zCmZyb20gLnNjb3JlciBpbXBvcnQgc2NvcmVfdmFsaWRhdGlvbgpmcm9tIC5zcGxpdCBpbXBvcnQgY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0CmZyb20gLnZpc3VhbHMgaW1wb3J0IHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMKCgpkZWYgcmVpbmZlcl92YWxpZGF0aW9uKAogICAgKiwKICAgIHJlcG9fcm9vdDogc3RyIHwgUGF0aCwKICAgIGRhdGFfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUsCiAgICBtb2RlbF9ydW5fcm9vdDogc3RyIHwgUGF0aCwKICAgIHR0YTogaW50LAogICAgb3ZlcmxheV9jb3VudDogaW50ID0gMTIsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJSZWNvbXB1dGUgaG9sZG91dCBwcm9iYWJpbGl0eSBtYXBzIGZyb20gYSBzYXZlZCBjaGVja3BvaW50IHdpdGggYW5vdGhlciBUVEEgY291bnQuIiIiCiAgICByZXBvX3Jvb3QgPSBQYXRoKHJlcG9fcm9vdCkucmVzb2x2ZSgpCiAgICBtb2RlbF9ydW5fcm9vdCA9IFBhdGgobW9kZWxfcnVuX3Jvb3QpLnJlc29sdmUoKQogICAga2FnZ2xlX3J1bnRpbWUgPSBfaXNfa2FnZ2xlX3J1bnRpbWUoKQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihtb2RlbF9ydW5fcm9vdCwgcmVwb19yb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgbW9kZWwgYXJ0aWZhY3RzIG11c3Qgc3RheSBpbnNpZGUgdGhlIHJlcG9zaXRvcnk6IHttb2RlbF9ydW5fcm9vdH0iKQogICAgaWYgdHRhIG5vdCBpbiAoMSwgMiwgNCwgOCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHRhIG11c3QgYmUgMSwgMiwgNCwgb3IgOCIpCgogICAgZGF0YXNldF9yb290ID0gX3Jlc29sdmVfZGF0YV9yb290KGRhdGFfcm9vdCwgcmVwb19yb290KQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihkYXRhc2V0X3Jvb3QsIHJlcG9fcm9vdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkxvY2FsIGNvbXBldGl0aW9uIGRhdGEgbXVzdCBzdGF5IGluc2lkZSB0aGUgcmVwb3NpdG9yeToge2RhdGFzZXRfcm9vdH0iKQogICAgbW9kZWxfbWV0cmljc19wYXRoID0gbW9kZWxfcnVuX3Jvb3QgLyAibWV0cmljcy5qc29uIgogICAgY2hlY2twb2ludF9wYXRoID0gbW9kZWxfcnVuX3Jvb3QgLyAiYmVzdF9tb2RlbC5wdCIKICAgIGlmIG5vdCBtb2RlbF9tZXRyaWNzX3BhdGguaXNfZmlsZSgpIG9yIG5vdCBjaGVja3BvaW50X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKCJtb2RlbF9ydW5fcm9vdCBtdXN0IGNvbnRhaW4gbWV0cmljcy5qc29uIGFuZCBiZXN0X21vZGVsLnB0IikKICAgIG1vZGVsX21ldHJpY3MgPSBqc29uLmxvYWRzKG1vZGVsX21ldHJpY3NfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBtb2RlbF9jb25maWd1cmF0aW9uID0gbW9kZWxfbWV0cmljcy5nZXQoIm1ldGhvZCIsIHt9KQogICAgcGF0Y2hfc2l6ZSA9IGludChtb2RlbF9jb25maWd1cmF0aW9uWyJwYXRjaF9zaXplIl0pCiAgICBkYXRhc2V0ID0gbG9hZF9kYXRhc2V0KGRhdGFzZXRfcm9vdCkKICAgIGRhdGFfaW52ZW50b3J5ID0gaW52ZW50b3J5KGRhdGFzZXQpCiAgICBjb25maWdfcGF0aCA9IHJlcG9fcm9vdCAvICJjb25maWdzIiAvICJ2YWxpZGF0aW9uLWJhc2VsaW5lLmpzb24iCiAgICBjb25maWcgPSBqc29uLmxvYWRzKGNvbmZpZ19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHNwbGl0ID0gY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgIHZhbGlkYXRpb25fZnJhY3Rpb249ZmxvYXQoY29uZmlnWyJ2YWxpZGF0aW9uX2ZyYWN0aW9uIl0pLAogICAgKQogICAgaWYgc3BsaXQuc2hhMjU2ICE9IGNvbmZpZ1siZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il06CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgZnJvemVuIHZhbGlkYXRpb24gc3BsaXQgZGlnZXN0IGNoYW5nZWQiKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIGlmIGxlbih2YWxpZGF0aW9uX25hbWVzKSAhPSAxNDI6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiRXhwZWN0ZWQgYWxsIDE0MiBob2xkb3V0IG9ic2VydmF0aW9ucywgZm91bmQge2xlbih2YWxpZGF0aW9uX25hbWVzKX0iKQoKICAgIG91dHB1dF9yb290ID0gbW9kZWxfcnVuX3Jvb3QgLyBmInR0YS17dHRhfSIKICAgIGlmIG91dHB1dF9yb290LmV4aXN0cygpIGFuZCBhbnkob3V0cHV0X3Jvb3QuaXRlcmRpcigpKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZiJUVEEgb3V0cHV0IGFscmVhZHkgZXhpc3RzOyBjaG9vc2Ugb3IgY2xlYXIgYSBuZXcgcnVuIGRpcmVjdG9yeToge291dHB1dF9yb290fSIpCiAgICBwcm9iYWJpbGl0eV9kaXIgPSBvdXRwdXRfcm9vdCAvICJwcm9iYWJpbGl0aWVzIgogICAgcHJvYmFiaWxpdHlfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYTowIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBtb2RlbCA9IF9sb2FkX21vZGVsKGNoZWNrcG9pbnRfcGF0aCwgZGV2aWNlKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoZGV2aWNlKQogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXV0gPSB7fQogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGluZmVyZW5jZV9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIGluZGV4LCBmaWxlX25hbWUgaW4gZW51bWVyYXRlKHZhbGlkYXRpb25fbmFtZXMsIHN0YXJ0PTEpOgogICAgICAgIGltYWdlID0gcmVhZF9ncmF5c2NhbGUoZGF0YXNldC50cmFpbl9pbWFnZV9kaXIgLyBmaWxlX25hbWUpCiAgICAgICAgZm9yZWdyb3VuZCwgYm91bmRhcnkgPSBwcmVkaWN0X3Byb2JhYmlsaXRpZXMoCiAgICAgICAgICAgIG1vZGVsLAogICAgICAgICAgICBpbWFnZSwKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgcGF0Y2hfc2l6ZT1wYXRjaF9zaXplLAogICAgICAgICAgICBzdHJpZGU9bWF4KHBhdGNoX3NpemUgLy8gMiwgaW50KHBhdGNoX3NpemUgKiAwLjc1KSksCiAgICAgICAgICAgIHR0YT10dGEsCiAgICAgICAgKQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgICAgIHByb2JhYmlsaXR5X2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiLAogICAgICAgICAgICBmb3JlZ3JvdW5kPWZvcmVncm91bmQuYXN0eXBlKG5wLmZsb2F0MTYpLAogICAgICAgICAgICBib3VuZGFyeT1ib3VuZGFyeS5hc3R5cGUobnAuZmxvYXQxNiksCiAgICAgICAgKQogICAgICAgIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0gPSBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICBmb3JlZ3JvdW5kLAogICAgICAgICAgICBib3VuZGFyeSwKICAgICAgICAgICAgZmlsZV9uYW1lLAogICAgICAgICAgICBJbnN0YW5jZUNvbmZpZygpLAogICAgICAgICkKICAgICAgICBpZiBpbmRleCAlIDIwID09IDAgb3IgaW5kZXggPT0gbGVuKHZhbGlkYXRpb25fbmFtZXMpOgogICAgICAgICAgICBwcmludChmIlJlLWluZmVycmVkIHtpbmRleH0ve2xlbih2YWxpZGF0aW9uX25hbWVzKX0gb2JzZXJ2YXRpb25zIHdpdGggVFRBPXt0dGF9IiwgZmx1c2g9VHJ1ZSkKICAgIGluZmVyZW5jZV9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyZW5jZV9zdGFydGVkCiAgICBwZWFrX2dwdV9tZW1vcnlfYnl0ZXMgPSAoCiAgICAgICAgaW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoZGV2aWNlKSkgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGVsc2UgMAogICAgKQoKICAgIHNjb3Jpbmdfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIG1ldHJpY3MsIG9ic2VydmF0aW9uX3Jvd3MgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcHJlZGljdGlvbnMsCiAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICBpb3VfdGhyZXNob2xkPWZsb2F0KGNvbmZpZ1siaW91X3RocmVzaG9sZCJdKSwKICAgICAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkPWZsb2F0KGNvbmZpZ1siZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCJdKSwKICAgICkKICAgIHNjb3Jpbmdfc2Vjb25kcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzY29yaW5nX3N0YXJ0ZWQKICAgIF93cml0ZV9wcmVkaWN0aW9ucyhvdXRwdXRfcm9vdCAvICJwcmVkaWN0aW9ucy5jc3YiLCBwcmVkaWN0aW9ucykKICAgIF93cml0ZV9vYnNlcnZhdGlvbl9zY29yZXMob3V0cHV0X3Jvb3QgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIG9ic2VydmF0aW9uX3Jvd3MsIHByZWRpY3Rpb25zKQogICAgX3dyaXRlX292ZXJsYXlzKGRhdGFzZXQsIHByZWRpY3Rpb25zLCBvdXRwdXRfcm9vdCAvICJvdmVybGF5cyIsIG92ZXJsYXlfY291bnQpCiAgICBjb21wYXJpc29uX292ZXJsYXlzID0gd3JpdGVfY29tcGFyaXNvbl9vdmVybGF5cygKICAgICAgICBkYXRhc2V0LCBwcmVkaWN0aW9ucywgb3V0cHV0X3Jvb3QgLyAiY29tcGFyaXNvbi1vdmVybGF5cyIsIG92ZXJsYXlfY291bnQKICAgICkKCiAgICByZXBvc2l0b3J5X2luZm8gPSBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3QpCiAgICByZXN1bHQgPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIsCiAgICAgICAgInJ1bl9uYW1lIjogZiJ7bW9kZWxfcnVuX3Jvb3QubmFtZX0tdHRhLXt0dGF9IiwKICAgICAgICAicGFyZW50X21vZGVsX3J1biI6IHN0cihtb2RlbF9ydW5fcm9vdC5yZWxhdGl2ZV90byhyZXBvX3Jvb3QpKQogICAgICAgIGlmIF93aXRoaW4obW9kZWxfcnVuX3Jvb3QsIHJlcG9fcm9vdCkKICAgICAgICBlbHNlIHN0cihtb2RlbF9ydW5fcm9vdCksCiAgICAgICAgInBhcmVudF9tb2RlbF9jb25maWd1cmF0aW9uIjogbW9kZWxfY29uZmlndXJhdGlvbiwKICAgICAgICAiY2hlY2twb2ludF9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihjaGVja3BvaW50X3BhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAidHRhIjogdHRhLAogICAgICAgICJwYXRjaF9zaXplIjogcGF0Y2hfc2l6ZSwKICAgICAgICAiZGVmYXVsdF9pbnN0YW5jZV9jb25maWd1cmF0aW9uIjogSW5zdGFuY2VDb25maWcoKS5fX2RpY3RfXywKICAgICAgICAicmVwb3NpdG9yeSI6IHJlcG9zaXRvcnlfaW5mbywKICAgICAgICAiY29uZmlndXJhdGlvbl9zaGEyNTYiOiBoYXNobGliLnNoYTI1Nihjb25maWdfcGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICJkYXRhc2V0X2ludmVudG9yeSI6IGRhdGFfaW52ZW50b3J5LAogICAgICAgICJzcGxpdCI6IHsKICAgICAgICAgICAgInNoYTI1NiI6IHNwbGl0LnNoYTI1NiwKICAgICAgICAgICAgInRyYWluaW5nX29ic2VydmF0aW9ucyI6IGxlbihzcGxpdC50cmFpbmluZ19uYW1lcyksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGxlbih2YWxpZGF0aW9uX25hbWVzKSwKICAgICAgICB9LAogICAgICAgICJtZXRyaWNzIjogbWV0cmljcywKICAgICAgICAicnVudGltZSI6IHsKICAgICAgICAgICAgKipoYXJkd2FyZV9pbmZvKCksCiAgICAgICAgICAgICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpLAogICAgICAgICAgICAiZGV2aWNlIjogc3RyKGRldmljZSksCiAgICAgICAgICAgICJpbmZlcmVuY2Vfc2Vjb25kcyI6IGluZmVyZW5jZV9zZWNvbmRzLAogICAgICAgICAgICAic2NvcmluZ19zZWNvbmRzIjogc2NvcmluZ19zZWNvbmRzLAogICAgICAgICAgICAiZW5kX3RvX2VuZF9zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQsCiAgICAgICAgICAgICJwZWFrX2dwdV9tZW1vcnlfYnl0ZXMiOiBwZWFrX2dwdV9tZW1vcnlfYnl0ZXMsCiAgICAgICAgICAgICJjb21wYXJpc29uX292ZXJsYXlzIjogY29tcGFyaXNvbl9vdmVybGF5cywKICAgICAgICB9LAogICAgfQogICAgKG91dHB1dF9yb290IC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKHsicnVuX25hbWUiOiByZXN1bHRbInJ1bl9uYW1lIl0sICJtZXRyaWNzIjogbWV0cmljc1siZ2xvYmFsIl0sICJzcGxpdF9zaGEyNTYiOiBzcGxpdC5zaGEyNTZ9LCBpbmRlbnQ9MikpCiAgICByZXR1cm4gcmVzdWx0Cg==","src/filament_baseline/runner.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSAuZGF0YSBpbXBvcnQgQ29tcGV0aXRpb25EYXRhc2V0LCBmaW5kX2RhdGFzZXRfcm9vdCwgaW52ZW50b3J5LCBsb2FkX2RhdGFzZXQKZnJvbSAucHJlZGljdG9yIGltcG9ydCBCYXNlbGluZUNvbmZpZywgUHJlZGljdGVkSW5zdGFuY2UsIHByZWRpY3RfaW1hZ2UsIHJlYWRfZ3JheXNjYWxlLCB3cml0ZV9wcmVkaWN0aW9uX292ZXJsYXkKZnJvbSAucnVudGltZSBpbXBvcnQgZ2l0X2luZm8sIGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAuc2NvcmVyIGltcG9ydCBPYnNlcnZhdGlvbkNvdW50cywgc2NvcmVfdmFsaWRhdGlvbgpmcm9tIC5zcGxpdCBpbXBvcnQgY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0CgpPRkZJQ0lBTF9TQ09SRVJfVVJMID0gKAogICAgImh0dHBzOi8vd3d3LmthZ2dsZS5jb20vY29kZS9hemltYWhtYWR6YWRlaC9zZWxmLWV2YWx1YXRpb24tbm90ZWJvb2siCikKCgpkZWYgX3dpdGhpbihwYXRoOiBQYXRoLCBwYXJlbnQ6IFBhdGgpIC0+IGJvb2w6CiAgICB0cnk6CiAgICAgICAgcGF0aC5yZXNvbHZlKCkucmVsYXRpdmVfdG8ocGFyZW50LnJlc29sdmUoKSkKICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIF9pc19rYWdnbGVfcnVudGltZSgpIC0+IGJvb2w6CiAgICByZXR1cm4gYm9vbChvcy5lbnZpcm9uLmdldCgiS0FHR0xFX0tFUk5FTF9SVU5fVFlQRSIpKSBvciAoCiAgICAgICAgUGF0aCgiL2thZ2dsZS9pbnB1dCIpLmlzX2RpcigpIGFuZCBQYXRoKCIva2FnZ2xlL3dvcmtpbmciKS5pc19kaXIoKQogICAgKQoKCmRlZiBfc291cmNlX2ZpbmdlcnByaW50KHJlcG9fcm9vdDogUGF0aCkgLT4gc3RyOgogICAgcGF0aHMgPSBbCiAgICAgICAgKnNvcnRlZCgocmVwb19yb290IC8gInNyYyIpLnJnbG9iKCIqLnB5IikpLAogICAgICAgICpzb3J0ZWQoKHJlcG9fcm9vdCAvICJzY3JpcHRzIikuZ2xvYigiKi5weSIpKSwKICAgICAgICByZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIiwKICAgICAgICByZXBvX3Jvb3QgLyAibm90ZWJvb2tzIiAvICJ2YWxpZGF0aW9uX2Jhc2VsaW5lLmlweW5iIiwKICAgICAgICByZXBvX3Jvb3QgLyAicmVxdWlyZW1lbnRzLnR4dCIsCiAgICAgICAgcmVwb19yb290IC8gInJlcXVpcmVtZW50cy1ncHUudHh0IiwKICAgIF0KICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIGZvciBwYXRoIGluIHBhdGhzOgogICAgICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBkaWdlc3QudXBkYXRlKHBhdGgucmVsYXRpdmVfdG8ocmVwb19yb290KS5hc19wb3NpeCgpLmVuY29kZSgidXRmLTgiKSkKICAgICAgICBkaWdlc3QudXBkYXRlKGIiXDAiKQogICAgICAgIGRpZ2VzdC51cGRhdGUocGF0aC5yZWFkX2J5dGVzKCkpCiAgICAgICAgZGlnZXN0LnVwZGF0ZShiIlwwIikKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgX2xvYWRfY29uZmlnKHJlcG9fcm9vdDogUGF0aCkgLT4gdHVwbGVbZGljdFtzdHIsIEFueV0sIEJhc2VsaW5lQ29uZmlnXToKICAgIHBhdGggPSByZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIgogICAgcGF5bG9hZCA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBiYXNlbGluZSA9IEJhc2VsaW5lQ29uZmlnKAogICAgICAgIGJsdXJfa2VybmVsPWludChwYXlsb2FkWyJibHVyX2tlcm5lbCJdKSwKICAgICAgICBzb2xhcl9kaXNrX3JhZGl1c19mcmFjdGlvbj1mbG9hdChwYXlsb2FkWyJzb2xhcl9kaXNrX3JhZGl1c19mcmFjdGlvbiJdKSwKICAgICAgICBjbG9zaW5nX2tlcm5lbD1pbnQocGF5bG9hZFsiY2xvc2luZ19rZXJuZWwiXSksCiAgICAgICAgY29tcG9uZW50X2Nvbm5lY3Rpdml0eT1pbnQocGF5bG9hZFsiY29tcG9uZW50X2Nvbm5lY3Rpdml0eSJdKSwKICAgICAgICBtaW5fY29tcG9uZW50X2FyZWE9aW50KHBheWxvYWRbIm1pbl9jb21wb25lbnRfYXJlYSJdKSwKICAgICAgICBtYXhfY29tcG9uZW50X2FyZWE9aW50KHBheWxvYWRbIm1heF9jb21wb25lbnRfYXJlYSJdKSwKICAgICkKICAgIHJldHVybiBwYXlsb2FkLCBiYXNlbGluZQoKCmRlZiBfcmVzb2x2ZV9kYXRhX3Jvb3QoZGF0YV9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSwgcmVwb19yb290OiBQYXRoKSAtPiBQYXRoOgogICAgY2FuZGlkYXRlczogbGlzdFtQYXRoXSA9IFtdCiAgICBpZiBkYXRhX3Jvb3QgaXMgbm90IE5vbmU6CiAgICAgICAgY2FuZGlkYXRlcy5hcHBlbmQoUGF0aChkYXRhX3Jvb3QpKQogICAgZW52X3Jvb3QgPSBvcy5lbnZpcm9uLmdldCgiRVFVSU5FWFVTX0RBVEFfUk9PVCIpCiAgICBpZiBlbnZfcm9vdDoKICAgICAgICBjYW5kaWRhdGVzLmFwcGVuZChQYXRoKGVudl9yb290KSkKICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKAogICAgICAgIFsKICAgICAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInJhdyIgLyAiTUFHRmlMT18xLjBfS2FnZ2xlXzIwMjYiLAogICAgICAgICAgICBQYXRoKCIva2FnZ2xlL2lucHV0IiksCiAgICAgICAgXQogICAgKQogICAgZXJyb3JzID0gW10KICAgIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlkYXRlczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBmaW5kX2RhdGFzZXRfcm9vdChjYW5kaWRhdGUpCiAgICAgICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvciwgUnVudGltZUVycm9yKSBhcyBleGM6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7Y2FuZGlkYXRlfToge2V4Y30iKQogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgIk5vIGNvbXBldGl0aW9uIGRhdGFzZXQgd2FzIGZvdW5kLiBTZXQgRVFVSU5FWFVTX0RBVEFfUk9PVCBvciBwYXNzIC0tZGF0YS1yb290LlxuIgogICAgICAgICsgIlxuIi5qb2luKGVycm9ycykKICAgICkKCgpkZWYgX3dyaXRlX3ByZWRpY3Rpb25zKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RbUHJlZGljdGVkSW5zdGFuY2VdXSwKKSAtPiBOb25lOgogICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3V0cHV0X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICB3cml0ZXIgPSBjc3Yud3JpdGVyKHN0cmVhbSwgbGluZXRlcm1pbmF0b3I9IlxuIikKICAgICAgICB3cml0ZXIud3JpdGVyb3coKCJmaWxhbWVudF9pZCIsICJzZWdtZW50YXRpb25fcmxlIikpCiAgICAgICAgZm9yIGZpbGVfbmFtZSBpbiBzb3J0ZWQocHJlZGljdGlvbnMpOgogICAgICAgICAgICBmb3IgcHJlZGljdGlvbiBpbiBwcmVkaWN0aW9uc1tmaWxlX25hbWVdOgogICAgICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KChwcmVkaWN0aW9uLmZpbGFtZW50X2lkLCBwcmVkaWN0aW9uLnJsZVsiY291bnRzIl0pKQoKCmRlZiBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICByb3dzOiBsaXN0W09ic2VydmF0aW9uQ291bnRzXSwKICAgIHByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV1dLAopIC0+IE5vbmU6CiAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZmllbGRzID0gKAogICAgICAgICJmaWxlX25hbWUiLAogICAgICAgICJ5ZWFyIiwKICAgICAgICAic2l0ZSIsCiAgICAgICAgImFubm90YXRvcl92aWV3cyIsCiAgICAgICAgImdyb3VuZF90cnV0aF9pbnN0YW5jZXNfYWNyb3NzX3ZpZXdzIiwKICAgICAgICAicHJlZGljdGVkX2luc3RhbmNlc19wZXJfb2JzZXJ2YXRpb24iLAogICAgICAgICJ0cF9lZGdlcyIsCiAgICAgICAgInRwX2lvdV9zdW0iLAogICAgICAgICJmcCIsCiAgICAgICAgImZuIiwKICAgICAgICAicHEiLAogICAgKQogICAgd2l0aCBvdXRwdXRfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1maWVsZHMsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAiZmlsZV9uYW1lIjogcm93LmZpbGVfbmFtZSwKICAgICAgICAgICAgICAgICAgICAieWVhciI6IHJvdy55ZWFyLAogICAgICAgICAgICAgICAgICAgICJzaXRlIjogcm93LnNpdGUsCiAgICAgICAgICAgICAgICAgICAgImFubm90YXRvcl92aWV3cyI6IHJvdy5hbm5vdGF0b3Jfdmlld3MsCiAgICAgICAgICAgICAgICAgICAgImdyb3VuZF90cnV0aF9pbnN0YW5jZXNfYWNyb3NzX3ZpZXdzIjogcm93Lmd0X2NvdW50LAogICAgICAgICAgICAgICAgICAgICJwcmVkaWN0ZWRfaW5zdGFuY2VzX3Blcl9vYnNlcnZhdGlvbiI6IGxlbigKICAgICAgICAgICAgICAgICAgICAgICAgcHJlZGljdGlvbnMuZ2V0KHJvdy5maWxlX25hbWUsIFtdKQogICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAgICAgInRwX2VkZ2VzIjogcm93LnRwLAogICAgICAgICAgICAgICAgICAgICJ0cF9pb3Vfc3VtIjogZiJ7cm93LnRwX2lvdV9zdW06LjlmfSIsCiAgICAgICAgICAgICAgICAgICAgImZwIjogcm93LmZwLAogICAgICAgICAgICAgICAgICAgICJmbiI6IHJvdy5mbiwKICAgICAgICAgICAgICAgICAgICAicHEiOiBmIntyb3cucHE6LjlmfSIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKCgpkZWYgX3dyaXRlX292ZXJsYXlzKAogICAgZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0LAogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXV0sCiAgICBvdXRwdXRfZGlyOiBQYXRoLAogICAgbWF4aW11bTogaW50LAopIC0+IGxpc3Rbc3RyXToKICAgIG5hbWVzID0gc29ydGVkKHByZWRpY3Rpb25zKQogICAgaWYgbm90IG5hbWVzIG9yIG1heGltdW0gPD0gMDoKICAgICAgICByZXR1cm4gW10KICAgIGlmIGxlbihuYW1lcykgPD0gbWF4aW11bToKICAgICAgICBzZWxlY3RlZCA9IG5hbWVzCiAgICBlbGlmIG1heGltdW0gPT0gMToKICAgICAgICBzZWxlY3RlZCA9IFtuYW1lc1tsZW4obmFtZXMpIC8vIDJdXQogICAgZWxzZToKICAgICAgICBpbmRpY2VzID0ge3JvdW5kKGluZGV4ICogKGxlbihuYW1lcykgLSAxKSAvIChtYXhpbXVtIC0gMSkpIGZvciBpbmRleCBpbiByYW5nZShtYXhpbXVtKX0KICAgICAgICBzZWxlY3RlZCA9IFtuYW1lc1tpbmRleF0gZm9yIGluZGV4IGluIHNvcnRlZChpbmRpY2VzKV0KICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZm9yIGZpbGVfbmFtZSBpbiBzZWxlY3RlZDoKICAgICAgICBncmF5ID0gcmVhZF9ncmF5c2NhbGUoZGF0YXNldC50cmFpbl9pbWFnZV9kaXIgLyBmaWxlX25hbWUpCiAgICAgICAgb3V0cHV0X3BhdGggPSBvdXRwdXRfZGlyIC8gZiJ7UGF0aChmaWxlX25hbWUpLnN0ZW19X3ByZWRpY3Rpb24ucG5nIgogICAgICAgIHdyaXRlX3ByZWRpY3Rpb25fb3ZlcmxheShncmF5LCBwcmVkaWN0aW9uc1tmaWxlX25hbWVdLCBvdXRwdXRfcGF0aCkKICAgIHJldHVybiBzZWxlY3RlZAoKCmRlZiBfZ2l0X3N0YXR1cyhyZXBvX3Jvb3Q6IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgaW5mbyA9IGdpdF9pbmZvKHJlcG9fcm9vdCkKICAgIGluZm9bInNvdXJjZV90cmVlX3NoYTI1NiJdID0gX3NvdXJjZV9maW5nZXJwcmludChyZXBvX3Jvb3QpCiAgICByZXR1cm4gaW5mbwoKCmRlZiBydW5fdmFsaWRhdGlvbigKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKICAgIG91dHB1dF9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICBsaW1pdF9vYnNlcnZhdGlvbnM6IGludCB8IE5vbmUgPSBOb25lLAogICAgb3ZlcmxheV9jb3VudDogaW50ID0gMTIsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBydW5fc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHJlcG9fcm9vdCA9IFBhdGgocmVwb19yb290KS5yZXNvbHZlKCkKICAgIGthZ2dsZV9ydW50aW1lID0gX2lzX2thZ2dsZV9ydW50aW1lKCkKICAgIGRhdGFzZXRfcm9vdCA9IF9yZXNvbHZlX2RhdGFfcm9vdChkYXRhX3Jvb3QsIHJlcG9fcm9vdCkKICAgIGlmIG5vdCBrYWdnbGVfcnVudGltZSBhbmQgbm90IF93aXRoaW4oZGF0YXNldF9yb290LCByZXBvX3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiTG9jYWwgY29tcGV0aXRpb24gZGF0YSBtdXN0IHN0YXkgaW5zaWRlIHRoZSByZXBvc2l0b3J5OiB7ZGF0YXNldF9yb290fSIKICAgICAgICApCgogICAgaWYgb3V0cHV0X3Jvb3QgaXMgTm9uZToKICAgICAgICBvdXRwdXRfcm9vdCA9ICgKICAgICAgICAgICAgUGF0aCgiL2thZ2dsZS93b3JraW5nL2ZpbGFtZW50LXZhbGlkYXRpb24tYmFzZWxpbmUiKQogICAgICAgICAgICBpZiBrYWdnbGVfcnVudGltZQogICAgICAgICAgICBlbHNlIHJlcG9fcm9vdCAvICJhcnRpZmFjdHMiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUiCiAgICAgICAgKQogICAgb3V0cHV0X3Jvb3QgPSBQYXRoKG91dHB1dF9yb290KQogICAgaWYgbm90IG91dHB1dF9yb290LmlzX2Fic29sdXRlKCk6CiAgICAgICAgb3V0cHV0X3Jvb3QgPSByZXBvX3Jvb3QgLyBvdXRwdXRfcm9vdAogICAgb3V0cHV0X3Jvb3QgPSBvdXRwdXRfcm9vdC5yZXNvbHZlKCkKICAgIGlmIG5vdCBrYWdnbGVfcnVudGltZSBhbmQgbm90IF93aXRoaW4ob3V0cHV0X3Jvb3QsIHJlcG9fcm9vdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkxvY2FsIHJ1biBvdXRwdXQgbXVzdCBzdGF5IGluc2lkZSB0aGUgcmVwb3NpdG9yeToge291dHB1dF9yb290fSIpCiAgICBvdXRwdXRfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgY29uZmlnX2pzb24sIGJhc2VsaW5lX2NvbmZpZyA9IF9sb2FkX2NvbmZpZyhyZXBvX3Jvb3QpCiAgICBkYXRhc2V0ID0gbG9hZF9kYXRhc2V0KGRhdGFzZXRfcm9vdCkKICAgIGlucHV0X2ludmVudG9yeSA9IGludmVudG9yeShkYXRhc2V0KQogICAgaWYgaW5wdXRfaW52ZW50b3J5WyJvYnNlcnZhdGlvbl9jb3VudCJdICE9IDcwNzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAiRXhwZWN0ZWQgNzA3IGNvbXBldGl0aW9uIHRyYWluaW5nIG9ic2VydmF0aW9ucywgZm91bmQgIgogICAgICAgICAgICBmIntpbnB1dF9pbnZlbnRvcnlbJ29ic2VydmF0aW9uX2NvdW50J119IgogICAgICAgICkKICAgIGlmIGlucHV0X2ludmVudG9yeVsiYW5ub3RhdG9yX3ZpZXdfY291bnQiXSAhPSAxMTU0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJFeHBlY3RlZCAxLDE1NCBhbm5vdGF0b3Igdmlld3MsIGZvdW5kICIKICAgICAgICAgICAgZiJ7aW5wdXRfaW52ZW50b3J5Wydhbm5vdGF0b3Jfdmlld19jb3VudCddfSIKICAgICAgICApCiAgICBpZiBpbnB1dF9pbnZlbnRvcnlbImFubm90YXRpb25fY291bnQiXSAhPSA4MTk5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiRXhwZWN0ZWQgOCwxOTkgdHJhaW5pbmcgYW5ub3RhdGlvbnMsIGZvdW5kIHtpbnB1dF9pbnZlbnRvcnlbJ2Fubm90YXRpb25fY291bnQnXX0iCiAgICAgICAgKQoKICAgIHNwbGl0ID0gY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnX2pzb25bInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdfanNvblsidmFsaWRhdGlvbl9mcmFjdGlvbiJdKSwKICAgICkKICAgIGV4cGVjdGVkX3NwbGl0X3NoYTI1NiA9IGNvbmZpZ19qc29uLmdldCgiZXhwZWN0ZWRfc3BsaXRfc2hhMjU2IikKICAgIGlmIGV4cGVjdGVkX3NwbGl0X3NoYTI1NiBhbmQgc3BsaXQuc2hhMjU2ICE9IGV4cGVjdGVkX3NwbGl0X3NoYTI1NjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICJPYnNlcnZhdGlvbiBzcGxpdCBkaWdlc3QgZGlmZmVycyBmcm9tIHRoZSBmcm96ZW4gYmFzZWxpbmU6ICIKICAgICAgICAgICAgZiJleHBlY3RlZCB7ZXhwZWN0ZWRfc3BsaXRfc2hhMjU2fSwgZ290IHtzcGxpdC5zaGEyNTZ9IgogICAgICAgICkKICAgIHZhbGlkYXRpb25fbmFtZXMgPSBsaXN0KHNwbGl0LnZhbGlkYXRpb25fbmFtZXMpCiAgICBmdWxsX2hvbGRvdXRfY291bnQgPSBsZW4odmFsaWRhdGlvbl9uYW1lcykKICAgIGlmIGxpbWl0X29ic2VydmF0aW9ucyBpcyBub3QgTm9uZToKICAgICAgICBpZiBsaW1pdF9vYnNlcnZhdGlvbnMgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsaW1pdF9vYnNlcnZhdGlvbnMgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgdmFsaWRhdGlvbl9uYW1lcyA9IHZhbGlkYXRpb25fbmFtZXNbOmxpbWl0X29ic2VydmF0aW9uc10KCiAgICBoYXJkd2FyZSA9IGhhcmR3YXJlX2luZm8oKQogICAgcHJlZGljdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W1ByZWRpY3RlZEluc3RhbmNlXV0gPSB7fQogICAgcHJlZGljdGlvbl9zdGFydCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGZvciBpbmRleCwgZmlsZV9uYW1lIGluIGVudW1lcmF0ZSh2YWxpZGF0aW9uX25hbWVzLCBzdGFydD0xKToKICAgICAgICBpbWFnZSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0gPSBwcmVkaWN0X2ltYWdlKGltYWdlLCBmaWxlX25hbWUsIGJhc2VsaW5lX2NvbmZpZykKICAgICAgICBpZiBpbmRleCAlIDIwID09IDAgb3IgaW5kZXggPT0gbGVuKHZhbGlkYXRpb25fbmFtZXMpOgogICAgICAgICAgICBwcmludChmIlByZWRpY3RlZCB7aW5kZXh9L3tsZW4odmFsaWRhdGlvbl9uYW1lcyl9IHZhbGlkYXRpb24gb2JzZXJ2YXRpb25zIikKICAgIHByZWRpY3Rpb25fc2Vjb25kcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBwcmVkaWN0aW9uX3N0YXJ0CgogICAgc2NvcmVyX3N0YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgbWV0cmljX3Jlc3VsdCwgb2JzZXJ2YXRpb25fcm93cyA9IHNjb3JlX3ZhbGlkYXRpb24oCiAgICAgICAgZGF0YXNldCwKICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICB2YWxpZGF0aW9uX25hbWVzLAogICAgICAgIGlvdV90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnX2pzb25bImlvdV90aHJlc2hvbGQiXSksCiAgICAgICAgZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZD1mbG9hdChjb25maWdfanNvblsiZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCJdKSwKICAgICkKICAgIHNjb3Jpbmdfc2Vjb25kcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzY29yZXJfc3RhcnQKCiAgICBfd3JpdGVfcHJlZGljdGlvbnMob3V0cHV0X3Jvb3QgLyAicHJlZGljdGlvbnMuY3N2IiwgcHJlZGljdGlvbnMpCiAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKAogICAgICAgIG91dHB1dF9yb290IC8gInBlcl9vYnNlcnZhdGlvbi5jc3YiLCBvYnNlcnZhdGlvbl9yb3dzLCBwcmVkaWN0aW9ucwogICAgKQogICAgb3ZlcmxheV9uYW1lcyA9IF93cml0ZV9vdmVybGF5cygKICAgICAgICBkYXRhc2V0LCBwcmVkaWN0aW9ucywgb3V0cHV0X3Jvb3QgLyAib3ZlcmxheXMiLCBvdmVybGF5X2NvdW50CiAgICApCiAgICBzcGxpdF9iYWxhbmNlOiBkaWN0W3N0ciwgZGljdFtzdHIsIGludF1dID0geyJ5ZWFyIjoge30sICJzaXRlIjoge319CiAgICBmb3IgZmlsZV9uYW1lIGluIHNwbGl0LnZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgdmlldyA9IGRhdGFzZXQudmlld3NfYnlfb2JzZXJ2YXRpb25bZmlsZV9uYW1lXVswXQogICAgICAgIHNwbGl0X2JhbGFuY2VbInllYXIiXVtzdHIodmlldy55ZWFyKV0gPSAoCiAgICAgICAgICAgIHNwbGl0X2JhbGFuY2VbInllYXIiXS5nZXQoc3RyKHZpZXcueWVhciksIDApICsgMQogICAgICAgICkKICAgICAgICBzcGxpdF9iYWxhbmNlWyJzaXRlIl1bdmlldy5zaXRlXSA9IHNwbGl0X2JhbGFuY2VbInNpdGUiXS5nZXQodmlldy5zaXRlLCAwKSArIDEKCiAgICB0b3RhbF9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHJ1bl9zdGFydGVkCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIHBlYWtfZ3B1X2J5dGVzID0gKAogICAgICAgICAgICBpbnQodG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpKQogICAgICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCiAgICAgICAgICAgIGVsc2UgMAogICAgICAgICkKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwZWFrX2dwdV9ieXRlcyA9IDAKCiAgICBydW5fcmVzdWx0OiBkaWN0W3N0ciwgQW55XSA9IHsKICAgICAgICAic3RhdHVzIjogImNvbXBsZXRlIiBpZiBsaW1pdF9vYnNlcnZhdGlvbnMgaXMgTm9uZSBlbHNlICJzbW9rZV9zdWJzZXQiLAogICAgICAgICJjcmVhdGVkX3V0YyI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogICAgICAgICJyZXBvc2l0b3J5IjogX2dpdF9zdGF0dXMocmVwb19yb290KSwKICAgICAgICAia2FnZ2xlX25vdGVib29rX3ZlcnNpb24iOiBvcy5lbnZpcm9uLmdldCgKICAgICAgICAgICAgIktBR0dMRV9LRVJORUxfVkVSU0lPTiIsIG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfTk9URUJPT0tfVkVSU0lPTiIpCiAgICAgICAgKSwKICAgICAgICAiZGF0YV9yb290Ijogc3RyKGRhdGFzZXQucm9vdCksCiAgICAgICAgIm91dHB1dF9yb290Ijogc3RyKG91dHB1dF9yb290KSwKICAgICAgICAiZGF0YXNldF9pbnZlbnRvcnkiOiBpbnB1dF9pbnZlbnRvcnksCiAgICAgICAgInNwbGl0IjogewogICAgICAgICAgICAibWFuaWZlc3QiOiBzdHIoc3BsaXQubWFuaWZlc3RfcGF0aCksCiAgICAgICAgICAgICJzaGEyNTYiOiBzcGxpdC5zaGEyNTYsCiAgICAgICAgICAgICJtYXRjaGVzX2V4cGVjdGVkX3NoYTI1NiI6ICgKICAgICAgICAgICAgICAgIHNwbGl0LnNoYTI1NiA9PSBleHBlY3RlZF9zcGxpdF9zaGEyNTYKICAgICAgICAgICAgICAgIGlmIGV4cGVjdGVkX3NwbGl0X3NoYTI1NgogICAgICAgICAgICAgICAgZWxzZSBOb25lCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJzZWVkIjogaW50KGNvbmZpZ19qc29uWyJzZWVkIl0pLAogICAgICAgICAgICAidmFsaWRhdGlvbl9mcmFjdGlvbiI6IGZsb2F0KGNvbmZpZ19qc29uWyJ2YWxpZGF0aW9uX2ZyYWN0aW9uIl0pLAogICAgICAgICAgICAic3RyYXRpZmljYXRpb24iOiBjb25maWdfanNvblsic3RyYXRpZnlfYnkiXSwKICAgICAgICAgICAgInRyYWluaW5nX29ic2VydmF0aW9ucyI6IGxlbihzcGxpdC50cmFpbmluZ19uYW1lcyksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX29ic2VydmF0aW9ucyI6IGZ1bGxfaG9sZG91dF9jb3VudCwKICAgICAgICAgICAgImV2YWx1YXRlZF9vYnNlcnZhdGlvbnMiOiBsZW4odmFsaWRhdGlvbl9uYW1lcyksCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX2JhbGFuY2UiOiBzcGxpdF9iYWxhbmNlLAogICAgICAgIH0sCiAgICAgICAgImJhc2VsaW5lIjogewogICAgICAgICAgICAqKmNvbmZpZ19qc29uLAogICAgICAgICAgICAibmFtZSI6ICJsb2NhbF9jb250cmFzdF9vdHN1X2Nvbm5lY3RlZF9jb21wb25lbnRzIiwKICAgICAgICAgICAgInRyYWluZWQiOiBGYWxzZSwKICAgICAgICAgICAgImRldmljZV91c2VkIjogImNwdSIsCiAgICAgICAgfSwKICAgICAgICAic2NvcmVyIjogewogICAgICAgICAgICAibmFtZSI6ICJvcmdhbml6ZXJfc2VsZl9ldmFsdWF0aW9uX25vdGVib29rX3NlbWFudGljcyIsCiAgICAgICAgICAgICJyZWZlcmVuY2VfdXJsIjogT0ZGSUNJQUxfU0NPUkVSX1VSTCwKICAgICAgICAgICAgImlvdV90aHJlc2hvbGRfaXNfc3RyaWN0IjogVHJ1ZSwKICAgICAgICAgICAgIm9uZV90b19vbmVfYXNzaWdubWVudCI6IEZhbHNlLAogICAgICAgICAgICAicHJlZGljdGlvbnNfcmV1c2VkX2Zvcl9lYWNoX2Fubm90YXRvcl92aWV3IjogVHJ1ZSwKICAgICAgICB9LAogICAgICAgICJydW50aW1lIjogewogICAgICAgICAgICAqKmhhcmR3YXJlLAogICAgICAgICAgICAiY3VkYV91c2VkX2J5X2Jhc2VsaW5lIjogRmFsc2UsCiAgICAgICAgICAgICJwZWFrX2dwdV9tZW1vcnlfYnl0ZXMiOiBwZWFrX2dwdV9ieXRlcywKICAgICAgICAgICAgInByZWRpY3Rpb25fc2Vjb25kcyI6IHByZWRpY3Rpb25fc2Vjb25kcywKICAgICAgICAgICAgInNjb3Jpbmdfc2Vjb25kcyI6IHNjb3Jpbmdfc2Vjb25kcywKICAgICAgICAgICAgInRvdGFsX3NlY29uZHMiOiB0b3RhbF9zZWNvbmRzLAogICAgICAgICAgICAicGFja2FnZV92ZXJzaW9ucyI6IHBhY2thZ2VfdmVyc2lvbnMoKSwKICAgICAgICB9LAogICAgICAgICJtZXRyaWNzIjogbWV0cmljX3Jlc3VsdCwKICAgICAgICAicHJlZGljdGlvbl9pbnN0YW5jZV9jb3VudCI6IHN1bShsZW4oaXRlbXMpIGZvciBpdGVtcyBpbiBwcmVkaWN0aW9ucy52YWx1ZXMoKSksCiAgICAgICAgIm92ZXJsYXlfb2JzZXJ2YXRpb25zIjogb3ZlcmxheV9uYW1lcywKICAgICAgICAiYXJ0aWZhY3RzIjogewogICAgICAgICAgICAicHJlZGljdGlvbnNfY3N2IjogInByZWRpY3Rpb25zLmNzdiIsCiAgICAgICAgICAgICJwZXJfb2JzZXJ2YXRpb25fY3N2IjogInBlcl9vYnNlcnZhdGlvbi5jc3YiLAogICAgICAgICAgICAib3ZlcmxheXNfZGlyIjogIm92ZXJsYXlzIiwKICAgICAgICB9LAogICAgfQogICAgKG91dHB1dF9yb290IC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhydW5fcmVzdWx0LCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwKICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgKQogICAgcHJpbnQoCiAgICAgICAgIlZhbGlkYXRpb24gUFE6ICIKICAgICAgICBmInttZXRyaWNfcmVzdWx0WydnbG9iYWwnXVsncHEnXTouNmZ9ICIKICAgICAgICBmIih7bWV0cmljX3Jlc3VsdFsnZ2xvYmFsJ11bJ3RwJ119IFRQIGVkZ2VzLCAiCiAgICAgICAgZiJ7bWV0cmljX3Jlc3VsdFsnZ2xvYmFsJ11bJ2ZwJ119IEZQLCB7bWV0cmljX3Jlc3VsdFsnZ2xvYmFsJ11bJ2ZuJ119IEZOKSIKICAgICkKICAgIHByaW50KGYiU3BsaXQgU0hBLTI1Njoge3NwbGl0LnNoYTI1Nn0iKQogICAgcHJpbnQoZiJSdW4gYXJ0aWZhY3RzOiB7b3V0cHV0X3Jvb3R9IikKICAgIHJldHVybiBydW5fcmVzdWx0CgoKZGVmIG1haW4oYXJndjogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJSdW4gdGhlIGZpeGVkIGdyb3VwZWQgdmFsaWRhdGlvbiBiYXNlbGluZS4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXBvLXJvb3QiLCBkZWZhdWx0PXN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXSkpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWxpbWl0LW9ic2VydmF0aW9ucyIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW92ZXJsYXktY291bnQiLCB0eXBlPWludCwgZGVmYXVsdD0xMikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncyhhcmd2KQogICAgcmVzdWx0ID0gcnVuX3ZhbGlkYXRpb24oCiAgICAgICAgcmVwb19yb290PWFyZ3MucmVwb19yb290LAogICAgICAgIGRhdGFfcm9vdD1hcmdzLmRhdGFfcm9vdCwKICAgICAgICBvdXRwdXRfcm9vdD1hcmdzLm91dHB1dF9yb290LAogICAgICAgIGxpbWl0X29ic2VydmF0aW9ucz1hcmdzLmxpbWl0X29ic2VydmF0aW9ucywKICAgICAgICBvdmVybGF5X2NvdW50PWFyZ3Mub3ZlcmxheV9jb3VudCwKICAgICkKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0WyJtZXRyaWNzIl1bImdsb2JhbCJdLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==","src/filament_baseline/runtime.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKClRSQUNLRURfUlVOVElNRV9QQUNLQUdFUyA9ICgKICAgICJudW1weSIsCiAgICAib3BlbmN2LXB5dGhvbi1oZWFkbGVzcyIsCiAgICAicGFuZGFzIiwKICAgICJzY2lraXQtbGVhcm4iLAogICAgInNjaXB5IiwKICAgICJtYXRwbG90bGliIiwKICAgICJweWNvY290b29scyIsCiAgICAidG9yY2giLAopCgoKZGVmIHBhY2thZ2VfdmVyc2lvbnMoKSAtPiBkaWN0W3N0ciwgc3RyIHwgTm9uZV06CiAgICB2ZXJzaW9uczogZGljdFtzdHIsIHN0ciB8IE5vbmVdID0ge30KICAgIGZvciBwYWNrYWdlIGluIFRSQUNLRURfUlVOVElNRV9QQUNLQUdFUzoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZlcnNpb25zW3BhY2thZ2VdID0gaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24ocGFja2FnZSkKICAgICAgICBleGNlcHQgaW1wb3J0bGliLm1ldGFkYXRhLlBhY2thZ2VOb3RGb3VuZEVycm9yOgogICAgICAgICAgICB2ZXJzaW9uc1twYWNrYWdlXSA9IE5vbmUKICAgIHJldHVybiB2ZXJzaW9ucwoKCmRlZiBoYXJkd2FyZV9pbmZvKCkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXN1bHQ6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgInB5dGhvbiI6IHN5cy52ZXJzaW9uLnNwbGl0KClbMF0sCiAgICAgICAgImN1ZGFfYXZhaWxhYmxlIjogRmFsc2UsCiAgICAgICAgInRvcmNoX3ZlcnNpb24iOiBOb25lLAogICAgICAgICJ0b3JjaF9jdWRhX3ZlcnNpb24iOiBOb25lLAogICAgICAgICJncHVfZGV2aWNlcyI6IFtdLAogICAgICAgICJudmlkaWFfc21pX3N1bW1hcnkiOiBOb25lLAogICAgfQogICAgdHJ5OgogICAgICAgIHJlc3VsdFsibnZpZGlhX3NtaV9zdW1tYXJ5Il0gPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgIm52aWRpYS1zbWkiLAogICAgICAgICAgICAgICAgIi0tcXVlcnktZ3B1PW5hbWUsbWVtb3J5LnRvdGFsLGRyaXZlcl92ZXJzaW9uIiwKICAgICAgICAgICAgICAgICItLWZvcm1hdD1jc3Ysbm9oZWFkZXIiLAogICAgICAgICAgICBdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKS5zcGxpdGxpbmVzKCkKICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3IpOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgcmVzdWx0WyJ0b3JjaF92ZXJzaW9uIl0gPSB0b3JjaC5fX3ZlcnNpb25fXwogICAgcmVzdWx0WyJ0b3JjaF9jdWRhX3ZlcnNpb24iXSA9IHRvcmNoLnZlcnNpb24uY3VkYQogICAgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdID0gYm9vbCh0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQogICAgaWYgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdOgogICAgICAgIGRldmljZXMgPSBbXQogICAgICAgIGZvciBpbmRleCBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKToKICAgICAgICAgICAgcHJvcHMgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhpbmRleCkKICAgICAgICAgICAgZGV2aWNlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImluZGV4IjogaW5kZXgsCiAgICAgICAgICAgICAgICAgICAgIm5hbWUiOiBwcm9wcy5uYW1lLAogICAgICAgICAgICAgICAgICAgICJ0b3RhbF9tZW1vcnlfYnl0ZXMiOiBpbnQocHJvcHMudG90YWxfbWVtb3J5KSwKICAgICAgICAgICAgICAgICAgICAiY29tcHV0ZV9jYXBhYmlsaXR5IjogZiJ7cHJvcHMubWFqb3J9Lntwcm9wcy5taW5vcn0iLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICAgICAgcmVzdWx0WyJncHVfZGV2aWNlcyJdID0gZGV2aWNlcwogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBnaXRfaW5mbyhyZXBvX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgcmVwb19yb290ID0gUGF0aChyZXBvX3Jvb3QpCiAgICB0cnk6CiAgICAgICAgY29tbWl0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZ2l0IiwgIi1DIiwgc3RyKHJlcG9fcm9vdCksICJyZXYtcGFyc2UiLCAiSEVBRCJdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKQogICAgICAgIHN0YXR1cyA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbImdpdCIsICItQyIsIHN0cihyZXBvX3Jvb3QpLCAic3RhdHVzIiwgIi0tcG9yY2VsYWluIl0sCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICApLnN0ZG91dAogICAgICAgIHJldHVybiB7ImNvbW1pdCI6IGNvbW1pdCwgIndvcmt0cmVlX2RpcnR5IjogYm9vbChzdGF0dXMuc3RyaXAoKSl9CiAgICBleGNlcHQgKE9TRXJyb3IsIHN1YnByb2Nlc3MuQ2FsbGVkUHJvY2Vzc0Vycm9yKToKICAgICAgICByZXR1cm4geyJjb21taXQiOiBOb25lLCAid29ya3RyZWVfZGlydHkiOiBOb25lfQo=","src/filament_baseline/scorer.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlciwgZGVmYXVsdGRpY3QKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBJdGVyYWJsZQoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gcHljb2NvdG9vbHMgaW1wb3J0IG1hc2sgYXMgbWFza191dGlscwoKZnJvbSAuZGF0YSBpbXBvcnQgQ29tcGV0aXRpb25EYXRhc2V0LCBJbWFnZVZpZXcKZnJvbSAucHJlZGljdG9yIGltcG9ydCBQcmVkaWN0ZWRJbnN0YW5jZQoKCkBkYXRhY2xhc3MKY2xhc3MgT2JzZXJ2YXRpb25Db3VudHM6CiAgICBmaWxlX25hbWU6IHN0cgogICAgeWVhcjogaW50CiAgICBzaXRlOiBzdHIKICAgIGFubm90YXRvcl92aWV3czogaW50ID0gMAogICAgZ3RfY291bnQ6IGludCA9IDAKICAgIHByZWRpY3Rpb25fY291bnQ6IGludCA9IDAKICAgIHRwX2lvdXM6IGxpc3RbZmxvYXRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBmcDogaW50ID0gMAogICAgZm46IGludCA9IDAKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0cChzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLnRwX2lvdXMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHBfaW91X3N1bShzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gZmxvYXQoc3VtKHNlbGYudHBfaW91cykpCgogICAgQHByb3BlcnR5CiAgICBkZWYgcHEoc2VsZikgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuIHBxX3ZhbHVlKHNlbGYudHBfaW91cywgc2VsZi5mcCwgc2VsZi5mbikKCgpAZGF0YWNsYXNzCmNsYXNzIFNjb3JlckRpYWdub3N0aWNzOgogICAgcG9zaXRpdmVfaW91czogbGlzdFtmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHBvc2l0aXZlX2RpY2VzOiBsaXN0W2Zsb2F0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZ3RfbWF0Y2hfZGVncmVlczogQ291bnRlcltpbnRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PUNvdW50ZXIpCiAgICBwcmVkaWN0aW9uX21hdGNoX2RlZ3JlZXM6IENvdW50ZXJbaW50XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1Db3VudGVyKQoKCmRlZiBfY29jb19ybGUocmxlOiBkaWN0W3N0ciwgQW55XSkgLT4gZGljdFtzdHIsIEFueV06CiAgICBjb3VudHMgPSBybGVbImNvdW50cyJdCiAgICBpZiBpc2luc3RhbmNlKGNvdW50cywgc3RyKToKICAgICAgICBjb3VudHMgPSBjb3VudHMuZW5jb2RlKCJhc2NpaSIpCiAgICByZXR1cm4geyJzaXplIjogW2ludChybGVbInNpemUiXVswXSksIGludChybGVbInNpemUiXVsxXSldLCAiY291bnRzIjogY291bnRzfQoKCmRlZiBhbm5vdGF0aW9uX3RvX3JsZShhbm5vdGF0aW9uOiBkaWN0W3N0ciwgQW55XSwgaGVpZ2h0OiBpbnQsIHdpZHRoOiBpbnQpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgc2VnbWVudGF0aW9uID0gYW5ub3RhdGlvblsic2VnbWVudGF0aW9uIl0KICAgIGlmIGlzaW5zdGFuY2Uoc2VnbWVudGF0aW9uLCBkaWN0KToKICAgICAgICByZXR1cm4gX2NvY29fcmxlKHNlZ21lbnRhdGlvbikKICAgIGlmIG5vdCBpc2luc3RhbmNlKHNlZ21lbnRhdGlvbiwgbGlzdCkgb3Igbm90IHNlZ21lbnRhdGlvbjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgb3IgZW1wdHkgc2VnbWVudGF0aW9uIGZvciBhbm5vdGF0aW9uIHthbm5vdGF0aW9uLmdldCgnaWQnKX0iKQogICAgcmxlcyA9IG1hc2tfdXRpbHMuZnJQeU9iamVjdHMoc2VnbWVudGF0aW9uLCBoZWlnaHQsIHdpZHRoKQogICAgbWVyZ2VkID0gbWFza191dGlscy5tZXJnZShybGVzKSBpZiBpc2luc3RhbmNlKHJsZXMsIGxpc3QpIGVsc2UgcmxlcwogICAgcmV0dXJuIF9jb2NvX3JsZShtZXJnZWQpCgoKZGVmIHBhaXJ3aXNlX2lvdSgKICAgIGd0X3JsZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dLAogICAgcHJlZGljdGlvbl9ybGVzOiBsaXN0W2RpY3Rbc3RyLCBBbnldXSwKKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIChuX2d0LCBuX3ByZWQpIGZsb2F0MzIgSW9VcywgbWF0Y2hpbmcgdGhlIG9yZ2FuaXplcidzIHRlbnNvciBkdHlwZS4iIiIKICAgIGlmIG5vdCBndF9ybGVzIG9yIG5vdCBwcmVkaWN0aW9uX3JsZXM6CiAgICAgICAgcmV0dXJuIG5wLnplcm9zKChsZW4oZ3RfcmxlcyksIGxlbihwcmVkaWN0aW9uX3JsZXMpKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICMgcHljb2NvdG9vbHMgcmV0dXJucyAoZGV0ZWN0aW9ucywgZ3JvdW5kIHRydXRoKSBpbiBmbG9hdDY0LiBUaGUgb3JnYW5pemVyCiAgICAjIGRlY29kZXMgdGhlIHNhbWUgUkxFcyB0byBmbG9hdDMyIGxheWVycyBhbmQgY29tcHV0ZXMgZmxvYXQzMiBJb1VzLgogICAgbWF0cml4ID0gbWFza191dGlscy5pb3UoCiAgICAgICAgW19jb2NvX3JsZShybGUpIGZvciBybGUgaW4gcHJlZGljdGlvbl9ybGVzXSwKICAgICAgICBbX2NvY29fcmxlKHJsZSkgZm9yIHJsZSBpbiBndF9ybGVzXSwKICAgICAgICBbMF0gKiBsZW4oZ3RfcmxlcyksCiAgICApCiAgICByZXR1cm4gbnAuYXNhcnJheShtYXRyaXgsIGR0eXBlPW5wLmZsb2F0MzIpLlQKCgpkZWYgcHFfdmFsdWUodHBfaW91czogSXRlcmFibGVbZmxvYXRdLCBmcDogaW50LCBmbjogaW50KSAtPiBmbG9hdDoKICAgIHRwX2lvdXMgPSBsaXN0KHRwX2lvdXMpCiAgICBkZW5vbWluYXRvciA9IGxlbih0cF9pb3VzKSArIDAuNSAqIGZwICsgMC41ICogZm4KICAgIHJldHVybiBmbG9hdChzdW0odHBfaW91cykgLyBkZW5vbWluYXRvcikgaWYgZGVub21pbmF0b3IgPiAwIGVsc2UgMC4wCgoKZGVmIF9hY2N1bXVsYXRlX3ZpZXcoCiAgICBjb3VudHM6IE9ic2VydmF0aW9uQ291bnRzLAogICAgZGlhZ25vc3RpY3M6IFNjb3JlckRpYWdub3N0aWNzLAogICAgZ3RfcmxlczogbGlzdFtkaWN0W3N0ciwgQW55XV0sCiAgICBwcmVkaWN0aW9uX3JsZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dLAogICAgaW91X3RocmVzaG9sZDogZmxvYXQsCiAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkOiBmbG9hdCwKKSAtPiBOb25lOgogICAgY291bnRzLmd0X2NvdW50ICs9IGxlbihndF9ybGVzKQogICAgY291bnRzLnByZWRpY3Rpb25fY291bnQgKz0gbGVuKHByZWRpY3Rpb25fcmxlcykKICAgIGlmIG5vdCBndF9ybGVzOgogICAgICAgIGNvdW50cy5mcCArPSBsZW4ocHJlZGljdGlvbl9ybGVzKQogICAgICAgIGRpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcy51cGRhdGUoezA6IGxlbihwcmVkaWN0aW9uX3JsZXMpfSkKICAgICAgICByZXR1cm4KICAgIGlmIG5vdCBwcmVkaWN0aW9uX3JsZXM6CiAgICAgICAgY291bnRzLmZuICs9IGxlbihndF9ybGVzKQogICAgICAgIGRpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMudXBkYXRlKHswOiBsZW4oZ3Rfcmxlcyl9KQogICAgICAgIHJldHVybgoKICAgIGlvdXMgPSBwYWlyd2lzZV9pb3UoZ3RfcmxlcywgcHJlZGljdGlvbl9ybGVzKQogICAgaGl0cyA9IGlvdXMgPiBpb3VfdGhyZXNob2xkCgogICAgIyBUaGlzIGludGVudGlvbmFsbHkgbWlycm9ycyB0aGUgcHJvdmlkZWQgc2NvcmVyOiBldmVyeSBxdWFsaWZ5aW5nIGVkZ2UgaXMKICAgICMgYSBUUDsgaXQgZG9lcyBub3QgcGVyZm9ybSBvbmUtdG8tb25lIGFzc2lnbm1lbnQuCiAgICBjb3VudHMudHBfaW91cy5leHRlbmQoZmxvYXQodmFsdWUpIGZvciB2YWx1ZSBpbiBpb3VzW2hpdHNdKQogICAgY291bnRzLmZwICs9IGludChucC5jb3VudF9ub256ZXJvKG5wLnN1bShoaXRzLCBheGlzPTApID09IDApKQogICAgY291bnRzLmZuICs9IGludChucC5jb3VudF9ub256ZXJvKG5wLnN1bShoaXRzLCBheGlzPTEpID09IDApKQoKICAgIHBvc2l0aXZlID0gaW91c1tpb3VzID4gMF0KICAgIGRpYWdub3N0aWNzLnBvc2l0aXZlX2lvdXMuZXh0ZW5kKGZsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gcG9zaXRpdmUpCiAgICBkaWFnbm9zdGljcy5wb3NpdGl2ZV9kaWNlcy5leHRlbmQoCiAgICAgICAgZmxvYXQodmFsdWUpIGZvciB2YWx1ZSBpbiAoMi4wICogcG9zaXRpdmUgLyAoMS4wICsgcG9zaXRpdmUpKQogICAgKQoKICAgIGFueV9vdmVybGFwID0gaW91cyA+IGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQKICAgIGRpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMudXBkYXRlKGludCh2YWx1ZSkgZm9yIHZhbHVlIGluIG5wLnN1bShhbnlfb3ZlcmxhcCwgYXhpcz0xKSkKICAgIGRpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcy51cGRhdGUoCiAgICAgICAgaW50KHZhbHVlKSBmb3IgdmFsdWUgaW4gbnAuc3VtKGFueV9vdmVybGFwLCBheGlzPTApCiAgICApCgoKZGVmIHNjb3JlX29ic2VydmF0aW9uKAogICAgZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0LAogICAgZmlsZV9uYW1lOiBzdHIsCiAgICBwcmVkaWN0aW9uczogbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV0sCiAgICAqLAogICAgaW91X3RocmVzaG9sZDogZmxvYXQgPSAwLjUsCiAgICBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkOiBmbG9hdCA9IDAuMCwKKSAtPiB0dXBsZVtPYnNlcnZhdGlvbkNvdW50cywgU2NvcmVyRGlhZ25vc3RpY3NdOgogICAgdmlld3MgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV0KICAgIGNvdW50cyA9IE9ic2VydmF0aW9uQ291bnRzKAogICAgICAgIGZpbGVfbmFtZT1maWxlX25hbWUsCiAgICAgICAgeWVhcj12aWV3c1swXS55ZWFyLAogICAgICAgIHNpdGU9dmlld3NbMF0uc2l0ZSwKICAgICAgICBhbm5vdGF0b3Jfdmlld3M9bGVuKHZpZXdzKSwKICAgICkKICAgIGRpYWdub3N0aWNzID0gU2NvcmVyRGlhZ25vc3RpY3MoKQogICAgcHJlZGljdGlvbl9ybGVzID0gW3ByZWRpY3Rpb24ucmxlIGZvciBwcmVkaWN0aW9uIGluIHByZWRpY3Rpb25zXQogICAgZm9yIHZpZXcgaW4gdmlld3M6CiAgICAgICAgYW5ub3RhdGlvbnMgPSBkYXRhc2V0LmFubm90YXRpb25zX2J5X2ltYWdlX2lkW3ZpZXcuaW1hZ2VfaWRdCiAgICAgICAgZ3RfcmxlcyA9IFsKICAgICAgICAgICAgYW5ub3RhdGlvbl90b19ybGUoYW5ub3RhdGlvbiwgdmlldy5oZWlnaHQsIHZpZXcud2lkdGgpCiAgICAgICAgICAgIGZvciBhbm5vdGF0aW9uIGluIGFubm90YXRpb25zCiAgICAgICAgXQogICAgICAgIF9hY2N1bXVsYXRlX3ZpZXcoCiAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgZGlhZ25vc3RpY3MsCiAgICAgICAgICAgIGd0X3JsZXMsCiAgICAgICAgICAgIHByZWRpY3Rpb25fcmxlcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZCwKICAgICAgICAgICAgZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCwKICAgICAgICApCiAgICByZXR1cm4gY291bnRzLCBkaWFnbm9zdGljcwoKCmRlZiBfYWdncmVnYXRlX2NvdW50cyhyb3dzOiBJdGVyYWJsZVtPYnNlcnZhdGlvbkNvdW50c10pIC0+IGRpY3Rbc3RyLCBmbG9hdCB8IGludF06CiAgICB0cF9pb3VzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICBmcCA9IDAKICAgIGZuID0gMAogICAgZ3RfY291bnQgPSAwCiAgICBwcmVkaWN0aW9uX2NvdW50ID0gMAogICAgYW5ub3RhdG9yX3ZpZXdzID0gMAogICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgIHRwX2lvdXMuZXh0ZW5kKHJvdy50cF9pb3VzKQogICAgICAgIGZwICs9IHJvdy5mcAogICAgICAgIGZuICs9IHJvdy5mbgogICAgICAgIGd0X2NvdW50ICs9IHJvdy5ndF9jb3VudAogICAgICAgIHByZWRpY3Rpb25fY291bnQgKz0gcm93LnByZWRpY3Rpb25fY291bnQKICAgICAgICBhbm5vdGF0b3Jfdmlld3MgKz0gcm93LmFubm90YXRvcl92aWV3cwogICAgcmV0dXJuIHsKICAgICAgICAicHEiOiBwcV92YWx1ZSh0cF9pb3VzLCBmcCwgZm4pLAogICAgICAgICJ0cCI6IGxlbih0cF9pb3VzKSwKICAgICAgICAidHBfaW91X3N1bSI6IGZsb2F0KHN1bSh0cF9pb3VzKSksCiAgICAgICAgImZwIjogZnAsCiAgICAgICAgImZuIjogZm4sCiAgICAgICAgImd0X2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiOiBndF9jb3VudCwKICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3NfYW5ub3RhdG9yX3ZpZXdzIjogcHJlZGljdGlvbl9jb3VudCwKICAgICAgICAiYW5ub3RhdG9yX3ZpZXdfY291bnQiOiBhbm5vdGF0b3Jfdmlld3MsCiAgICB9CgoKZGVmIF9oaXN0b2dyYW0odmFsdWVzOiBsaXN0W2Zsb2F0XSkgLT4gZGljdFtzdHIsIEFueV06CiAgICBiaW5zID0gbnAubGluc3BhY2UoMC4wLCAxLjAsIDUxLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgY291bnRzLCBlZGdlcyA9IG5wLmhpc3RvZ3JhbSh2YWx1ZXMsIGJpbnM9YmlucykKICAgIHRvdGFsID0gaW50KGNvdW50cy5zdW0oKSkKICAgIHJldHVybiB7CiAgICAgICAgImNvdW50IjogaW50KGxlbih2YWx1ZXMpKSwKICAgICAgICAibWVhbiI6IGZsb2F0KG5wLm1lYW4odmFsdWVzKSkgaWYgdmFsdWVzIGVsc2UgMC4wLAogICAgICAgICJiaW5zIjogW2Zsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gZWRnZXNdLAogICAgICAgICJjb3VudHMiOiBbaW50KHZhbHVlKSBmb3IgdmFsdWUgaW4gY291bnRzXSwKICAgICAgICAicHJvYmFiaWxpdGllcyI6IFtmbG9hdCh2YWx1ZSAvIHRvdGFsKSBmb3IgdmFsdWUgaW4gY291bnRzXSBpZiB0b3RhbCBlbHNlIFswLjBdICogNTAsCiAgICB9CgoKZGVmIF9kZWdyZWVfZGlzdHJpYnV0aW9uKGNvdW50ZXI6IENvdW50ZXJbaW50XSkgLT4gZGljdFtzdHIsIGludF06CiAgICByZXR1cm4ge3N0cihkZWdyZWUpOiBpbnQoY291bnRlcltkZWdyZWVdKSBmb3IgZGVncmVlIGluIHNvcnRlZChjb3VudGVyKX0KCgpkZWYgc2NvcmVfdmFsaWRhdGlvbigKICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgIHByZWRpY3Rpb25zX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgbGlzdFtQcmVkaWN0ZWRJbnN0YW5jZV1dLAogICAgdmFsaWRhdGlvbl9uYW1lczogSXRlcmFibGVbc3RyXSwKICAgICosCiAgICBpb3VfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwKICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ6IGZsb2F0ID0gMC4wLAopIC0+IHR1cGxlW2RpY3Rbc3RyLCBBbnldLCBsaXN0W09ic2VydmF0aW9uQ291bnRzXV06CiAgICByb3dzOiBsaXN0W09ic2VydmF0aW9uQ291bnRzXSA9IFtdCiAgICBhbGxfZGlhZ25vc3RpY3MgPSBTY29yZXJEaWFnbm9zdGljcygpCiAgICBmb3IgZmlsZV9uYW1lIGluIHNvcnRlZCh2YWxpZGF0aW9uX25hbWVzKToKICAgICAgICBjb3VudHMsIGRpYWdub3N0aWNzID0gc2NvcmVfb2JzZXJ2YXRpb24oCiAgICAgICAgICAgIGRhdGFzZXQsCiAgICAgICAgICAgIGZpbGVfbmFtZSwKICAgICAgICAgICAgcHJlZGljdGlvbnNfYnlfb2JzZXJ2YXRpb24uZ2V0KGZpbGVfbmFtZSwgW10pLAogICAgICAgICAgICBpb3VfdGhyZXNob2xkPWlvdV90aHJlc2hvbGQsCiAgICAgICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZCwKICAgICAgICApCiAgICAgICAgcm93cy5hcHBlbmQoY291bnRzKQogICAgICAgIGFsbF9kaWFnbm9zdGljcy5wb3NpdGl2ZV9pb3VzLmV4dGVuZChkaWFnbm9zdGljcy5wb3NpdGl2ZV9pb3VzKQogICAgICAgIGFsbF9kaWFnbm9zdGljcy5wb3NpdGl2ZV9kaWNlcy5leHRlbmQoZGlhZ25vc3RpY3MucG9zaXRpdmVfZGljZXMpCiAgICAgICAgYWxsX2RpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMudXBkYXRlKGRpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMpCiAgICAgICAgYWxsX2RpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcy51cGRhdGUoCiAgICAgICAgICAgIGRpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcwogICAgICAgICkKCiAgICBncm91cGVkOiBkaWN0W3N0ciwgZGljdFtzdHIsIGxpc3RbT2JzZXJ2YXRpb25Db3VudHNdXV0gPSB7CiAgICAgICAgInllYXIiOiBkZWZhdWx0ZGljdChsaXN0KSwKICAgICAgICAic2l0ZSI6IGRlZmF1bHRkaWN0KGxpc3QpLAogICAgfQogICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgIGdyb3VwZWRbInllYXIiXVtzdHIocm93LnllYXIpXS5hcHBlbmQocm93KQogICAgICAgIGdyb3VwZWRbInNpdGUiXVtyb3cuc2l0ZV0uYXBwZW5kKHJvdykKCiAgICByZXN1bHQgPSB7CiAgICAgICAgImdsb2JhbCI6IF9hZ2dyZWdhdGVfY291bnRzKHJvd3MpLAogICAgICAgICJieV95ZWFyIjogewogICAgICAgICAgICBsYWJlbDogX2FnZ3JlZ2F0ZV9jb3VudHMoZ3JvdXBfcm93cykKICAgICAgICAgICAgZm9yIGxhYmVsLCBncm91cF9yb3dzIGluIHNvcnRlZChncm91cGVkWyJ5ZWFyIl0uaXRlbXMoKSkKICAgICAgICB9LAogICAgICAgICJieV9zaXRlIjogewogICAgICAgICAgICBsYWJlbDogX2FnZ3JlZ2F0ZV9jb3VudHMoZ3JvdXBfcm93cykKICAgICAgICAgICAgZm9yIGxhYmVsLCBncm91cF9yb3dzIGluIHNvcnRlZChncm91cGVkWyJzaXRlIl0uaXRlbXMoKSkKICAgICAgICB9LAogICAgICAgICJpb3VfcG9zaXRpdmVfcGFpcnMiOiBfaGlzdG9ncmFtKGFsbF9kaWFnbm9zdGljcy5wb3NpdGl2ZV9pb3VzKSwKICAgICAgICAiZGljZV9wb3NpdGl2ZV9wYWlycyI6IF9oaXN0b2dyYW0oYWxsX2RpYWdub3N0aWNzLnBvc2l0aXZlX2RpY2VzKSwKICAgICAgICAib25lX3RvX21hbnlfbWFueV90b19vbmUiOiB7CiAgICAgICAgICAgICJ0aHJlc2hvbGQiOiBkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkLAogICAgICAgICAgICAiZ3RfbWF0Y2hfZGVncmVlX2NvdW50cyI6IF9kZWdyZWVfZGlzdHJpYnV0aW9uKAogICAgICAgICAgICAgICAgYWxsX2RpYWdub3N0aWNzLmd0X21hdGNoX2RlZ3JlZXMKICAgICAgICAgICAgKSwKICAgICAgICAgICAgInByZWRpY3Rpb25fbWF0Y2hfZGVncmVlX2NvdW50cyI6IF9kZWdyZWVfZGlzdHJpYnV0aW9uKAogICAgICAgICAgICAgICAgYWxsX2RpYWdub3N0aWNzLnByZWRpY3Rpb25fbWF0Y2hfZGVncmVlcwogICAgICAgICAgICApLAogICAgICAgIH0sCiAgICB9CiAgICByZXR1cm4gcmVzdWx0LCByb3dzCgoKZGVmIHZpZXdzX2Zvcl9vYnNlcnZhdGlvbihkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsIGZpbGVfbmFtZTogc3RyKSAtPiB0dXBsZVtJbWFnZVZpZXcsIC4uLl06CiAgICByZXR1cm4gZGF0YXNldC52aWV3c19ieV9vYnNlcnZhdGlvbltmaWxlX25hbWVdCg==","src/filament_baseline/split.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkU2h1ZmZsZVNwbGl0Cgpmcm9tIC5kYXRhIGltcG9ydCBDb21wZXRpdGlvbkRhdGFzZXQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBPYnNlcnZhdGlvblNwbGl0OgogICAgYXNzaWdubWVudHM6IGRpY3Rbc3RyLCBzdHJdCiAgICBtYW5pZmVzdF9wYXRoOiBQYXRoCiAgICBzaGEyNTY6IHN0cgoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHZhbGlkYXRpb25fbmFtZXMoc2VsZikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgICAgIHJldHVybiB0dXBsZSgKICAgICAgICAgICAgc29ydGVkKG5hbWUgZm9yIG5hbWUsIHBhcnRpdGlvbiBpbiBzZWxmLmFzc2lnbm1lbnRzLml0ZW1zKCkgaWYgcGFydGl0aW9uID09ICJ2YWxpZGF0aW9uIikKICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHJhaW5pbmdfbmFtZXMoc2VsZikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgICAgIHJldHVybiB0dXBsZSgKICAgICAgICAgICAgc29ydGVkKG5hbWUgZm9yIG5hbWUsIHBhcnRpdGlvbiBpbiBzZWxmLmFzc2lnbm1lbnRzLml0ZW1zKCkgaWYgcGFydGl0aW9uID09ICJ0cmFpbiIpCiAgICAgICAgKQoKCmRlZiBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICBkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsCiAgICBzcGxpdF9kaXI6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgc2VlZDogaW50ID0gNDIsCiAgICB2YWxpZGF0aW9uX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMiwKKSAtPiBPYnNlcnZhdGlvblNwbGl0OgogICAgbmFtZXMgPSBsaXN0KGRhdGFzZXQub2JzZXJ2YXRpb25fbmFtZXMpCiAgICB5ZWFycyA9IFtkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdLnllYXIgZm9yIG5hbWUgaW4gbmFtZXNdCiAgICBzcGxpdHRlciA9IFN0cmF0aWZpZWRTaHVmZmxlU3BsaXQoCiAgICAgICAgbl9zcGxpdHM9MSwgdGVzdF9zaXplPXZhbGlkYXRpb25fZnJhY3Rpb24sIHJhbmRvbV9zdGF0ZT1zZWVkCiAgICApCiAgICB0cmFpbl9pbmRpY2VzLCB2YWxpZGF0aW9uX2luZGljZXMgPSBuZXh0KHNwbGl0dGVyLnNwbGl0KG5hbWVzLCB5ZWFycykpCiAgICBhc3NpZ25tZW50cyA9IHtuYW1lOiAidHJhaW4iIGZvciBuYW1lIGluIG5hbWVzfQogICAgZm9yIGluZGV4IGluIHZhbGlkYXRpb25faW5kaWNlczoKICAgICAgICBhc3NpZ25tZW50c1tuYW1lc1tpbnQoaW5kZXgpXV0gPSAidmFsaWRhdGlvbiIKCiAgICByb3dzID0gW10KICAgIGZvciBuYW1lIGluIG5hbWVzOgogICAgICAgIHZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJmaWxlX25hbWUiOiBuYW1lLAogICAgICAgICAgICAgICAgInllYXIiOiB2aWV3LnllYXIsCiAgICAgICAgICAgICAgICAic2l0ZSI6IHZpZXcuc2l0ZSwKICAgICAgICAgICAgICAgICJwYXJ0aXRpb24iOiBhc3NpZ25tZW50c1tuYW1lXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBzcGxpdF9kaXIgPSBQYXRoKHNwbGl0X2RpcikKICAgIHNwbGl0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBtYW5pZmVzdF9wYXRoID0gc3BsaXRfZGlyIC8gZiJvYnNlcnZhdGlvbl9zcGxpdF9zZWVke3NlZWR9LmNzdiIKICAgIGZyb20gaW8gaW1wb3J0IFN0cmluZ0lPCgogICAgYnVmZmVyID0gU3RyaW5nSU8obmV3bGluZT0iIikKICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKAogICAgICAgIGJ1ZmZlciwKICAgICAgICBmaWVsZG5hbWVzPSgiZmlsZV9uYW1lIiwgInllYXIiLCAic2l0ZSIsICJwYXJ0aXRpb24iKSwKICAgICAgICBsaW5ldGVybWluYXRvcj0iXG4iLAogICAgKQogICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIG1hbmlmZXN0X2J5dGVzID0gYnVmZmVyLmdldHZhbHVlKCkuZW5jb2RlKCJ1dGYtOCIpCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihtYW5pZmVzdF9ieXRlcykuaGV4ZGlnZXN0KCkKCiAgICBpZiBtYW5pZmVzdF9wYXRoLmV4aXN0cygpOgogICAgICAgIGV4aXN0aW5nID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgICAgICBpZiBleGlzdGluZyAhPSBtYW5pZmVzdF9ieXRlczoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgZiJFeGlzdGluZyBzcGxpdCBtYW5pZmVzdCBkaWZmZXJzIGZyb20gdGhlIGRldGVybWluaXN0aWMgc3BsaXQ6IHttYW5pZmVzdF9wYXRofSIKICAgICAgICAgICAgKQogICAgZWxzZToKICAgICAgICBtYW5pZmVzdF9wYXRoLndyaXRlX2J5dGVzKG1hbmlmZXN0X2J5dGVzKQoKICAgIGlmIHNldChhc3NpZ25tZW50cy52YWx1ZXMoKSkgIT0geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIlRoZSBzcGxpdCBtdXN0IGNvbnRhaW4gYm90aCB0cmFpbmluZyBhbmQgdmFsaWRhdGlvbiBvYnNlcnZhdGlvbnMiKQogICAgaWYgc2V0KGFzc2lnbm1lbnRzKSAhPSBzZXQobmFtZXMpOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJUaGUgc3BsaXQgbWFuaWZlc3QgZG9lcyBub3QgY292ZXIgZXZlcnkgb3JpZ2luYWwgb2JzZXJ2YXRpb24iKQogICAgaWYgc2V0KGRhdGFzZXQub2JzZXJ2YXRpb25fbmFtZXMpICE9IHNldChhc3NpZ25tZW50cyk6CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIk9ic2VydmF0aW9uIHNwbGl0IGtleXMgZG8gbm90IG1hdGNoIENPQ08gZmlsZV9uYW1lIGdyb3VwcyIpCgogICAgcmV0dXJuIE9ic2VydmF0aW9uU3BsaXQoYXNzaWdubWVudHMsIG1hbmlmZXN0X3BhdGgsIGRpZ2VzdCkK","src/filament_baseline/structure_sweep.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmRhdGEgaW1wb3J0IGxvYWRfZGF0YXNldApmcm9tIC5pbmZlcmVuY2UgaW1wb3J0IEluc3RhbmNlQ29uZmlnLCBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzCmZyb20gLnJ1bm5lciBpbXBvcnQgKAogICAgX3Jlc29sdmVfZGF0YV9yb290LAogICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcywKICAgIF93cml0ZV9vdmVybGF5cywKICAgIF93cml0ZV9wcmVkaWN0aW9ucywKKQpmcm9tIC5zY29yZXIgaW1wb3J0IHNjb3JlX3ZhbGlkYXRpb24KZnJvbSAuc3BsaXQgaW1wb3J0IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdApmcm9tIC52aXN1YWxzIGltcG9ydCB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzCgoKZGVmIHJ1bl9pbnN0YW5jZV9zdHJ1Y3R1cmVfc3dlZXAoCiAgICAqLAogICAgcmVwb19yb290OiBzdHIgfCBQYXRoLAogICAgZGF0YV9yb290OiBzdHIgfCBQYXRoIHwgTm9uZSwKICAgIHJ1bl9yb290OiBzdHIgfCBQYXRoLAogICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ6IGZsb2F0ID0gMC41NSwKICAgIGJvdW5kYXJ5X3RocmVzaG9sZDogZmxvYXQgPSAwLjYwLAogICAgYm91bmRhcnlfd2VpZ2h0OiBmbG9hdCA9IDEuNSwKICAgIGJvdW5kYXJ5X3dlaWdodHM6IHR1cGxlW2Zsb2F0LCAuLi5dIHwgTm9uZSA9IE5vbmUsCiAgICBzZWVkX3RocmVzaG9sZDogZmxvYXQgPSAwLjQ1LAopIC0+IGxpc3RbZGljdFtzdHIsIEFueV1dOgogICAgcmVwb19yb290ID0gUGF0aChyZXBvX3Jvb3QpLnJlc29sdmUoKQogICAgcnVuX3Jvb3QgPSBQYXRoKHJ1bl9yb290KS5yZXNvbHZlKCkKICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoX3Jlc29sdmVfZGF0YV9yb290KGRhdGFfcm9vdCwgcmVwb19yb290KSkKICAgIHZhbGlkYXRpb25fY29uZmlnID0ganNvbi5sb2FkcygKICAgICAgICAocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dCgKICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IgogICAgICAgICkKICAgICkKICAgIHNwbGl0ID0gY3JlYXRlX29ic2VydmF0aW9uX3NwbGl0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIsCiAgICAgICAgc2VlZD1pbnQodmFsaWRhdGlvbl9jb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdCh2YWxpZGF0aW9uX2NvbmZpZ1sidmFsaWRhdGlvbl9mcmFjdGlvbiJdKSwKICAgICkKICAgIGlmIHNwbGl0LnNoYTI1NiAhPSB2YWxpZGF0aW9uX2NvbmZpZ1siZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il06CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgc3RydWN0dXJlIHN3ZWVwIHJlcXVpcmVzIHRoZSBmcm96ZW4gdmFsaWRhdGlvbiBzcGxpdCIpCgogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIHByb2JhYmlsaXR5X2RpciA9IHJ1bl9yb290IC8gInByb2JhYmlsaXRpZXMiCiAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgY2FjaGVfcGF0aCA9IHByb2JhYmlsaXR5X2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiCiAgICAgICAgaWYgbm90IGNhY2hlX3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgcHJvYmFiaWxpdHkgY2FjaGU6IHtjYWNoZV9wYXRofSIpCgogICAgd2VpZ2h0cyA9IGJvdW5kYXJ5X3dlaWdodHMgb3IgKGJvdW5kYXJ5X3dlaWdodCwpCiAgICBjb25maWd1cmF0aW9ucyA9IFsKICAgICAgICBJbnN0YW5jZUNvbmZpZygKICAgICAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9Zm9yZWdyb3VuZF90aHJlc2hvbGQsCiAgICAgICAgICAgIGJvdW5kYXJ5X3RocmVzaG9sZD1ib3VuZGFyeV90aHJlc2hvbGQsCiAgICAgICAgICAgIGJvdW5kYXJ5X3dlaWdodD13ZWlnaHQsCiAgICAgICAgICAgIHNlZWRfdGhyZXNob2xkPXNlZWRfdGhyZXNob2xkLAogICAgICAgICAgICBzZWVkX21pbl9hcmVhPXNlZWRfYXJlYSwKICAgICAgICAgICAgbWluX2luc3RhbmNlX2FyZWE9aW5zdGFuY2VfYXJlYSwKICAgICAgICApCiAgICAgICAgZm9yIHdlaWdodCwgKGluc3RhbmNlX2FyZWEsIHNlZWRfYXJlYSkgaW4gaXRlcnRvb2xzLnByb2R1Y3QoCiAgICAgICAgICAgIHdlaWdodHMsCiAgICAgICAgICAgIGl0ZXJ0b29scy5wcm9kdWN0KAogICAgICAgICAgICAgICAgKDUwLCAxMDAsIDIwMCwgMzAwLCA0MDAsIDUwMCwgNjAwLCA3MDAsIDgwMCwgMV8wMDAsIDFfMjAwKSwKICAgICAgICAgICAgICAgICg4LCAzMiwgMTI4KSwKICAgICAgICAgICAgKSwKICAgICAgICApCiAgICBdCiAgICByZXN1bHRzID0gW10KICAgIGJlc3RfcHEgPSAtMS4wCiAgICBmb3IgaW5kZXgsIGNhbmRpZGF0ZSBpbiBlbnVtZXJhdGUoY29uZmlndXJhdGlvbnMsIHN0YXJ0PTEpOgogICAgICAgIHByZWRpY3Rpb25zID0ge30KICAgICAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgICAgIHdpdGggbnAubG9hZChwcm9iYWJpbGl0eV9kaXIgLyBmIntQYXRoKGZpbGVfbmFtZSkuc3RlbX0ubnB6IikgYXMgY2FjaGU6CiAgICAgICAgICAgICAgICBwcmVkaWN0aW9uc1tmaWxlX25hbWVdID0gaW5zdGFuY2VzX2Zyb21fcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgICAgICAgICBjYWNoZVsiZm9yZWdyb3VuZCJdLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgICAgICBjYWNoZVsiYm91bmRhcnkiXS5hc3R5cGUobnAuZmxvYXQzMiksCiAgICAgICAgICAgICAgICAgICAgZmlsZV9uYW1lLAogICAgICAgICAgICAgICAgICAgIGNhbmRpZGF0ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICBtZXRyaWNzLCBvYnNlcnZhdGlvbnMgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgICAgICBkYXRhc2V0LAogICAgICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdCh2YWxpZGF0aW9uX2NvbmZpZ1siaW91X3RocmVzaG9sZCJdKSwKICAgICAgICAgICAgZGlhZ25vc3RpY19vdmVybGFwX3RocmVzaG9sZD1mbG9hdCgKICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0KICAgICAgICAgICAgKSwKICAgICAgICApCiAgICAgICAgZ2xvYmFsX21ldHJpY3MgPSBtZXRyaWNzWyJnbG9iYWwiXQogICAgICAgIHJvdyA9IHsKICAgICAgICAgICAgImNvbmZpZ3VyYXRpb24iOiBjYW5kaWRhdGUuX19kaWN0X18sCiAgICAgICAgICAgICJwcSI6IGdsb2JhbF9tZXRyaWNzWyJwcSJdLAogICAgICAgICAgICAidHAiOiBnbG9iYWxfbWV0cmljc1sidHAiXSwKICAgICAgICAgICAgImZwIjogZ2xvYmFsX21ldHJpY3NbImZwIl0sCiAgICAgICAgICAgICJmbiI6IGdsb2JhbF9tZXRyaWNzWyJmbiJdLAogICAgICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3NfYW5ub3RhdG9yX3ZpZXdzIjogZ2xvYmFsX21ldHJpY3NbCiAgICAgICAgICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3NfYW5ub3RhdG9yX3ZpZXdzIgogICAgICAgICAgICBdLAogICAgICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgfQogICAgICAgIHJlc3VsdHMuYXBwZW5kKHJvdykKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJbe2luZGV4fS97bGVuKGNvbmZpZ3VyYXRpb25zKX1dIG1pbl9hcmVhPXtjYW5kaWRhdGUubWluX2luc3RhbmNlX2FyZWF9ICIKICAgICAgICAgICAgZiJzZWVkX2FyZWE9e2NhbmRpZGF0ZS5zZWVkX21pbl9hcmVhfSBQUT17cm93WydwcSddOi42Zn0gIgogICAgICAgICAgICBmIlRQPXtyb3dbJ3RwJ119IEZQPXtyb3dbJ2ZwJ119IEZOPXtyb3dbJ2ZuJ119IiwKICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICApCiAgICAgICAgaWYgcm93WyJwcSJdID4gYmVzdF9wcToKICAgICAgICAgICAgYmVzdF9wcSA9IHJvd1sicHEiXQogICAgICAgICAgICBiZXN0X2RpciA9IHJ1bl9yb290IC8gImJlc3QtaW5zdGFuY2Utc3RydWN0dXJlIgogICAgICAgICAgICBiZXN0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIChiZXN0X2RpciAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJzcGxpdF9zaGEyNTYiOiBzcGxpdC5zaGEyNTYsCiAgICAgICAgICAgICAgICAgICAgICAgICJjb25maWd1cmF0aW9uIjogY2FuZGlkYXRlLl9fZGljdF9fLAogICAgICAgICAgICAgICAgICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICAgICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICsgIlxuIiwKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKICAgICAgICAgICAgX3dyaXRlX3ByZWRpY3Rpb25zKGJlc3RfZGlyIC8gInByZWRpY3Rpb25zLmNzdiIsIHByZWRpY3Rpb25zKQogICAgICAgICAgICBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzKAogICAgICAgICAgICAgICAgYmVzdF9kaXIgLyAicGVyX29ic2VydmF0aW9uLmNzdiIsIG9ic2VydmF0aW9ucywgcHJlZGljdGlvbnMKICAgICAgICAgICAgKQogICAgICAgICAgICBfd3JpdGVfb3ZlcmxheXMoZGF0YXNldCwgcHJlZGljdGlvbnMsIGJlc3RfZGlyIC8gIm92ZXJsYXlzIiwgMTIpCiAgICAgICAgICAgIHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMoCiAgICAgICAgICAgICAgICBkYXRhc2V0LCBwcmVkaWN0aW9ucywgYmVzdF9kaXIgLyAiY29tcGFyaXNvbi1vdmVybGF5cyIsIDEyCiAgICAgICAgICAgICkKCiAgICByZXN1bHRzLnNvcnQoa2V5PWxhbWJkYSBpdGVtOiBpdGVtWyJwcSJdLCByZXZlcnNlPVRydWUpCiAgICAocnVuX3Jvb3QgLyAiaW5zdGFuY2Utc3RydWN0dXJlLXN3ZWVwLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVzdWx0cywgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHdpdGggKHJ1bl9yb290IC8gImluc3RhbmNlLXN0cnVjdHVyZS1zd2VlcC5jc3YiKS5vcGVuKAogICAgICAgICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIgogICAgKSBhcyBzdHJlYW06CiAgICAgICAgZmllbGRzID0gKAogICAgICAgICAgICAqSW5zdGFuY2VDb25maWcoKS5fX2RpY3RfXy5rZXlzKCksCiAgICAgICAgICAgICJwcSIsCiAgICAgICAgICAgICJ0cCIsCiAgICAgICAgICAgICJmcCIsCiAgICAgICAgICAgICJmbiIsCiAgICAgICAgICAgICJwcmVkaWN0aW9uX2NvdW50X2Fjcm9zc19hbm5vdGF0b3Jfdmlld3MiLAogICAgICAgICkKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihzdHJlYW0sIGZpZWxkbmFtZXM9ZmllbGRzLCBsaW5ldGVybWluYXRvcj0iXG4iKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIHJvdyBpbiByZXN1bHRzOgogICAgICAgICAgICBjb25maWdfdmFsdWVzID0gcm93WyJjb25maWd1cmF0aW9uIl0KICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICoqY29uZmlnX3ZhbHVlcywKICAgICAgICAgICAgICAgICAgICAqKnsKICAgICAgICAgICAgICAgICAgICAgICAgZmllbGQ6IHJvd1tmaWVsZF0KICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGZpZWxkIGluIGZpZWxkc1tsZW4oY29uZmlnX3ZhbHVlcykgOl0KICAgICAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICByZXR1cm4gcmVzdWx0cwo=","src/filament_baseline/submission.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIC5kYXRhIGltcG9ydCBJTUFHRV9TSVpFLCBpbnZlbnRvcnksIGxvYWRfZGF0YXNldApmcm9tIC5pbmZlcmVuY2UgaW1wb3J0IEluc3RhbmNlQ29uZmlnLCBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzLCBwcmVkaWN0X3Byb2JhYmlsaXRpZXMKZnJvbSAubGVhcm5pbmcgaW1wb3J0IEZpbGFtZW50UGF0Y2hEYXRhc2V0LCBUYXJnZXRNb2RlCmZyb20gLm1vZGVsX3J1bm5lciBpbXBvcnQgX2lzX2thZ2dsZV9ydW50aW1lLCBfbG9hZF9tb2RlbCwgX3dpdGhpbgpmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IGRlY29kZV9ybGUsIHJlYWRfZ3JheXNjYWxlCmZyb20gLnJ1bm5lciBpbXBvcnQgX2dpdF9zdGF0dXMsIF9yZXNvbHZlX2RhdGFfcm9vdCwgX3dyaXRlX3ByZWRpY3Rpb25zCmZyb20gLnJ1bnRpbWUgaW1wb3J0IGhhcmR3YXJlX2luZm8sIHBhY2thZ2VfdmVyc2lvbnMKZnJvbSAudHJhaW5pbmcgaW1wb3J0IHRyYWluX21vZGVsCgoKZGVmIF9jaGVja19zdWJtaXNzaW9uKAogICAgb3V0cHV0X3BhdGg6IFBhdGgsCiAgICBkYXRhc2V0X3Jvb3Q6IFBhdGgsCiAgICB0ZXN0X25hbWVzOiBsaXN0W3N0cl0sCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RdLAopIC0+IGRpY3Rbc3RyLCBpbnRdOgogICAgZXhwZWN0ZWRfY29sdW1ucyA9IFsiZmlsYW1lbnRfaWQiLCAic2VnbWVudGF0aW9uX3JsZSJdCiAgICB3aXRoIG91dHB1dF9wYXRoLm9wZW4oInIiLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgcmVhZGVyID0gY3N2LkRpY3RSZWFkZXIoc3RyZWFtKQogICAgICAgIGlmIHJlYWRlci5maWVsZG5hbWVzICE9IGV4cGVjdGVkX2NvbHVtbnM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmV4cGVjdGVkIHN1Ym1pc3Npb24gY29sdW1uczoge3JlYWRlci5maWVsZG5hbWVzfSIpCiAgICAgICAgcm93cyA9IGxpc3QocmVhZGVyKQoKICAgIGV4cGVjdGVkX2lkcyA9IHsKICAgICAgICBpbnN0YW5jZS5maWxhbWVudF9pZAogICAgICAgIGZvciBmaWxlX25hbWUgaW4gdGVzdF9uYW1lcwogICAgICAgIGZvciBpbnN0YW5jZSBpbiBwcmVkaWN0aW9uc1tmaWxlX25hbWVdCiAgICB9CiAgICByb3dfaWRzID0gW3Jvd1siZmlsYW1lbnRfaWQiXSBmb3Igcm93IGluIHJvd3NdCiAgICBpZiBsZW4ocm93X2lkcykgIT0gbGVuKHNldChyb3dfaWRzKSkgb3Igc2V0KHJvd19pZHMpICE9IGV4cGVjdGVkX2lkczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTdWJtaXNzaW9uIGZpbGFtZW50IElEcyBhcmUgZHVwbGljYXRlZCBvciBkaWZmZXIgZnJvbSBwcmVkaWN0aW9ucyIpCiAgICBpZiBhbnkobm90IHJvd1sic2VnbWVudGF0aW9uX3JsZSJdIGZvciByb3cgaW4gcm93cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3VibWlzc2lvbiBjb250YWlucyBhbiBlbXB0eSBSTEUgc3RyaW5nIikKCiAgICBpbnN0YW5jZV9jb3VudCA9IDAKICAgIGZvciBmaWxlX25hbWUgaW4gdGVzdF9uYW1lczoKICAgICAgICBpbWFnZSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXRfcm9vdCAvICJ0ZXN0IiAvICJ0ZXN0X2ltYWdlcyIgLyBmaWxlX25hbWUpCiAgICAgICAgaWYgaW1hZ2Uuc2hhcGUgIT0gKElNQUdFX1NJWkUsIElNQUdFX1NJWkUpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCB0ZXN0IGltYWdlIHNpemUgZm9yIHtmaWxlX25hbWV9OiB7aW1hZ2Uuc2hhcGV9IikKICAgICAgICBvY2N1cGllZCA9IG5wLnplcm9zKChJTUFHRV9TSVpFLCBJTUFHRV9TSVpFKSwgZHR5cGU9Ym9vbCkKICAgICAgICBzZWVuX2Zvcl9pbWFnZSA9IHNldCgpCiAgICAgICAgZm9yIGluc3RhbmNlIGluIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV06CiAgICAgICAgICAgIGlmIG5vdCBpbnN0YW5jZS5maWxhbWVudF9pZC5zdGFydHN3aXRoKGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfV8iKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJGaWxhbWVudCBJRCBkb2VzIG5vdCBiZWxvbmcgdG8ge2ZpbGVfbmFtZX0iKQogICAgICAgICAgICBpZiBpbnN0YW5jZS5maWxhbWVudF9pZCBpbiBzZWVuX2Zvcl9pbWFnZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJEdXBsaWNhdGUgZmlsYW1lbnQgSUQgaW4ge2ZpbGVfbmFtZX0iKQogICAgICAgICAgICBzZWVuX2Zvcl9pbWFnZS5hZGQoaW5zdGFuY2UuZmlsYW1lbnRfaWQpCiAgICAgICAgICAgIG1hc2sgPSBkZWNvZGVfcmxlKGluc3RhbmNlLnJsZSkKICAgICAgICAgICAgaWYgbWFzay5zaGFwZSAhPSAoSU1BR0VfU0laRSwgSU1BR0VfU0laRSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBtYXNrIGRpbWVuc2lvbnMgZm9yIHtpbnN0YW5jZS5maWxhbWVudF9pZH0iKQogICAgICAgICAgICBpZiBpbnQobWFzay5zdW0oKSkgIT0gaW5zdGFuY2UuYXJlYToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJSTEUgYXJlYSBtaXNtYXRjaCBmb3Ige2luc3RhbmNlLmZpbGFtZW50X2lkfSIpCiAgICAgICAgICAgIGlmIG5wLmFueShvY2N1cGllZCAmIG1hc2suYXN0eXBlKGJvb2wpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJQcmVkaWN0ZWQgaW5zdGFuY2VzIG92ZXJsYXAgaW4ge2ZpbGVfbmFtZX0iKQogICAgICAgICAgICBvY2N1cGllZCB8PSBtYXNrLmFzdHlwZShib29sKQogICAgICAgICAgICBpbnN0YW5jZV9jb3VudCArPSAxCgogICAgaWYgbGVuKHJvd3MpICE9IGluc3RhbmNlX2NvdW50OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN1Ym1pc3Npb24gcm93IGNvdW50IGRvZXMgbm90IG1hdGNoIHZhbGlkYXRlZCB0ZXN0IHByZWRpY3Rpb25zIikKICAgIHJldHVybiB7InRlc3RfaW1hZ2VzIjogbGVuKHRlc3RfbmFtZXMpLCAicHJlZGljdGVkX2luc3RhbmNlcyI6IGluc3RhbmNlX2NvdW50fQoKCmRlZiBydW5fZmluYWxfc3VibWlzc2lvbigKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lLAogICAgcnVuX25hbWU6IHN0ciwKICAgIHRhcmdldF9tb2RlOiBUYXJnZXRNb2RlID0gImNvbnNlbnN1cyIsCiAgICBzdGVwczogaW50ID0gNV8wMDAsCiAgICBiYXRjaF9zaXplOiBpbnQgPSAyLAogICAgcGF0Y2hfc2l6ZTogaW50ID0gNTEyLAogICAgdHRhOiBpbnQgPSAxLAogICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ6IGZsb2F0ID0gMC40NSwKICAgIGJvdW5kYXJ5X3RocmVzaG9sZDogZmxvYXQgPSAwLjY1LAogICAgYm91bmRhcnlfd2VpZ2h0OiBmbG9hdCA9IDEuNSwKICAgIHNlZWRfdGhyZXNob2xkOiBmbG9hdCA9IDAuNDUsCiAgICBzZWVkX21pbl9hcmVhOiBpbnQgPSA4LAogICAgbWluX2luc3RhbmNlX2FyZWE6IGludCA9IDUwLAopIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyIltBLVphLXowLTldW0EtWmEtejAtOS5fLV17MCw2M30iLCBydW5fbmFtZSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicnVuX25hbWUgbXVzdCB1c2Ugb25seSBsZXR0ZXJzLCBkaWdpdHMsIGRvdCwgdW5kZXJzY29yZSwgb3IgZGFzaCIpCiAgICByZXBvX3Jvb3QgPSBQYXRoKHJlcG9fcm9vdCkucmVzb2x2ZSgpCiAgICBrYWdnbGVfcnVudGltZSA9IF9pc19rYWdnbGVfcnVudGltZSgpCiAgICBkYXRhc2V0X3Jvb3QgPSBfcmVzb2x2ZV9kYXRhX3Jvb3QoZGF0YV9yb290LCByZXBvX3Jvb3QpCiAgICBpZiBub3Qga2FnZ2xlX3J1bnRpbWUgYW5kIG5vdCBfd2l0aGluKGRhdGFzZXRfcm9vdCwgcmVwb19yb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgY29tcGV0aXRpb24gZGF0YSBtdXN0IHN0YXkgaW4gdGhlIHJlcG9zaXRvcnk6IHtkYXRhc2V0X3Jvb3R9IikKICAgIHJ1bl9yb290ID0gKAogICAgICAgIFBhdGgoIi9rYWdnbGUvd29ya2luZyIpIC8gImVxdWluZXh1cyIgLyBydW5fbmFtZQogICAgICAgIGlmIGthZ2dsZV9ydW50aW1lCiAgICAgICAgZWxzZSByZXBvX3Jvb3QgLyAiYXJ0aWZhY3RzIiAvICJzdWJtaXNzaW9ucyIgLyBydW5fbmFtZQogICAgKQogICAgaWYgbm90IGthZ2dsZV9ydW50aW1lIGFuZCBub3QgX3dpdGhpbihydW5fcm9vdCwgcmVwb19yb290KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgb3V0cHV0cyBtdXN0IHN0YXkgaW4gdGhlIHJlcG9zaXRvcnk6IHtydW5fcm9vdH0iKQogICAgcnVuX3Jvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmVwb19pbmZvID0gX2dpdF9zdGF0dXMocmVwb19yb290KQoKICAgIGNvbmZpZ19qc29uID0ganNvbi5sb2FkcygKICAgICAgICAocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgKQogICAgY29uZmlnX3NoYTI1NiA9IGhhc2hsaWIuc2hhMjU2KAogICAgICAgIChyZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIikucmVhZF9ieXRlcygpCiAgICApLmhleGRpZ2VzdCgpCiAgICBkYXRhc2V0ID0gbG9hZF9kYXRhc2V0KGRhdGFzZXRfcm9vdCkKICAgIGRhdGFfaW52ZW50b3J5ID0gaW52ZW50b3J5KGRhdGFzZXQpCiAgICBpZiBkYXRhX2ludmVudG9yeVsib2JzZXJ2YXRpb25fY291bnQiXSAhPSA3MDc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRmluYWwgdHJhaW5pbmcgbXVzdCB1c2UgYWxsIDcwNyBvZmZpY2lhbCBjb21wZXRpdGlvbiBvYnNlcnZhdGlvbnMiKQogICAgaWYgZGF0YV9pbnZlbnRvcnlbImFubm90YXRvcl92aWV3X2NvdW50Il0gIT0gMV8xNTQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRmluYWwgdHJhaW5pbmcgbXVzdCB1c2UgYWxsIDEsMTU0IG9mZmljaWFsIGFubm90YXRvciB2aWV3cyIpCiAgICBpZiBkYXRhX2ludmVudG9yeVsiYW5ub3RhdGlvbl9jb3VudCJdICE9IDhfMTk5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgbnVtYmVyIG9mIGNvbXBldGl0aW9uLXByb3ZpZGVkIHRyYWluaW5nIGFubm90YXRpb25zIikKICAgIGlmIGRhdGFfaW52ZW50b3J5WyJ0ZXN0X2ltYWdlX2ZpbGVfY291bnQiXSAhPSAxODA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgMTgwIHN1cHBsaWVkIEgtYWxwaGEgdGVzdCBpbWFnZXMiKQoKICAgIHRyYWluX25hbWVzID0gbGlzdChkYXRhc2V0Lm9ic2VydmF0aW9uX25hbWVzKQogICAgdHJhaW5fZGF0YXNldCA9IEZpbGFtZW50UGF0Y2hEYXRhc2V0KAogICAgICAgIGRhdGFzZXQsCiAgICAgICAgdHJhaW5fbmFtZXMsCiAgICAgICAgdGFyZ2V0X21vZGU9dGFyZ2V0X21vZGUsCiAgICAgICAgcGF0Y2hfc2l6ZT1wYXRjaF9zaXplLAogICAgICAgIGF1Z21lbnQ9VHJ1ZSwKICAgICAgICBzZWVkPWludChjb25maWdfanNvblsic2VlZCJdKSwKICAgICkKICAgIGNoZWNrcG9pbnRfcGF0aCA9IHJ1bl9yb290IC8gImZpbmFsX21vZGVsLnB0IgogICAgc3RhcnRfdGltZSA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHRyYWluaW5nID0gdHJhaW5fbW9kZWwoCiAgICAgICAgdHJhaW5fZGF0YXNldD10cmFpbl9kYXRhc2V0LAogICAgICAgIHZhbGlkYXRpb25fZGF0YXNldD1Ob25lLAogICAgICAgIG91dHB1dF9wYXRoPWNoZWNrcG9pbnRfcGF0aCwKICAgICAgICBzdGVwcz1zdGVwcywKICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnX2pzb25bInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9pbnRlcnZhbD1tYXgoMTAwLCBtaW4oNTAwLCBzdGVwcyAvLyA1KSksCiAgICApCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsID0gX2xvYWRfbW9kZWwoY2hlY2twb2ludF9wYXRoLCBkZXZpY2UpCiAgICBpbnN0YW5jZV9jb25maWcgPSBJbnN0YW5jZUNvbmZpZygKICAgICAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZD1mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV90aHJlc2hvbGQ9Ym91bmRhcnlfdGhyZXNob2xkLAogICAgICAgIGJvdW5kYXJ5X3dlaWdodD1ib3VuZGFyeV93ZWlnaHQsCiAgICAgICAgc2VlZF90aHJlc2hvbGQ9c2VlZF90aHJlc2hvbGQsCiAgICAgICAgc2VlZF9taW5fYXJlYT1zZWVkX21pbl9hcmVhLAogICAgICAgIG1pbl9pbnN0YW5jZV9hcmVhPW1pbl9pbnN0YW5jZV9hcmVhLAogICAgKQogICAgdGVzdF9uYW1lcyA9IHNvcnRlZCgKICAgICAgICBwYXRoLm5hbWUgZm9yIHBhdGggaW4gZGF0YXNldC50ZXN0X2ltYWdlX2Rpci5pdGVyZGlyKCkgaWYgcGF0aC5pc19maWxlKCkKICAgICkKICAgIHByZWRpY3Rpb25zOiBkaWN0W3N0ciwgbGlzdF0gPSB7fQogICAgY292ZXJhZ2Vfcm93cyA9IFtdCiAgICBpbmZlcmVuY2Vfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBmb3IgaW5kZXgsIGZpbGVfbmFtZSBpbiBlbnVtZXJhdGUodGVzdF9uYW1lcywgc3RhcnQ9MSk6CiAgICAgICAgaW1hZ2UgPSByZWFkX2dyYXlzY2FsZShkYXRhc2V0LnRlc3RfaW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIGZvcmVncm91bmQsIGJvdW5kYXJ5ID0gcHJlZGljdF9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICBtb2RlbCwKICAgICAgICAgICAgaW1hZ2UsCiAgICAgICAgICAgIGRldmljZT1kZXZpY2UsCiAgICAgICAgICAgIHBhdGNoX3NpemU9cGF0Y2hfc2l6ZSwKICAgICAgICAgICAgc3RyaWRlPW1heChwYXRjaF9zaXplIC8vIDIsIGludChwYXRjaF9zaXplICogMC43NSkpLAogICAgICAgICAgICB0dGE9dHRhLAogICAgICAgICkKICAgICAgICBwcmVkaWN0aW9uc1tmaWxlX25hbWVdID0gaW5zdGFuY2VzX2Zyb21fcHJvYmFiaWxpdGllcygKICAgICAgICAgICAgZm9yZWdyb3VuZCwgYm91bmRhcnksIGZpbGVfbmFtZSwgaW5zdGFuY2VfY29uZmlnCiAgICAgICAgKQogICAgICAgIGNvdmVyYWdlX3Jvd3MuYXBwZW5kKAogICAgICAgICAgICB7ImZpbGVfbmFtZSI6IGZpbGVfbmFtZSwgImluc3RhbmNlX2NvdW50IjogbGVuKHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0pfQogICAgICAgICkKICAgICAgICBpZiBpbmRleCAlIDEwID09IDAgb3IgaW5kZXggPT0gbGVuKHRlc3RfbmFtZXMpOgogICAgICAgICAgICBwcmludChmIlByZWRpY3RlZCB7aW5kZXh9L3tsZW4odGVzdF9uYW1lcyl9IHRlc3QgaW1hZ2VzIiwgZmx1c2g9VHJ1ZSkKICAgIGluZmVyZW5jZV9zZWNvbmRzID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyZW5jZV9zdGFydAoKICAgIHN1Ym1pc3Npb25fcGF0aCA9IHJ1bl9yb290IC8gInN1Ym1pc3Npb24uY3N2IgogICAgX3dyaXRlX3ByZWRpY3Rpb25zKHN1Ym1pc3Npb25fcGF0aCwgcHJlZGljdGlvbnMpCiAgICB2YWxpZGF0aW9uID0gX2NoZWNrX3N1Ym1pc3Npb24oCiAgICAgICAgc3VibWlzc2lvbl9wYXRoLAogICAgICAgIGRhdGFzZXQucm9vdCwKICAgICAgICB0ZXN0X25hbWVzLAogICAgICAgIHByZWRpY3Rpb25zLAogICAgKQogICAgY292ZXJhZ2VfcGF0aCA9IHJ1bl9yb290IC8gInRlc3RfY292ZXJhZ2UuY3N2IgogICAgd2l0aCBjb3ZlcmFnZV9wYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoCiAgICAgICAgICAgIHN0cmVhbSwgZmllbGRuYW1lcz0oImZpbGVfbmFtZSIsICJpbnN0YW5jZV9jb3VudCIpLCBsaW5ldGVybWluYXRvcj0iXG4iCiAgICAgICAgKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhjb3ZlcmFnZV9yb3dzKQoKICAgIGlmIG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCIpOgogICAgICAgIHJlcG9faW5mb1siY29tbWl0Il0gPSBvcy5lbnZpcm9uWyJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCJdCiAgICAgICAgcmVwb19pbmZvWyJ3b3JrdHJlZV9kaXJ0eSJdID0gVHJ1ZQogICAgcmVwb19pbmZvWyJzb3VyY2VfYnVuZGxlX3NoYTI1NiJdID0gb3MuZW52aXJvbi5nZXQoCiAgICAgICAgIkVRVUlORVhVU19TT1VSQ0VfQlVORExFX1NIQTI1NiIKICAgICkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic3RhdHVzIjogImNvbXBsZXRlIiwKICAgICAgICAiY3JlYXRlZF91dGMiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZFQlSDolTTolU1oiLCB0aW1lLmdtdGltZSgpKSwKICAgICAgICAicnVuX25hbWUiOiBydW5fbmFtZSwKICAgICAgICAicmVwb3NpdG9yeSI6IHJlcG9faW5mbywKICAgICAgICAiY29uZmlndXJhdGlvbl9zaGEyNTYiOiBjb25maWdfc2hhMjU2LAogICAgICAgICJrYWdnbGVfbm90ZWJvb2tfdmVyc2lvbiI6IG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfTk9URUJPT0tfVkVSU0lPTiIpLAogICAgICAgICJkYXRhX2JvdW5kYXJ5IjogewogICAgICAgICAgICAidHJhaW5pbmdfbGFiZWxzIjogImNvbXBldGl0aW9uLXByb3ZpZGVkIHRyYWluIGFubm90YXRpb25zIG9ubHkiLAogICAgICAgICAgICAiaW5mZXJlbmNlX2lucHV0cyI6ICJhdHRhY2hlZCBjb21wZXRpdGlvbiBILWFscGhhIHRlc3QgaW1hZ2VzIG9ubHkiLAogICAgICAgICAgICAiZXh0ZXJuYWxfZ3JvdW5kX3RydXRoIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICAiZGF0YXNldF9pbnZlbnRvcnkiOiBkYXRhX2ludmVudG9yeSwKICAgICAgICAidmFsaWRhdGlvbl9yZWZlcmVuY2UiOiB7CiAgICAgICAgICAgICJzcGxpdF9zaGEyNTYiOiBjb25maWdfanNvblsiZXhwZWN0ZWRfc3BsaXRfc2hhMjU2Il0sCiAgICAgICAgICAgICJzZWxlY3Rpb25fbm90ZSI6ICJQb3N0cHJvY2Vzc2luZyBzZWxlY3RlZCBvbiB0aGUgZnJvemVuIG9ic2VydmF0aW9uLWdyb3VwZWQgaG9sZG91dCBiZWZvcmUgcmVmaXR0aW5nIG9uIGFsbCBvZmZpY2lhbCB0cmFpbmluZyBvYnNlcnZhdGlvbnMuIiwKICAgICAgICB9LAogICAgICAgICJ0cmFpbmluZyI6IHsqKnRyYWluaW5nLCAidHJhaW5pbmdfb2JzZXJ2YXRpb25zIjogbGVuKHRyYWluX25hbWVzKX0sCiAgICAgICAgIm1ldGhvZCI6IHsKICAgICAgICAgICAgImFyY2hpdGVjdHVyZSI6ICJmcm9tX3NjcmF0Y2hfZml2ZV9sZXZlbF91bmV0X2ZvcmVncm91bmRfYm91bmRhcnkiLAogICAgICAgICAgICAidGFyZ2V0X21vZGUiOiB0YXJnZXRfbW9kZSwKICAgICAgICAgICAgInBhdGNoX3NpemUiOiBwYXRjaF9zaXplLAogICAgICAgICAgICAic3RlcHMiOiBzdGVwcywKICAgICAgICAgICAgImJhdGNoX3NpemUiOiBiYXRjaF9zaXplLAogICAgICAgICAgICAidHRhIjogdHRhLAogICAgICAgICAgICAiaW5zdGFuY2VfY29uZmlnIjogaW5zdGFuY2VfY29uZmlnLl9fZGljdF9fLAogICAgICAgICAgICAiZXh0ZXJuYWxfc3VwZXJ2aXNlZF93ZWlnaHRzIjogRmFsc2UsCiAgICAgICAgfSwKICAgICAgICAidGVzdF92YWxpZGF0aW9uIjogdmFsaWRhdGlvbiwKICAgICAgICAicnVudGltZSI6IHsKICAgICAgICAgICAgKipoYXJkd2FyZV9pbmZvKCksCiAgICAgICAgICAgICJpbmZlcmVuY2Vfc2Vjb25kcyI6IGluZmVyZW5jZV9zZWNvbmRzLAogICAgICAgICAgICAiZW5kX3RvX2VuZF9zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0X3RpbWUsCiAgICAgICAgICAgICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpLAogICAgICAgIH0sCiAgICAgICAgImFydGlmYWN0cyI6IHsKICAgICAgICAgICAgImNoZWNrcG9pbnQiOiBjaGVja3BvaW50X3BhdGgubmFtZSwKICAgICAgICAgICAgInRyYWluaW5nX2hpc3RvcnkiOiBjaGVja3BvaW50X3BhdGgud2l0aF9zdWZmaXgoIi5qc29ubCIpLm5hbWUsCiAgICAgICAgICAgICJzdWJtaXNzaW9uIjogc3VibWlzc2lvbl9wYXRoLm5hbWUsCiAgICAgICAgICAgICJjb3ZlcmFnZSI6IGNvdmVyYWdlX3BhdGgubmFtZSwKICAgICAgICB9LAogICAgfQogICAgKHJ1bl9yb290IC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludCgKICAgICAgICBmIlZhbGlkYXRlZCBzdWJtaXNzaW9uIHdpdGgge3ZhbGlkYXRpb25bJ3ByZWRpY3RlZF9pbnN0YW5jZXMnXX0gaW5zdGFuY2VzICIKICAgICAgICBmImZvciB7dmFsaWRhdGlvblsndGVzdF9pbWFnZXMnXX0gaW1hZ2VzOiB7c3VibWlzc2lvbl9wYXRofSIsCiAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICkKICAgIHJldHVybiByZXN1bHQK","src/filament_baseline/sweep.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmRhdGEgaW1wb3J0IGxvYWRfZGF0YXNldApmcm9tIC5pbmZlcmVuY2UgaW1wb3J0IEluc3RhbmNlQ29uZmlnLCBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzCmZyb20gLnJ1bm5lciBpbXBvcnQgX3Jlc29sdmVfZGF0YV9yb290LCBfd3JpdGVfb2JzZXJ2YXRpb25fc2NvcmVzLCBfd3JpdGVfb3ZlcmxheXMsIF93cml0ZV9wcmVkaWN0aW9ucwpmcm9tIC5zY29yZXIgaW1wb3J0IHNjb3JlX3ZhbGlkYXRpb24KZnJvbSAuc3BsaXQgaW1wb3J0IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdApmcm9tIC52aXN1YWxzIGltcG9ydCB3cml0ZV9jb21wYXJpc29uX292ZXJsYXlzCgoKZGVmIHJ1bl9wb3N0cHJvY2Vzc2luZ19zd2VlcCgKICAgICosCiAgICByZXBvX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lLAogICAgcnVuX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBvYnNlcnZhdGlvbl9saW1pdDogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtkaWN0W3N0ciwgQW55XV06CiAgICByZXBvX3Jvb3QgPSBQYXRoKHJlcG9fcm9vdCkucmVzb2x2ZSgpCiAgICBydW5fcm9vdCA9IFBhdGgocnVuX3Jvb3QpLnJlc29sdmUoKQogICAgZGF0YXNldCA9IGxvYWRfZGF0YXNldChfcmVzb2x2ZV9kYXRhX3Jvb3QoZGF0YV9yb290LCByZXBvX3Jvb3QpKQogICAgY29uZmlnID0ganNvbi5sb2FkcygocmVwb19yb290IC8gImNvbmZpZ3MiIC8gInZhbGlkYXRpb24tYmFzZWxpbmUuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgc3BsaXQgPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgZGF0YXNldCwKICAgICAgICByZXBvX3Jvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBzcGxpdC5zaGEyNTYgIT0gY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBwb3N0cHJvY2Vzc2luZyBzd2VlcCByZXF1aXJlcyB0aGUgZnJvemVuIHZhbGlkYXRpb24gc3BsaXQiKQogICAgdmFsaWRhdGlvbl9uYW1lcyA9IGxpc3Qoc3BsaXQudmFsaWRhdGlvbl9uYW1lcykKICAgIGlmIG9ic2VydmF0aW9uX2xpbWl0IGlzIG5vdCBOb25lOgogICAgICAgIHZhbGlkYXRpb25fbmFtZXMgPSB2YWxpZGF0aW9uX25hbWVzWzpvYnNlcnZhdGlvbl9saW1pdF0KICAgIGNhY2hlX2RpciA9IHJ1bl9yb290IC8gInByb2JhYmlsaXRpZXMiCiAgICBmb3IgZmlsZV9uYW1lIGluIHZhbGlkYXRpb25fbmFtZXM6CiAgICAgICAgY2FjaGVfcGF0aCA9IGNhY2hlX2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiCiAgICAgICAgaWYgbm90IGNhY2hlX3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgY2FjaGVkIHZhbGlkYXRpb24gcHJvYmFiaWxpdHkgbWFwOiB7Y2FjaGVfcGF0aH0iKQoKICAgIGNvbmZpZ3VyYXRpb25zID0gWwogICAgICAgIEluc3RhbmNlQ29uZmlnKAogICAgICAgICAgICBmb3JlZ3JvdW5kX3RocmVzaG9sZD1mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICAgICAgYm91bmRhcnlfdGhyZXNob2xkPWJvdW5kYXJ5X3RocmVzaG9sZCwKICAgICAgICAgICAgYm91bmRhcnlfd2VpZ2h0PTEuNSwKICAgICAgICAgICAgbWluX2luc3RhbmNlX2FyZWE9NTAsCiAgICAgICAgKQogICAgICAgIGZvciBmb3JlZ3JvdW5kX3RocmVzaG9sZCwgYm91bmRhcnlfdGhyZXNob2xkIGluIGl0ZXJ0b29scy5wcm9kdWN0KAogICAgICAgICAgICAoMC4yNSwgMC4zNSwgMC40NSwgMC41NSwgMC42NSksCiAgICAgICAgICAgICgwLjQ1LCAwLjUwLCAwLjU1LCAwLjYwLCAwLjY1LCAwLjcwLCAwLjc1LCAwLjgwLCAwLjkwKSwKICAgICAgICApCiAgICBdCiAgICByZXN1bHRzID0gW10KICAgIGZvciBpbmRleCwgaW5zdGFuY2VfY29uZmlnIGluIGVudW1lcmF0ZShjb25maWd1cmF0aW9ucywgc3RhcnQ9MSk6CiAgICAgICAgcHJlZGljdGlvbnMgPSB7fQogICAgICAgIGZvciBmaWxlX25hbWUgaW4gdmFsaWRhdGlvbl9uYW1lczoKICAgICAgICAgICAgY2FjaGVfcGF0aCA9IGNhY2hlX2RpciAvIGYie1BhdGgoZmlsZV9uYW1lKS5zdGVtfS5ucHoiCiAgICAgICAgICAgIHdpdGggbnAubG9hZChjYWNoZV9wYXRoKSBhcyBjYWNoZToKICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV0gPSBpbnN0YW5jZXNfZnJvbV9wcm9iYWJpbGl0aWVzKAogICAgICAgICAgICAgICAgICAgIGNhY2hlWyJmb3JlZ3JvdW5kIl0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAgICAgICAgIGNhY2hlWyJib3VuZGFyeSJdLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgICAgICBmaWxlX25hbWUsCiAgICAgICAgICAgICAgICAgICAgaW5zdGFuY2VfY29uZmlnLAogICAgICAgICAgICAgICAgKQogICAgICAgIG1ldHJpY3MsIHJvd3MgPSBzY29yZV92YWxpZGF0aW9uKAogICAgICAgICAgICBkYXRhc2V0LAogICAgICAgICAgICBwcmVkaWN0aW9ucywKICAgICAgICAgICAgdmFsaWRhdGlvbl9uYW1lcywKICAgICAgICAgICAgaW91X3RocmVzaG9sZD1mbG9hdChjb25maWdbImlvdV90aHJlc2hvbGQiXSksCiAgICAgICAgICAgIGRpYWdub3N0aWNfb3ZlcmxhcF90aHJlc2hvbGQ9ZmxvYXQoY29uZmlnWyJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIl0pLAogICAgICAgICkKICAgICAgICBnbG9iYWxfbWV0cmljcyA9IG1ldHJpY3NbImdsb2JhbCJdCiAgICAgICAgcm93ID0gewogICAgICAgICAgICAiY29uZmlndXJhdGlvbiI6IGluc3RhbmNlX2NvbmZpZy5fX2RpY3RfXywKICAgICAgICAgICAgInBxIjogZ2xvYmFsX21ldHJpY3NbInBxIl0sCiAgICAgICAgICAgICJ0cCI6IGdsb2JhbF9tZXRyaWNzWyJ0cCJdLAogICAgICAgICAgICAiZnAiOiBnbG9iYWxfbWV0cmljc1siZnAiXSwKICAgICAgICAgICAgImZuIjogZ2xvYmFsX21ldHJpY3NbImZuIl0sCiAgICAgICAgICAgICJwcmVkaWN0aW9uX2NvdW50X2Fjcm9zc192aWV3cyI6IGdsb2JhbF9tZXRyaWNzWwogICAgICAgICAgICAgICAgInByZWRpY3Rpb25fY291bnRfYWNyb3NzX2Fubm90YXRvcl92aWV3cyIKICAgICAgICAgICAgXSwKICAgICAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLAogICAgICAgIH0KICAgICAgICByZXN1bHRzLmFwcGVuZChyb3cpCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiW3tpbmRleH0ve2xlbihjb25maWd1cmF0aW9ucyl9XSBmZz17aW5zdGFuY2VfY29uZmlnLmZvcmVncm91bmRfdGhyZXNob2xkOi4yZn0gIgogICAgICAgICAgICBmImJvdW5kYXJ5PXtpbnN0YW5jZV9jb25maWcuYm91bmRhcnlfdGhyZXNob2xkOi4yZn0gIgogICAgICAgICAgICBmIlBRPXtnbG9iYWxfbWV0cmljc1sncHEnXTouNmZ9IFRQPXtnbG9iYWxfbWV0cmljc1sndHAnXX0gIgogICAgICAgICAgICBmIkZQPXtnbG9iYWxfbWV0cmljc1snZnAnXX0gRk49e2dsb2JhbF9tZXRyaWNzWydmbiddfSIsCiAgICAgICAgICAgIGZsdXNoPVRydWUsCiAgICAgICAgKQoKICAgICAgICBpZiBpbmRleCA9PSAxIG9yIHJvd1sicHEiXSA+IG1heChpdGVtWyJwcSJdIGZvciBpdGVtIGluIHJlc3VsdHNbOi0xXSk6CiAgICAgICAgICAgIGJlc3RfZGlyID0gcnVuX3Jvb3QgLyAiYmVzdC1wb3N0cHJvY2Vzc2luZyIKICAgICAgICAgICAgYmVzdF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICAoYmVzdF9kaXIgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgIGpzb24uZHVtcHMoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAic3BsaXRfc2hhMjU2Ijogc3BsaXQuc2hhMjU2LAogICAgICAgICAgICAgICAgICAgICAgICAiY29uZmlndXJhdGlvbiI6IGluc3RhbmNlX2NvbmZpZy5fX2RpY3RfXywKICAgICAgICAgICAgICAgICAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLAogICAgICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgICAgICAgICAgaW5kZW50PTIsCiAgICAgICAgICAgICAgICAgICAgc29ydF9rZXlzPVRydWUsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICArICJcbiIsCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIF93cml0ZV9wcmVkaWN0aW9ucyhiZXN0X2RpciAvICJwcmVkaWN0aW9ucy5jc3YiLCBwcmVkaWN0aW9ucykKICAgICAgICAgICAgX3dyaXRlX29ic2VydmF0aW9uX3Njb3JlcyhiZXN0X2RpciAvICJwZXJfb2JzZXJ2YXRpb24uY3N2Iiwgcm93cywgcHJlZGljdGlvbnMpCiAgICAgICAgICAgIF93cml0ZV9vdmVybGF5cyhkYXRhc2V0LCBwcmVkaWN0aW9ucywgYmVzdF9kaXIgLyAib3ZlcmxheXMiLCAxMikKICAgICAgICAgICAgd3JpdGVfY29tcGFyaXNvbl9vdmVybGF5cygKICAgICAgICAgICAgICAgIGRhdGFzZXQsIHByZWRpY3Rpb25zLCBiZXN0X2RpciAvICJjb21wYXJpc29uLW92ZXJsYXlzIiwgMTIKICAgICAgICAgICAgKQoKICAgIHJlc3VsdF9wYXRoID0gcnVuX3Jvb3QgLyAicG9zdHByb2Nlc3Npbmdfc3dlZXAuanNvbiIKICAgIHJlc3VsdF9wYXRoLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHRzLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IgogICAgKQogICAgd2l0aCAocnVuX3Jvb3QgLyAicG9zdHByb2Nlc3Npbmdfc3dlZXAuY3N2Iikub3BlbigKICAgICAgICAidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIKICAgICkgYXMgc3RyZWFtOgogICAgICAgIGZpZWxkcyA9ICgKICAgICAgICAgICAgKkluc3RhbmNlQ29uZmlnKCkuX19kaWN0X18ua2V5cygpLAogICAgICAgICAgICAicHEiLAogICAgICAgICAgICAidHAiLAogICAgICAgICAgICAiZnAiLAogICAgICAgICAgICAiZm4iLAogICAgICAgICAgICAicHJlZGljdGlvbl9jb3VudF9hY3Jvc3Nfdmlld3MiLAogICAgICAgICkKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihzdHJlYW0sIGZpZWxkbmFtZXM9ZmllbGRzLCBsaW5ldGVybWluYXRvcj0iXG4iKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIHJvdyBpbiByZXN1bHRzOgogICAgICAgICAgICB3cml0ZXIud3JpdGVyb3coCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgKipyb3dbImNvbmZpZ3VyYXRpb24iXSwKICAgICAgICAgICAgICAgICAgICAqKntrZXk6IHJvd1trZXldIGZvciBrZXkgaW4gZmllbGRzW2xlbihJbnN0YW5jZUNvbmZpZygpLl9fZGljdF9fKSA6XX0sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIHJldHVybiBzb3J0ZWQocmVzdWx0cywga2V5PWxhbWJkYSBpdGVtOiBpdGVtWyJwcSJdLCByZXZlcnNlPVRydWUpCg==","src/filament_baseline/training.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKCmZyb20gLmxlYXJuaW5nIGltcG9ydCBGaWxhbWVudFBhdGNoRGF0YXNldCwgVGFyZ2V0TW9kZSwgdHJhaW5pbmdfbG9zcwpmcm9tIC5tb2RlbCBpbXBvcnQgRmlsYW1lbnRVTmV0CgoKZGVmIHNlZWRfZXZlcnl0aGluZyhzZWVkOiBpbnQpIC0+IE5vbmU6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gVHJ1ZQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGV2YWx1YXRlX3BhdGNoX2RpY2UoCiAgICBtb2RlbDogRmlsYW1lbnRVTmV0LAogICAgbG9hZGVyOiBEYXRhTG9hZGVyLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UsCiAgICAqLAogICAgbWF4X2JhdGNoZXM6IGludCA9IDY0LAopIC0+IGZsb2F0OgogICAgbW9kZWwuZXZhbCgpCiAgICBzY29yZXMgPSBbXQogICAgZm9yIGluZGV4LCAoaW1hZ2VzLCB0YXJnZXRzKSBpbiBlbnVtZXJhdGUobG9hZGVyKToKICAgICAgICBpZiBpbmRleCA+PSBtYXhfYmF0Y2hlczoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpbWFnZXMgPSBpbWFnZXMudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSkKICAgICAgICB0YXJnZXRzID0gdGFyZ2V0cy50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoCiAgICAgICAgICAgIGRldmljZV90eXBlPWRldmljZS50eXBlLAogICAgICAgICAgICBkdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgICAgICBlbmFibGVkPWRldmljZS50eXBlID09ICJjdWRhIiwKICAgICAgICApOgogICAgICAgICAgICBsb2dpdHMgPSBtb2RlbChpbWFnZXMpCiAgICAgICAgcHJvYmFiaWxpdGllcyA9IGxvZ2l0c1s6LCAwOjFdLmZsb2F0KCkuc2lnbW9pZCgpCiAgICAgICAgZXhwZWN0ZWQgPSB0YXJnZXRzWzosIDA6MV0KICAgICAgICBpbnRlcnNlY3Rpb24gPSAocHJvYmFiaWxpdGllcyAqIGV4cGVjdGVkKS5zdW0oZGltPSgxLCAyLCAzKSkKICAgICAgICBkZW5vbWluYXRvciA9IHByb2JhYmlsaXRpZXMuc3VtKGRpbT0oMSwgMiwgMykpICsgZXhwZWN0ZWQuc3VtKGRpbT0oMSwgMiwgMykpCiAgICAgICAgc2NvcmVzLmV4dGVuZCgoKDIgKiBpbnRlcnNlY3Rpb24gKyAxKSAvIChkZW5vbWluYXRvciArIDEpKS5jcHUoKS50b2xpc3QoKSkKICAgIHJldHVybiBmbG9hdChucC5tZWFuKHNjb3JlcykpIGlmIHNjb3JlcyBlbHNlIDAuMAoKCmRlZiB0cmFpbl9tb2RlbCgKICAgICosCiAgICB0cmFpbl9kYXRhc2V0OiBGaWxhbWVudFBhdGNoRGF0YXNldCwKICAgIHZhbGlkYXRpb25fZGF0YXNldDogRmlsYW1lbnRQYXRjaERhdGFzZXQgfCBOb25lLAogICAgb3V0cHV0X3BhdGg6IHN0ciB8IFBhdGgsCiAgICBzdGVwczogaW50LAogICAgYmF0Y2hfc2l6ZTogaW50LAogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAzZS00LAogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDFlLTQsCiAgICBzZWVkOiBpbnQgPSA0MiwKICAgIHZhbGlkYXRpb25faW50ZXJ2YWw6IGludCA9IDUwMCwKICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbjogaW50ID0gMSwKKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGlmIHN0ZXBzIDwgMSBvciBiYXRjaF9zaXplIDwgMSBvciBncmFkaWVudF9hY2N1bXVsYXRpb24gPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInN0ZXBzLCBiYXRjaF9zaXplLCBhbmQgZ3JhZGllbnRfYWNjdW11bGF0aW9uIG11c3QgYmUgcG9zaXRpdmUiKQogICAgc2VlZF9ldmVyeXRoaW5nKHNlZWQpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGE6MCIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbW9kZWwgPSBGaWxhbWVudFVOZXQoKS50byhkZXZpY2UpCiAgICBpZiBkZXZpY2UudHlwZSA9PSAiY3VkYSI6CiAgICAgICAgbW9kZWwgPSBtb2RlbC50byhtZW1vcnlfZm9ybWF0PXRvcmNoLmNoYW5uZWxzX2xhc3QpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVygKICAgICAgICBtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWxlYXJuaW5nX3JhdGUsIHdlaWdodF9kZWNheT13ZWlnaHRfZGVjYXkKICAgICkKICAgIHdhcm11cF9zdGVwcyA9IG1pbigzMDAsIG1heCgxLCBzdGVwcyAvLyAxMCkpCgogICAgZGVmIHNjaGVkdWxlKHN0ZXA6IGludCkgLT4gZmxvYXQ6CiAgICAgICAgaWYgc3RlcCA8IHdhcm11cF9zdGVwczoKICAgICAgICAgICAgcmV0dXJuIG1heCgwLjA1LCAoc3RlcCArIDEpIC8gd2FybXVwX3N0ZXBzKQogICAgICAgIHByb2dyZXNzID0gKHN0ZXAgLSB3YXJtdXBfc3RlcHMpIC8gbWF4KDEsIHN0ZXBzIC0gd2FybXVwX3N0ZXBzKQogICAgICAgIHJldHVybiAwLjAxICsgMC45OSAqIDAuNSAqICgxLjAgKyBtYXRoLmNvcyhtYXRoLnBpICogcHJvZ3Jlc3MpKQoKICAgIHNjaGVkdWxlciA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5MYW1iZGFMUihvcHRpbWl6ZXIsIHNjaGVkdWxlKQogICAgc2NhbGVyID0gdG9yY2guYW1wLkdyYWRTY2FsZXIoImN1ZGEiLCBlbmFibGVkPWRldmljZS50eXBlID09ICJjdWRhIikKICAgIHRyYWluX2xvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgdHJhaW5fZGF0YXNldCwKICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgc2h1ZmZsZT1UcnVlLAogICAgICAgIG51bV93b3JrZXJzPTAsCiAgICAgICAgcGluX21lbW9yeT1kZXZpY2UudHlwZSA9PSAiY3VkYSIsCiAgICAgICAgZHJvcF9sYXN0PVRydWUsCiAgICApCiAgICB2YWxpZGF0aW9uX2xvYWRlciA9ICgKICAgICAgICBEYXRhTG9hZGVyKAogICAgICAgICAgICB2YWxpZGF0aW9uX2RhdGFzZXQsCiAgICAgICAgICAgIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwKICAgICAgICAgICAgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgbnVtX3dvcmtlcnM9MCwKICAgICAgICAgICAgcGluX21lbW9yeT1kZXZpY2UudHlwZSA9PSAiY3VkYSIsCiAgICAgICAgKQogICAgICAgIGlmIHZhbGlkYXRpb25fZGF0YXNldCBpcyBub3QgTm9uZQogICAgICAgIGVsc2UgTm9uZQogICAgKQogICAgaWYgbGVuKHRyYWluX2xvYWRlcikgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaGUgdHJhaW5pbmcgcGF0Y2ggbG9hZGVyIHByb2R1Y2VkIG5vIGJhdGNoZXMiKQoKICAgIG91dHB1dF9wYXRoID0gUGF0aChvdXRwdXRfcGF0aCkKICAgIG91dHB1dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBoaXN0b3J5X3BhdGggPSBvdXRwdXRfcGF0aC53aXRoX3N1ZmZpeCgiLmpzb25sIikKICAgIGJlc3RfZGljZSA9IC0xLjAKICAgIGJlc3Rfc3RlcCA9IDAKICAgIHJ1bm5pbmdfbG9zcyA9IDAuMAogICAgbG9nZ2VkX3N0ZXBzID0gMAogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGl0ZXJhdG9yID0gaXRlcih0cmFpbl9sb2FkZXIpCiAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICBpZiBkZXZpY2UudHlwZSA9PSAiY3VkYSI6CiAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cyhkZXZpY2UpCgogICAgd2l0aCBoaXN0b3J5X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhpc3Rvcnk6CiAgICAgICAgZm9yIHN0ZXAgaW4gcmFuZ2UoMSwgc3RlcHMgKyAxKToKICAgICAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBpbWFnZXMsIHRhcmdldHMgPSBuZXh0KGl0ZXJhdG9yKQogICAgICAgICAgICBleGNlcHQgU3RvcEl0ZXJhdGlvbjoKICAgICAgICAgICAgICAgIHRyYWluX2RhdGFzZXQuc2V0X2Vwb2NoKHN0ZXAgLy8gbGVuKHRyYWluX2xvYWRlcikpCiAgICAgICAgICAgICAgICBpdGVyYXRvciA9IGl0ZXIodHJhaW5fbG9hZGVyKQogICAgICAgICAgICAgICAgaW1hZ2VzLCB0YXJnZXRzID0gbmV4dChpdGVyYXRvcikKICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIHRhcmdldHMgPSB0YXJnZXRzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICAgICAgICAgIGltYWdlcyA9IGltYWdlcy5jb250aWd1b3VzKG1lbW9yeV9mb3JtYXQ9dG9yY2guY2hhbm5lbHNfbGFzdCkKICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgKICAgICAgICAgICAgICAgIGRldmljZV90eXBlPWRldmljZS50eXBlLAogICAgICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgICAgIGVuYWJsZWQ9ZGV2aWNlLnR5cGUgPT0gImN1ZGEiLAogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2VzKQogICAgICAgICAgICAgICAgbG9zcyA9IHRyYWluaW5nX2xvc3MobG9naXRzLCB0YXJnZXRzKSAvIGdyYWRpZW50X2FjY3VtdWxhdGlvbgogICAgICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKQogICAgICAgICAgICBpZiBzdGVwICUgZ3JhZGllbnRfYWNjdW11bGF0aW9uID09IDAgb3Igc3RlcCA9PSBzdGVwczoKICAgICAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgICAgIHNjYWxlci51cGRhdGUoKQogICAgICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIHJ1bm5pbmdfbG9zcyArPSBmbG9hdChsb3NzLmRldGFjaCgpKSAqIGdyYWRpZW50X2FjY3VtdWxhdGlvbgogICAgICAgICAgICBsb2dnZWRfc3RlcHMgKz0gMQoKICAgICAgICAgICAgc2hvdWxkX3ZhbGlkYXRlID0gKAogICAgICAgICAgICAgICAgc3RlcCAlIHZhbGlkYXRpb25faW50ZXJ2YWwgPT0gMCBvciBzdGVwID09IHN0ZXBzCiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgc2hvdWxkX3ZhbGlkYXRlOgogICAgICAgICAgICAgICAgcGF0Y2hfZGljZSA9ICgKICAgICAgICAgICAgICAgICAgICBldmFsdWF0ZV9wYXRjaF9kaWNlKG1vZGVsLCB2YWxpZGF0aW9uX2xvYWRlciwgZGV2aWNlLCBtYXhfYmF0Y2hlcz02NCkKICAgICAgICAgICAgICAgICAgICBpZiB2YWxpZGF0aW9uX2xvYWRlciBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgICAgIGVsc2UgTm9uZQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcm93ID0gewogICAgICAgICAgICAgICAgICAgICJzdGVwIjogc3RlcCwKICAgICAgICAgICAgICAgICAgICAibWVhbl90cmFpbl9sb3NzX3NpbmNlX2xvZyI6IHJ1bm5pbmdfbG9zcwogICAgICAgICAgICAgICAgICAgIC8gbWF4KDEsIGxvZ2dlZF9zdGVwcyksCiAgICAgICAgICAgICAgICAgICAgInZhbGlkYXRpb25fcGF0Y2hfZGljZSI6IHBhdGNoX2RpY2UsCiAgICAgICAgICAgICAgICAgICAgImxlYXJuaW5nX3JhdGUiOiBvcHRpbWl6ZXIucGFyYW1fZ3JvdXBzWzBdWyJsciJdLAogICAgICAgICAgICAgICAgICAgICJlbGFwc2VkX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZCwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGhpc3Rvcnkud3JpdGUoanNvbi5kdW1wcyhyb3csIHNvcnRfa2V5cz1UcnVlKSArICJcbiIpCiAgICAgICAgICAgICAgICBoaXN0b3J5LmZsdXNoKCkKICAgICAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMocm93LCBzb3J0X2tleXM9VHJ1ZSksIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICBydW5uaW5nX2xvc3MgPSAwLjAKICAgICAgICAgICAgICAgIGxvZ2dlZF9zdGVwcyA9IDAKICAgICAgICAgICAgICAgIGlmIHBhdGNoX2RpY2UgaXMgTm9uZSBvciBwYXRjaF9kaWNlID4gYmVzdF9kaWNlOgogICAgICAgICAgICAgICAgICAgIGlmIHBhdGNoX2RpY2UgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RfZGljZSA9IHBhdGNoX2RpY2UKICAgICAgICAgICAgICAgICAgICBiZXN0X3N0ZXAgPSBzdGVwCiAgICAgICAgICAgICAgICAgICAgdG9yY2guc2F2ZSgKICAgICAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGVsIjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGVsX2NvbmZpZyI6IHsiYmFzZV9jaGFubmVscyI6IDE2LCAiaW5fY2hhbm5lbHMiOiAxfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0YXJnZXRfbW9kZSI6IHRyYWluX2RhdGFzZXQudGFyZ2V0X21vZGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAic3RlcCI6IHN0ZXAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmFsaWRhdGlvbl9wYXRjaF9kaWNlIjogcGF0Y2hfZGljZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWVkIjogc2VlZCwKICAgICAgICAgICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICAgICAgICAgb3V0cHV0X3BhdGgsCiAgICAgICAgICAgICAgICAgICAgKQoKICAgIHBlYWtfbWVtb3J5ID0gKAogICAgICAgIGludCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGRldmljZSkpIGlmIGRldmljZS50eXBlID09ICJjdWRhIiBlbHNlIDAKICAgICkKICAgIHJldHVybiB7CiAgICAgICAgImNoZWNrcG9pbnQiOiBzdHIob3V0cHV0X3BhdGgpLAogICAgICAgICJoaXN0b3J5Ijogc3RyKGhpc3RvcnlfcGF0aCksCiAgICAgICAgImRldmljZSI6IHN0cihkZXZpY2UpLAogICAgICAgICJzdGVwcyI6IHN0ZXBzLAogICAgICAgICJzZWVkIjogc2VlZCwKICAgICAgICAiYmVzdF9zdGVwIjogYmVzdF9zdGVwLAogICAgICAgICJiZXN0X3ZhbGlkYXRpb25fcGF0Y2hfZGljZSI6IGJlc3RfZGljZSBpZiBiZXN0X2RpY2UgPj0gMC4wIGVsc2UgTm9uZSwKICAgICAgICAic2VsZWN0aW9uX21vZGUiOiAiYmVzdF92YWxpZGF0aW9uX3BhdGNoX2RpY2UiCiAgICAgICAgaWYgdmFsaWRhdGlvbl9sb2FkZXIgaXMgbm90IE5vbmUKICAgICAgICBlbHNlICJmaW5hbF9zdGVwIiwKICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQsCiAgICAgICAgInBlYWtfZ3B1X21lbW9yeV9ieXRlcyI6IHBlYWtfbWVtb3J5LAogICAgICAgICJ0b3JjaF92ZXJzaW9uIjogdG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgImN1ZGFfdmVyc2lvbiI6IHRvcmNoLnZlcnNpb24uY3VkYSwKICAgIH0K","src/filament_baseline/visuals.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucApmcm9tIHB5Y29jb3Rvb2xzIGltcG9ydCBtYXNrIGFzIG1hc2tfdXRpbHMKCmZyb20gLmRhdGEgaW1wb3J0IENvbXBldGl0aW9uRGF0YXNldApmcm9tIC5wcmVkaWN0b3IgaW1wb3J0IFByZWRpY3RlZEluc3RhbmNlLCBkZWNvZGVfcmxlLCByZWFkX2dyYXlzY2FsZQpmcm9tIC5zY29yZXIgaW1wb3J0IGFubm90YXRpb25fdG9fcmxlCgoKZGVmIHdyaXRlX2NvbXBhcmlzb25fb3ZlcmxheXMoCiAgICBkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsCiAgICBwcmVkaWN0aW9uczogZGljdFtzdHIsIGxpc3RbUHJlZGljdGVkSW5zdGFuY2VdXSwKICAgIG91dHB1dF9kaXI6IHN0ciB8IFBhdGgsCiAgICBtYXhpbXVtOiBpbnQgPSAxMiwKKSAtPiBsaXN0W3N0cl06CiAgICBuYW1lcyA9IHNvcnRlZChwcmVkaWN0aW9ucykKICAgIGlmIG5vdCBuYW1lcyBvciBtYXhpbXVtIDw9IDA6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBsZW4obmFtZXMpIDw9IG1heGltdW06CiAgICAgICAgc2VsZWN0ZWQgPSBuYW1lcwogICAgZWxpZiBtYXhpbXVtID09IDE6CiAgICAgICAgc2VsZWN0ZWQgPSBbbmFtZXNbbGVuKG5hbWVzKSAvLyAyXV0KICAgIGVsc2U6CiAgICAgICAgaW5kaWNlcyA9IHsKICAgICAgICAgICAgcm91bmQoaW5kZXggKiAobGVuKG5hbWVzKSAtIDEpIC8gKG1heGltdW0gLSAxKSkKICAgICAgICAgICAgZm9yIGluZGV4IGluIHJhbmdlKG1heGltdW0pCiAgICAgICAgfQogICAgICAgIHNlbGVjdGVkID0gW25hbWVzW2luZGV4XSBmb3IgaW5kZXggaW4gc29ydGVkKGluZGljZXMpXQoKICAgIG91dHB1dF9kaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNhdmVkID0gW10KICAgIGZvciBmaWxlX25hbWUgaW4gc2VsZWN0ZWQ6CiAgICAgICAgZ3JheSA9IHJlYWRfZ3JheXNjYWxlKGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyIC8gZmlsZV9uYW1lKQogICAgICAgIHZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV1bMF0KICAgICAgICBncm91bmRfdHJ1dGggPSBucC56ZXJvcyhncmF5LnNoYXBlLCBkdHlwZT1ib29sKQogICAgICAgIGZvciBhbm5vdGF0aW9uIGluIGRhdGFzZXQuYW5ub3RhdGlvbnNfYnlfaW1hZ2VfaWRbdmlldy5pbWFnZV9pZF06CiAgICAgICAgICAgIHJsZSA9IGFubm90YXRpb25fdG9fcmxlKGFubm90YXRpb24sIHZpZXcuaGVpZ2h0LCB2aWV3LndpZHRoKQogICAgICAgICAgICBlbmNvZGVkID0gewogICAgICAgICAgICAgICAgInNpemUiOiBybGVbInNpemUiXSwKICAgICAgICAgICAgICAgICJjb3VudHMiOiBybGVbImNvdW50cyJdLmVuY29kZSgiYXNjaWkiKQogICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShybGVbImNvdW50cyJdLCBzdHIpCiAgICAgICAgICAgICAgICBlbHNlIHJsZVsiY291bnRzIl0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZ3JvdW5kX3RydXRoIHw9IG1hc2tfdXRpbHMuZGVjb2RlKGVuY29kZWQpLmFzdHlwZShib29sKQogICAgICAgIHByZWRpY3RlZCA9IG5wLnplcm9zKGdyYXkuc2hhcGUsIGR0eXBlPWJvb2wpCiAgICAgICAgZm9yIGluc3RhbmNlIGluIHByZWRpY3Rpb25zW2ZpbGVfbmFtZV06CiAgICAgICAgICAgIHByZWRpY3RlZCB8PSBkZWNvZGVfcmxlKGluc3RhbmNlLnJsZSkuYXN0eXBlKGJvb2wpCgogICAgICAgIGNhbnZhcyA9IGN2Mi5jdnRDb2xvcihncmF5LCBjdjIuQ09MT1JfR1JBWTJSR0IpCiAgICAgICAgb3ZlcmxheSA9IGNhbnZhcy5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBndF9vbmx5ID0gZ3JvdW5kX3RydXRoICYgfnByZWRpY3RlZAogICAgICAgIHByZWRfb25seSA9IHByZWRpY3RlZCAmIH5ncm91bmRfdHJ1dGgKICAgICAgICBvdmVybGFwID0gZ3JvdW5kX3RydXRoICYgcHJlZGljdGVkCiAgICAgICAgb3ZlcmxheVtndF9vbmx5XSA9IDAuNTUgKiBvdmVybGF5W2d0X29ubHldICsgMC40NSAqIG5wLmFycmF5KFs0MCwgMjU1LCA4MF0pCiAgICAgICAgb3ZlcmxheVtwcmVkX29ubHldID0gMC41NSAqIG92ZXJsYXlbcHJlZF9vbmx5XSArIDAuNDUgKiBucC5hcnJheShbMjU1LCA1MCwgNTBdKQogICAgICAgIG92ZXJsYXlbb3ZlcmxhcF0gPSAwLjQ1ICogb3ZlcmxheVtvdmVybGFwXSArIDAuNTUgKiBucC5hcnJheShbMjU1LCAyMzAsIDMwXSkKICAgICAgICBjYW52YXMgPSBucC5jbGlwKG92ZXJsYXksIDAsIDI1NSkuYXN0eXBlKG5wLnVpbnQ4KQogICAgICAgIGN2Mi5wdXRUZXh0KAogICAgICAgICAgICBjYW52YXMsCiAgICAgICAgICAgIGYiR1QgdW5pb246IHtsZW4oZGF0YXNldC5hbm5vdGF0aW9uc19ieV9pbWFnZV9pZFt2aWV3LmltYWdlX2lkXSl9IGluc3RhbmNlcyB8ICIKICAgICAgICAgICAgZiJQcmVkaWN0ZWQ6IHtsZW4ocHJlZGljdGlvbnNbZmlsZV9uYW1lXSl9IHwgZ3JlZW4gR1QsIHJlZCBwcmVkLCB5ZWxsb3cgb3ZlcmxhcCIsCiAgICAgICAgICAgICgyNCwgNDgpLAogICAgICAgICAgICBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsCiAgICAgICAgICAgIDAuNywKICAgICAgICAgICAgKDI1NSwgMjU1LCAyNTUpLAogICAgICAgICAgICAyLAogICAgICAgICAgICBjdjIuTElORV9BQSwKICAgICAgICApCiAgICAgICAgZGVzdGluYXRpb24gPSBvdXRwdXRfZGlyIC8gZiJ7UGF0aChmaWxlX25hbWUpLnN0ZW19X2NvbXBhcmlzb24ucG5nIgogICAgICAgIGlmIG5vdCBjdjIuaW13cml0ZShzdHIoZGVzdGluYXRpb24pLCBjdjIuY3Z0Q29sb3IoY2FudmFzLCBjdjIuQ09MT1JfUkdCMkJHUikpOgogICAgICAgICAgICByYWlzZSBPU0Vycm9yKGYiRmFpbGVkIHRvIHNhdmUgdmFsaWRhdGlvbiBjb21wYXJpc29uIG92ZXJsYXk6IHtkZXN0aW5hdGlvbn0iKQogICAgICAgIHNhdmVkLmFwcGVuZChkZXN0aW5hdGlvbi5uYW1lKQogICAgcmV0dXJuIHNhdmVkCg==","scripts/run_model_experiment.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJFUE9fUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQT19ST09UIC8gInNyYyIpKQoKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5tb2RlbF9ydW5uZXIgaW1wb3J0IG1haW4gICMgbm9xYTogRTQwMgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK","scripts/run_final_submission.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuc3VibWlzc2lvbiBpbXBvcnQgcnVuX2ZpbmFsX3N1Ym1pc3Npb24gICMgbm9xYTogRTQwMgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigKICAgICAgICBkZXNjcmlwdGlvbj0iRml0IG9uIGFsbCBvZmZpY2lhbCB0cmFpbmluZyBvYnNlcnZhdGlvbnMgYW5kIHdyaXRlIGEgS2FnZ2xlIHN1Ym1pc3Npb24uIgogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXBvLXJvb3QiLCBkZWZhdWx0PXN0cihSRVBPX1JPT1QpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1uYW1lIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdGFyZ2V0LW1vZGUiLCBjaG9pY2VzPSgiYW5ub3RhdG9yIiwgImNvbnNlbnN1cyIpLCBkZWZhdWx0PSJjb25zZW5zdXMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTVfMDAwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcGF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTUxMikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdHRhIiwgdHlwZT1pbnQsIGNob2ljZXM9KDEsIDIsIDQsIDgpLCBkZWZhdWx0PTEpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZvcmVncm91bmQtdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjQ1KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ib3VuZGFyeS10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNjUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXdlaWdodCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MS41KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkLXRocmVzaG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC40NSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZC1taW4tYXJlYSIsIHR5cGU9aW50LCBkZWZhdWx0PTgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1pbi1pbnN0YW5jZS1hcmVhIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NTApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmVzdWx0ID0gcnVuX2ZpbmFsX3N1Ym1pc3Npb24oCiAgICAgICAgcmVwb19yb290PWFyZ3MucmVwb19yb290LAogICAgICAgIGRhdGFfcm9vdD1hcmdzLmRhdGFfcm9vdCwKICAgICAgICBydW5fbmFtZT1hcmdzLnJ1bl9uYW1lLAogICAgICAgIHRhcmdldF9tb2RlPWFyZ3MudGFyZ2V0X21vZGUsCiAgICAgICAgc3RlcHM9YXJncy5zdGVwcywKICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICBwYXRjaF9zaXplPWFyZ3MucGF0Y2hfc2l6ZSwKICAgICAgICB0dGE9YXJncy50dGEsCiAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9YXJncy5mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV90aHJlc2hvbGQ9YXJncy5ib3VuZGFyeV90aHJlc2hvbGQsCiAgICAgICAgYm91bmRhcnlfd2VpZ2h0PWFyZ3MuYm91bmRhcnlfd2VpZ2h0LAogICAgICAgIHNlZWRfdGhyZXNob2xkPWFyZ3Muc2VlZF90aHJlc2hvbGQsCiAgICAgICAgc2VlZF9taW5fYXJlYT1hcmdzLnNlZWRfbWluX2FyZWEsCiAgICAgICAgbWluX2luc3RhbmNlX2FyZWE9YXJncy5taW5faW5zdGFuY2VfYXJlYSwKICAgICkKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpKQo=","scripts/sweep_model_postprocessing.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuc3dlZXAgaW1wb3J0IHJ1bl9wb3N0cHJvY2Vzc2luZ19zd2VlcCAgIyBub3FhOiBFNDAyCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJTd2VlcCBpbnN0YW5jZSB0aHJlc2hvbGRzIG9uIGNhY2hlZCB2YWxpZGF0aW9uIG1hcHMuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwby1yb290IiwgZGVmYXVsdD1zdHIoUkVQT19ST09UKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ydW4tcm9vdCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW9ic2VydmF0aW9uLWxpbWl0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFyZ3VtZW50cyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHJlc3VsdHMgPSBydW5fcG9zdHByb2Nlc3Npbmdfc3dlZXAoCiAgICAgICAgcmVwb19yb290PWFyZ3VtZW50cy5yZXBvX3Jvb3QsCiAgICAgICAgZGF0YV9yb290PWFyZ3VtZW50cy5kYXRhX3Jvb3QsCiAgICAgICAgcnVuX3Jvb3Q9YXJndW1lbnRzLnJ1bl9yb290LAogICAgICAgIG9ic2VydmF0aW9uX2xpbWl0PWFyZ3VtZW50cy5vYnNlcnZhdGlvbl9saW1pdCwKICAgICkKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0c1s6M10sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCg==","scripts/sweep_model_instance_structure.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuc3RydWN0dXJlX3N3ZWVwIGltcG9ydCBydW5faW5zdGFuY2Vfc3RydWN0dXJlX3N3ZWVwICAjIG5vcWE6IEU0MDIKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IlN3ZWVwIGluc3RhbmNlIGFuZCBzZWVkIGFyZWEgY3V0b2ZmcyBvdmVyIGNhY2hlZCB2YWxpZGF0aW9uIG1hcHMuIgogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXBvLXJvb3QiLCBkZWZhdWx0PXN0cihSRVBPX1JPT1QpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1yb290IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZm9yZWdyb3VuZC10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNTUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXRocmVzaG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC42MCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYm91bmRhcnktd2VpZ2h0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xLjUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvdW5kYXJ5LXdlaWdodHMiLCB0eXBlPWZsb2F0LCBuYXJncz0iKyIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZC10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNDUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmVzdWx0ID0gcnVuX2luc3RhbmNlX3N0cnVjdHVyZV9zd2VlcCgKICAgICAgICByZXBvX3Jvb3Q9YXJncy5yZXBvX3Jvb3QsCiAgICAgICAgZGF0YV9yb290PWFyZ3MuZGF0YV9yb290LAogICAgICAgIHJ1bl9yb290PWFyZ3MucnVuX3Jvb3QsCiAgICAgICAgZm9yZWdyb3VuZF90aHJlc2hvbGQ9YXJncy5mb3JlZ3JvdW5kX3RocmVzaG9sZCwKICAgICAgICBib3VuZGFyeV90aHJlc2hvbGQ9YXJncy5ib3VuZGFyeV90aHJlc2hvbGQsCiAgICAgICAgYm91bmRhcnlfd2VpZ2h0PWFyZ3MuYm91bmRhcnlfd2VpZ2h0LAogICAgICAgIGJvdW5kYXJ5X3dlaWdodHM9dHVwbGUoYXJncy5ib3VuZGFyeV93ZWlnaHRzKSBpZiBhcmdzLmJvdW5kYXJ5X3dlaWdodHMgZWxzZSBOb25lLAogICAgICAgIHNlZWRfdGhyZXNob2xkPWFyZ3Muc2VlZF90aHJlc2hvbGQsCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdFs6NV0sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCg==","configs/validation-baseline.json":"ewogICJzZWVkIjogNDIsCiAgInZhbGlkYXRpb25fZnJhY3Rpb24iOiAwLjIsCiAgInN0cmF0aWZ5X2J5IjogInllYXIiLAogICJleHBlY3RlZF9zcGxpdF9zaGEyNTYiOiAiMzBjYjJlYzUyNDM3ZGFmOTdlNzY5YTNhNjFiZjgzMzUwMDYyZTZjZjRmNWQwMGFmMWY4Mzk5NjQ5MDY1ZTQ4ZSIsCiAgImJsdXJfa2VybmVsIjogMzEsCiAgInNvbGFyX2Rpc2tfcmFkaXVzX2ZyYWN0aW9uIjogMC40NCwKICAiY2xvc2luZ19rZXJuZWwiOiAzLAogICJjb21wb25lbnRfY29ubmVjdGl2aXR5IjogOCwKICAibWluX2NvbXBvbmVudF9hcmVhIjogNTAsCiAgIm1heF9jb21wb25lbnRfYXJlYSI6IDUwMDAwLAogICJpb3VfdGhyZXNob2xkIjogMC41LAogICJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIjogMC4wCn0K","notebooks/validation_baseline.ipynb":"ew0KICJjZWxscyI6IFsNCiAgew0KICAgImNlbGxfdHlwZSI6ICJtYXJrZG93biIsDQogICAibWV0YWRhdGEiOiB7fSwNCiAgICJzb3VyY2UiOiBbDQogICAgIiMgRXF1aU5leHVzIGdyb3VwZWQgdmFsaWRhdGlvbiBiYXNlbGluZVxuIiwNCiAgICAiXG4iLA0KICAgICJBdHRhY2ggdGhlIFNvbGFyIEZpbGFtZW50IFNlZ21lbnRhdGlvbiBDaGFsbGVuZ2UgZGF0YSBpbiBLYWdnbGUgYW5kIGVuYWJsZSBpbnRlcm5ldCBmb3IgdGhlIHB1YmxpYyBHaXRIdWIgY2hlY2tvdXQgYW5kIHBpbm5lZCBkZXBlbmRlbmN5IGNoZWNrLiBUaGlzIG5vdGVib29rIHJlY29yZHMgdGhlIHNvdXJjZSBjb21taXQsIGRldGVjdHMgdGhlIGF0dGFjaGVkIGlucHV0LCBhbmQgcnVucyB0aGUgc2hhcmVkIGxvY2FsLWNvbnRyYXN0IGJhc2VsaW5lLiBUaGUgcHJlZGljdG9yIGFuZCBzY29yZXIgcnVuIG9uIENQVTsgYSBHUFUgYWNjZWxlcmF0b3IgaXMgbm90IG5lZWRlZCBmb3IgdGhpcyBiYXNlbGluZS4iDQogICBdDQogIH0sDQogIHsNCiAgICJjZWxsX3R5cGUiOiAiY29kZSIsDQogICAiZXhlY3V0aW9uX2NvdW50IjogbnVsbCwNCiAgICJtZXRhZGF0YSI6IHt9LA0KICAgIm91dHB1dHMiOiBbXSwNCiAgICJzb3VyY2UiOiBbDQogICAgImZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aFxuIiwNCiAgICAiaW1wb3J0IHN1YnByb2Nlc3NcbiIsDQogICAgImltcG9ydCBzeXNcbiIsDQogICAgIlxuIiwNCiAgICAiUkVQT1NJVE9SWV9VUkwgPSBcImh0dHBzOi8vZ2l0aHViLmNvbS9Ccm93bkFzc2Fzc2luL0VxdWlOZXh1cy5naXRcIlxuIiwNCiAgICAiUkVQT1NJVE9SWV9SRUYgPSBcIm1haW5cIlxuIiwNCiAgICAiUkVQT1NJVE9SWV9QQVRIID0gUGF0aChcIi9rYWdnbGUvd29ya2luZy9FcXVpTmV4dXNcIilcbiIsDQogICAgIlxuIiwNCiAgICAiaWYgbm90IChSRVBPU0lUT1JZX1BBVEggLyBcIi5naXRcIikuaXNfZGlyKCk6XG4iLA0KICAgICIgICAgc3VicHJvY2Vzcy5ydW4oW1wiZ2l0XCIsIFwiY2xvbmVcIiwgXCItLWRlcHRoXCIsIFwiMVwiLCBcIi0tYnJhbmNoXCIsIFJFUE9TSVRPUllfUkVGLCBSRVBPU0lUT1JZX1VSTCwgc3RyKFJFUE9TSVRPUllfUEFUSCldLCBjaGVjaz1UcnVlKVxuIiwNCiAgICAiZWxzZTpcbiIsDQogICAgIiAgICBzdWJwcm9jZXNzLnJ1bihbXCJnaXRcIiwgXCItQ1wiLCBzdHIoUkVQT1NJVE9SWV9QQVRIKSwgXCJmZXRjaFwiLCBcIi0tZGVwdGhcIiwgXCIxXCIsIFwib3JpZ2luXCIsIFJFUE9TSVRPUllfUkVGXSwgY2hlY2s9VHJ1ZSlcbiIsDQogICAgIiAgICBzdWJwcm9jZXNzLnJ1bihbXCJnaXRcIiwgXCItQ1wiLCBzdHIoUkVQT1NJVE9SWV9QQVRIKSwgXCJjaGVja291dFwiLCBcIi0tZGV0YWNoXCIsIFwiRkVUQ0hfSEVBRFwiXSwgY2hlY2s9VHJ1ZSlcbiIsDQogICAgIlxuIiwNCiAgICAiU09VUkNFX0NPTU1JVCA9IHN1YnByb2Nlc3MucnVuKFtcImdpdFwiLCBcIi1DXCIsIHN0cihSRVBPU0lUT1JZX1BBVEgpLCBcInJldi1wYXJzZVwiLCBcIkhFQURcIl0sIGNoZWNrPVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkuc3Rkb3V0LnN0cmlwKClcbiIsDQogICAgInN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQT1NJVE9SWV9QQVRIIC8gXCJzcmNcIikpXG4iLA0KICAgICJwcmludChmXCJFcXVpTmV4dXMgc291cmNlIGNvbW1pdDoge1NPVVJDRV9DT01NSVR9XCIpIg0KICAgXQ0KICB9LA0KICB7DQogICAiY2VsbF90eXBlIjogImNvZGUiLA0KICAgImV4ZWN1dGlvbl9jb3VudCI6IG51bGwsDQogICAibWV0YWRhdGEiOiB7fSwNCiAgICJvdXRwdXRzIjogW10sDQogICAic291cmNlIjogWw0KICAgICJpbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhXG4iLA0KICAgICJcbiIsDQogICAgInJlcXVpcmVtZW50cyA9IFJFUE9TSVRPUllfUEFUSCAvIFwicmVxdWlyZW1lbnRzLnR4dFwiXG4iLA0KICAgICJwaW5uZWRfcGFja2FnZXMgPSBbXVxuIiwNCiAgICAiZm9yIGxpbmUgaW4gcmVxdWlyZW1lbnRzLnJlYWRfdGV4dChlbmNvZGluZz1cInV0Zi04XCIpLnNwbGl0bGluZXMoKTpcbiIsDQogICAgIiAgICBsaW5lID0gbGluZS5zdHJpcCgpXG4iLA0KICAgICIgICAgaWYgbm90IGxpbmUgb3IgbGluZS5zdGFydHN3aXRoKFwiI1wiKSBvciBsaW5lLnN0YXJ0c3dpdGgoXCItLVwiKTpcbiIsDQogICAgIiAgICAgICAgY29udGludWVcbiIsDQogICAgIiAgICBwYWNrYWdlLCBleHBlY3RlZCA9IGxpbmUuc3BsaXQoXCI9PVwiLCAxKVxuIiwNCiAgICAiICAgIHBpbm5lZF9wYWNrYWdlcy5hcHBlbmQoKHBhY2thZ2UsIGV4cGVjdGVkKSlcbiIsDQogICAgIlxuIiwNCiAgICAiZGVmIGZpbmRfbWlzbWF0Y2hlcygpOlxuIiwNCiAgICAiICAgIG1pc21hdGNoZXMgPSBbXVxuIiwNCiAgICAiICAgIGZvciBwYWNrYWdlLCBleHBlY3RlZCBpbiBwaW5uZWRfcGFja2FnZXM6XG4iLA0KICAgICIgICAgICAgIHRyeTpcbiIsDQogICAgIiAgICAgICAgICAgIGluc3RhbGxlZCA9IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKHBhY2thZ2UpXG4iLA0KICAgICIgICAgICAgIGV4Y2VwdCBpbXBvcnRsaWIubWV0YWRhdGEuUGFja2FnZU5vdEZvdW5kRXJyb3I6XG4iLA0KICAgICIgICAgICAgICAgICBpbnN0YWxsZWQgPSBOb25lXG4iLA0KICAgICIgICAgICAgIGlmIGluc3RhbGxlZCAhPSBleHBlY3RlZDpcbiIsDQogICAgIiAgICAgICAgICAgIG1pc21hdGNoZXMuYXBwZW5kKChwYWNrYWdlLCBpbnN0YWxsZWQsIGV4cGVjdGVkKSlcbiIsDQogICAgIiAgICByZXR1cm4gbWlzbWF0Y2hlc1xuIiwNCiAgICAiXG4iLA0KICAgICJtaXNtYXRjaGVzID0gZmluZF9taXNtYXRjaGVzKClcbiIsDQogICAgImlmIG1pc21hdGNoZXM6XG4iLA0KICAgICIgICAgcHJpbnQoXCJJbnN0YWxsaW5nIHBpbm5lZCBzaGFyZWQgYmFzZWxpbmUgcGFja2FnZXM6XCIsIG1pc21hdGNoZXMpXG4iLA0KICAgICIgICAgc3VicHJvY2Vzcy5ydW4oXG4iLA0KICAgICIgICAgICAgIFtcbiIsDQogICAgIiAgICAgICAgICAgIHN5cy5leGVjdXRhYmxlLFxuIiwNCiAgICAiICAgICAgICAgICAgXCItbVwiLFxuIiwNCiAgICAiICAgICAgICAgICAgXCJwaXBcIixcbiIsDQogICAgIiAgICAgICAgICAgIFwiaW5zdGFsbFwiLFxuIiwNCiAgICAiICAgICAgICAgICAgXCItLWRpc2FibGUtcGlwLXZlcnNpb24tY2hlY2tcIixcbiIsDQogICAgIiAgICAgICAgICAgIFwiLS1uby1jYWNoZS1kaXJcIixcbiIsDQogICAgIiAgICAgICAgICAgIFwiLXJcIixcbiIsDQogICAgIiAgICAgICAgICAgIHN0cihyZXF1aXJlbWVudHMpLFxuIiwNCiAgICAiICAgICAgICBdLFxuIiwNCiAgICAiICAgICAgICBjaGVjaz1UcnVlLFxuIiwNCiAgICAiICAgIClcbiIsDQogICAgIiAgICByZW1haW5pbmcgPSBmaW5kX21pc21hdGNoZXMoKVxuIiwNCiAgICAiICAgIGlmIHJlbWFpbmluZzpcbiIsDQogICAgIiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGZcIlNoYXJlZCBwYWNrYWdlIHBpbnMgcmVtYWluIHVuc2F0aXNmaWVkOiB7cmVtYWluaW5nfVwiKVxuIiwNCiAgICAiICAgIHByaW50KFwiU2hhcmVkIHBhY2thZ2VzIGluc3RhbGxlZCBhbmQgdmVyaWZpZWQ7IGNvbnRpbnVpbmcgdGhpcyBLYWdnbGUgcnVuLlwiKVxuIiwNCiAgICAiZWxzZTpcbiIsDQogICAgIiAgICBwcmludChcIlBpbm5lZCBzaGFyZWQgYmFzZWxpbmUgcGFja2FnZXMgYXJlIHByZXNlbnQuXCIpXG4iLA0KICAgICJcbiIsDQogICAgIiMgVGhlIGJhc2VsaW5lIHJ1bnMgb24gQ1BVLiBLZWVwIEthZ2dsZSdzIHByb3ZpZGVkIFB5VG9yY2ggYnVpbGQgaW50YWN0OyB0aGVcbiIsDQogICAgIiMgQ1VEQS1zcGVjaWZpYyBwaW4gaW4gcmVxdWlyZW1lbnRzLWdwdS50eHQgaXMgZm9yIGxvY2FsIEdQVSBzbW9rZS9tb2RlbCB3b3JrLlxuIg0KICAgXQ0KICB9LA0KICB7DQogICAiY2VsbF90eXBlIjogImNvZGUiLA0KICAgImV4ZWN1dGlvbl9jb3VudCI6IG51bGwsDQogICAibWV0YWRhdGEiOiB7fSwNCiAgICJvdXRwdXRzIjogW10sDQogICAic291cmNlIjogWw0KICAgICJpbXBvcnQganNvblxuIiwNCiAgICAiXG4iLA0KICAgICJmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGhcbiIsDQogICAgIlxuIiwNCiAgICAiREFUQV9TRUFSQ0hfUk9PVCA9IFBhdGgoXCIva2FnZ2xlL2lucHV0XCIpXG4iLA0KICAgICJPVVRQVVRfUk9PVCA9IFBhdGgoXCIva2FnZ2xlL3dvcmtpbmcvZmlsYW1lbnQtdmFsaWRhdGlvbi1iYXNlbGluZVwiKVxuIiwNCiAgICAiUlVOTkVSID0gUkVQT1NJVE9SWV9QQVRIIC8gXCJzY3JpcHRzXCIgLyBcInJ1bl92YWxpZGF0aW9uLnB5XCJcbiIsDQogICAgInN1YnByb2Nlc3MucnVuKFxuIiwNCiAgICAiICAgIFtcbiIsDQogICAgIiAgICAgICAgc3lzLmV4ZWN1dGFibGUsXG4iLA0KICAgICIgICAgICAgIHN0cihSVU5ORVIpLFxuIiwNCiAgICAiICAgICAgICBcIi0tcmVwby1yb290XCIsXG4iLA0KICAgICIgICAgICAgIHN0cihSRVBPU0lUT1JZX1BBVEgpLFxuIiwNCiAgICAiICAgICAgICBcIi0tZGF0YS1yb290XCIsXG4iLA0KICAgICIgICAgICAgIHN0cihEQVRBX1NFQVJDSF9ST09UKSxcbiIsDQogICAgIiAgICAgICAgXCItLW91dHB1dC1yb290XCIsXG4iLA0KICAgICIgICAgICAgIHN0cihPVVRQVVRfUk9PVCksXG4iLA0KICAgICIgICAgICAgIFwiLS1vdmVybGF5LWNvdW50XCIsXG4iLA0KICAgICIgICAgICAgIFwiMTJcIixcbiIsDQogICAgIiAgICBdLFxuIiwNCiAgICAiICAgIGNoZWNrPVRydWUsXG4iLA0KICAgICIpXG4iLA0KICAgICJSRVNVTFQgPSBqc29uLmxvYWRzKChPVVRQVVRfUk9PVCAvIFwibWV0cmljcy5qc29uXCIpLnJlYWRfdGV4dChlbmNvZGluZz1cInV0Zi04XCIpKVxuIiwNCiAgICAicHJpbnQoXCJWYWxpZGF0aW9uIFBROlwiLCBSRVNVTFRbXCJtZXRyaWNzXCJdW1wiZ2xvYmFsXCJdW1wicHFcIl0pXG4iLA0KICAgICJwcmludChcIlNwbGl0IFNIQS0yNTY6XCIsIFJFU1VMVFtcInNwbGl0XCJdW1wic2hhMjU2XCJdKVxuIiwNCiAgICAicHJpbnQoXCJTcGxpdCBtYXRjaGVzIGZyb3plbiBkaWdlc3Q6XCIsIFJFU1VMVFtcInNwbGl0XCJdW1wibWF0Y2hlc19leHBlY3RlZF9zaGEyNTZcIl0pXG4iLA0KICAgICJwcmludChcIlJ1biBvdXRwdXQ6XCIsIE9VVFBVVF9ST09UKVxuIg0KICAgXQ0KICB9LA0KICB7DQogICAiY2VsbF90eXBlIjogImNvZGUiLA0KICAgImV4ZWN1dGlvbl9jb3VudCI6IG51bGwsDQogICAibWV0YWRhdGEiOiB7fSwNCiAgICJvdXRwdXRzIjogW10sDQogICAic291cmNlIjogWw0KICAgICJmcm9tIElQeXRob24uZGlzcGxheSBpbXBvcnQgZGlzcGxheSwgSW1hZ2VcbiIsDQogICAgIlxuIiwNCiAgICAiZm9yIGltYWdlX3BhdGggaW4gc29ydGVkKChPVVRQVVRfUk9PVCAvIFwib3ZlcmxheXNcIikuZ2xvYihcIioucG5nXCIpKVs6Nl06XG4iLA0KICAgICIgICAgcHJpbnQoaW1hZ2VfcGF0aC5uYW1lKVxuIiwNCiAgICAiICAgIGRpc3BsYXkoSW1hZ2UoZmlsZW5hbWU9c3RyKGltYWdlX3BhdGgpKSlcbiIsDQogICAgIlxuIiwNCiAgICAicHJpbnQoKE9VVFBVVF9ST09UIC8gXCJtZXRyaWNzLmpzb25cIikucmVhZF90ZXh0KGVuY29kaW5nPVwidXRmLThcIikpIg0KICAgXQ0KICB9DQogXSwNCiAibWV0YWRhdGEiOiB7DQogICJrZXJuZWxzcGVjIjogew0KICAgImRpc3BsYXlfbmFtZSI6ICJQeXRob24gMyIsDQogICAibGFuZ3VhZ2UiOiAicHl0aG9uIiwNCiAgICJuYW1lIjogInB5dGhvbjMiDQogIH0sDQogICJsYW5ndWFnZV9pbmZvIjogew0KICAgIm5hbWUiOiAicHl0aG9uIiwNCiAgICJ2ZXJzaW9uIjogIjMuMTIiDQogIH0NCiB9LA0KICJuYmZvcm1hdCI6IDQsDQogIm5iZm9ybWF0X21pbm9yIjogNQ0KfQ0K","requirements.txt":"bnVtcHk9PTIuMi42Cm9wZW5jdi1weXRob24taGVhZGxlc3M9PTQuMTIuMC44OApwYW5kYXM9PTMuMS4xCnNjaWtpdC1sZWFybj09MS45LjEKc2NpcHk9PTEuMTguMQpzY2lraXQtaW1hZ2U9PTAuMjUuMgptYXRwbG90bGliPT0zLjExLjIKcHljb2NvdG9vbHM9PTIuMC4xMQo=","requirements-gpu.txt":"LS1leHRyYS1pbmRleC11cmwgaHR0cHM6Ly9kb3dubG9hZC5weXRvcmNoLm9yZy93aGwvY3UxMzIKdG9yY2g9PTIuMTUuMCtjdTEzMgo="}
for relative, payload in SOURCE_FILES.items():
    destination = REPOSITORY_PATH / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(payload))
print(f'Embedded {len(SOURCE_FILES)} source files from base commit {SOURCE_COMMIT}')
print(f'Source bundle SHA-256: {SOURCE_BUNDLE_SHA256}')


In [ ]:
import importlib.metadata
import subprocess
import sys
requirements = REPOSITORY_PATH / 'requirements.txt'
pins = []
for line in requirements.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if line and not line.startswith('#') and not line.startswith('--'):
        pins.append(tuple(line.split('==', 1)))
mismatches = []
for package, expected in pins:
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        mismatches.append((package, installed, expected))
if mismatches:
    print('Installing pinned non-Torch packages:', mismatches)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '-r', str(requirements)], check=True)
print('Kaggle runtime inventory:')
subprocess.run([sys.executable, '-c', "import torch, numpy, cv2, skimage; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'available', torch.cuda.is_available(), 'GPUs', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]); print('numpy', numpy.__version__, 'opencv', cv2.__version__, 'scikit-image', skimage.__version__)"], check=True)


In [ ]:
import os
import subprocess
import sys
os.environ['EQUINEXUS_SOURCE_COMMIT'] = SOURCE_COMMIT
os.environ['EQUINEXUS_SOURCE_BUNDLE_SHA256'] = SOURCE_BUNDLE_SHA256
os.environ['EQUINEXUS_NOTEBOOK_VERSION'] = 'model-sprint-v2'
os.environ['PYTHONUNBUFFERED'] = '1'
probe = "import json, torch; print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'available':torch.cuda.is_available(),'devices':[{'name':torch.cuda.get_device_name(i),'total_memory_bytes':torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]})); assert torch.cuda.is_available(), 'Kaggle GPU is required'"
subprocess.run([sys.executable, '-c', probe], check=True)


In [ ]:
command = [sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'run_model_experiment.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', '/kaggle/input', '--run-name', 'unet-annotator-1024-kaggle-v2', '--target-mode', 'annotator', '--steps', '5000', '--batch-size', '2', '--patch-size', '1024', '--tta', '2', '--overlay-count', '12']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
sweep_command = [sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'sweep_model_postprocessing.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', '/kaggle/input', '--run-root', '/kaggle/working/equinexus/unet-annotator-1024-kaggle-v2']
print('Sweeping validation postprocessing:', ' '.join(sweep_command))
subprocess.run(sweep_command, check=True)
import json
from pathlib import Path
best_config = json.loads(Path('/kaggle/working/equinexus/unet-annotator-1024-kaggle-v2/best-postprocessing/metrics.json').read_text(encoding='utf-8'))['configuration']
structure_command = [sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'sweep_model_instance_structure.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', '/kaggle/input', '--run-root', '/kaggle/working/equinexus/unet-annotator-1024-kaggle-v2', '--foreground-threshold', str(best_config['foreground_threshold']), '--boundary-threshold', str(best_config['boundary_threshold']), '--boundary-weight', str(best_config['boundary_weight']), '--seed-threshold', str(best_config['seed_threshold'])]
print('Sweeping instance-size and seed cutoffs:', ' '.join(structure_command))
subprocess.run(structure_command, check=True)


In [ ]:
# Enable only after reviewing the completed holdout sweeps and selecting this model.
RUN_FINAL_SUBMISSION = False
if RUN_FINAL_SUBMISSION:
    selected = json.loads(Path('/kaggle/working/equinexus/unet-annotator-1024-kaggle-v2/best-instance-structure/metrics.json').read_text(encoding='utf-8'))['configuration']
    final_command = [sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'run_final_submission.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', '/kaggle/input', '--run-name', 'final-annotator-1024-kaggle-v2', '--target-mode', 'annotator', '--steps', '5000', '--batch-size', '2', '--patch-size', '1024', '--tta', '2', '--foreground-threshold', str(selected['foreground_threshold']), '--boundary-threshold', str(selected['boundary_threshold']), '--boundary-weight', str(selected['boundary_weight']), '--seed-threshold', str(selected['seed_threshold']), '--seed-min-area', str(selected['seed_min_area']), '--min-instance-area', str(selected['min_instance_area'])]
    print('Training on all official observations and writing the validated submission:', ' '.join(final_command))
    subprocess.run(final_command, check=True)
else:
    print('Final training is deferred until the validation model and postprocessing are selected.')


In [ ]:
from pathlib import Path
for run in ('unet-annotator-1024-kaggle-v2', 'final-annotator-1024-kaggle-v2'):
    metrics_path = Path('/kaggle/working/equinexus') / run / 'metrics.json'
    if metrics_path.is_file():
        print(f'\n{run}:\n{metrics_path.read_text(encoding="utf-8")}')
submission_path = Path('/kaggle/working/equinexus/final-annotator-1024-kaggle-v2/submission.csv')
print('Submission path:', submission_path if submission_path.is_file() else 'not generated by this validation-only version')
